OBJETIVO: comparar otros sintetizadores contra Gaussian Copula V2.1, usando exactamente el mismo protocolo experimental para que la comparación sea justa.

### 6.1 Objetivo

Objetivo

Esta notebook tiene como objetivo comparar diferentes métodos de generación de datos sintéticos utilizando un protocolo experimental común, tomando como referencia la versión optimizada de Gaussian Copula (V2.1) desarrollada en la notebook anterior.

La comparación se realizará considerando tanto la fidelidad estadística como la utilidad predictiva de los datasets sintéticos generados.

Para evaluar la utilidad predictiva se utilizarán tres escenarios:

- TRTR (Train Real, Test Real): baseline de referencia.
- TSTR (Train Synthetic, Test Real): evaluación de la capacidad de transferencia del conocimiento aprendido a partir de datos sintéticos hacia datos reales.
- TSTS (Train Synthetic, Test Synthetic): evaluación de la utilidad interna del propio dataset sintético.

Durante esta fase se utilizará la misma configuración de modelo predictivo para todos los sintetizadores, evitando optimizar los clasificadores de forma independiente. De esta manera, cualquier diferencia observada podrá atribuirse principalmente a la calidad del proceso de generación sintética.

Gaussian Copula V2.1 será utilizada como benchmark sintético inicial.

Los valores de referencia son:

- TRTR Real: ROC-AUC = 0.765600
- TSTR Gaussian Copula V2.1: ROC-AUC = 0.726674
- TSTS Gaussian Copula V2.1: ROC-AUC = 0.621809

### 6.2 Carga de datasets y configuración común

#### 6.2.1 Dataset real

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
#Leer desde drive
from google.colab import drive
drive.mount('/content/drive')

pd.set_option("display.max_columns", None)

ruta = "/content/drive/MyDrive/Maestria VIU/Tesis/Notebooks/TFM/02_Data_Preparation/dataset_real_final.parquet"

df_real = pd.read_parquet(ruta)

print("Dataset real:", df_real.shape)

print("\nTARGET REAL:")
print(df_real["TARGET"].value_counts(normalize=True))

Mounted at /content/drive
Dataset real: (307507, 41)

TARGET REAL:
TARGET
0    0.91927
1    0.08073
Name: proportion, dtype: float64


#### 6.2.2 Gaussian Copula V2.1

In [ ]:
df_gc_v21 = pd.read_parquet("/content/drive/MyDrive/Maestria VIU/Tesis/Notebooks/TFM//05_Synthetic_Data_Optimization/df_synth_v21.parquet")

print("Gaussian Copula V2.1:", df_gc_v21.shape)

print("\nTARGET GC V2.1:")
print(df_gc_v21["TARGET"].value_counts(normalize=True))

Gaussian Copula V2.1: (307507, 41)

TARGET GC V2.1:
TARGET
0    0.91927
1    0.08073
Name: proportion, dtype: float64


In [ ]:
print(
    "¿Mismas columnas?:",
    set(df_real.columns) == set(df_gc_v21.columns)
)

¿Mismas columnas?: True


#### 6.2.3 Configuración experimental común

In [ ]:
#Congelamos las reglas que no cambiaremos entre sintetizadores
RANDOM_STATE = 42
TEST_SIZE = 0.20

AUC_TRTR_BASELINE = 0.765600
AUC_TSTR_GC_V21 = 0.726674
AUC_TSTS_GC_V21 = 0.621809

print("Benchmark:")
print("TRTR:", AUC_TRTR_BASELINE)
print("TSTR Gaussian V2.1:", AUC_TSTR_GC_V21)
print("TSTS Gaussian V2.1:", AUC_TSTS_GC_V21)

Benchmark:
TRTR: 0.7656
TSTR Gaussian V2.1: 0.726674
TSTS Gaussian V2.1: 0.621809


In [ ]:
#El modelo del diagnostico tmb queda congelado (Esto es importante: no cambiamos estos hiperparámetros para CTGAN, TVAE u otro sintetizador.)
from lightgbm import LGBMClassifier

def crear_modelo_diagnostico():
    return LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

**¿Cuál debería ser el segundo sintetizador?**

Aquí yo empezaría por CTGAN.

¿Por qué?

Porque en la Notebook 03 ya lo habíamos evaluado y quedó segundo en fidelidad estadística, detrás de Gaussian Copula. Además, al ser un modelo generativo basado en GAN, tiene mayor capacidad teórica para representar relaciones no lineales y distribuciones complejas que una copula gaussiana.

Entonces nuestra pregunta sería:

¿CTGAN sacrifica algo de fidelidad estadística global, pero conserva mejor la utilidad predictiva que Gaussian Copula V2.1?

Ese es exactamente el experimento que vale la pena hacer ahora.

Por tanto, después de terminar 6.1 y 6.2, seguiríamos con:

6.3 Benchmark de referencia

y luego:

6.4 CTGAN como segundo sintetizador.

### 6.3 Benchmark de referencia

Antes de evaluar nuevos métodos de generación sintética, se establece un benchmark común que permita realizar una comparación homogénea entre sintetizadores.

Se consideran tres escenarios de evaluación:

- **TRTR (Train Real – Test Real):** representa el rendimiento de referencia obtenido mediante entrenamiento y evaluación con datos reales.
- **TSTR (Train Synthetic – Test Real):** mide la capacidad de transferencia de un modelo entrenado exclusivamente con datos sintéticos hacia observaciones reales.
- **TSTS (Train Synthetic – Test Synthetic):** mide la utilidad predictiva interna del dataset sintético.

Gaussian Copula V2.1, obtenida tras el proceso de diagnóstico y optimización desarrollado en la Notebook 05, constituye el benchmark sintético frente al cual se compararán los métodos posteriores.

Durante la comparación de sintetizadores se mantendrá constante la configuración del modelo predictivo y el conjunto real de evaluación, evitando que diferencias en la optimización del clasificador interfieran en la evaluación de la calidad de los datos sintéticos.

#### 6.3.1 Resultados de referencia

In [ ]:
# =========================================================
# 6.3.1 Benchmark experimental de referencia
# =========================================================

benchmark = pd.DataFrame({
    "Escenario": [
        "TRTR",
        "TSTR",
        "TSTS"
    ],

    "Entrenamiento": [
        "Datos reales",
        "Gaussian Copula V2.1",
        "Gaussian Copula V2.1"
    ],

    "Evaluacion": [
        "Real Test",
        "Real Test",
        "Synthetic Test V2.1"
    ],

    "ROC_AUC": [
        AUC_TRTR_BASELINE,
        AUC_TSTR_GC_V21,
        AUC_TSTS_GC_V21
    ]
})

benchmark

,Escenario,Entrenamiento,Evaluacion,ROC_AUC
0,TRTR,Datos reales,Real Test,0.765600
1,TSTR,Gaussian Copula V2.1,Real Test,0.726674
2,TSTS,Gaussian Copula V2.1,Synthetic Test V2.1,0.621809


#### 6.3.2 Cálculo de gaps respecto al baseline

In [ ]:
#Metrica para comparar los futuros sintetizadores
# =========================================================
# 6.3.2 Gap respecto al baseline TRTR
# =========================================================

benchmark["Gap_vs_TRTR"] = (
    AUC_TRTR_BASELINE -
    benchmark["ROC_AUC"]
)

benchmark["ROC_AUC_Relativo_pct"] = (
    benchmark["ROC_AUC"] /
    AUC_TRTR_BASELINE
) * 100

benchmark.round(4)

,Escenario,Entrenamiento,Evaluacion,ROC_AUC,Gap_vs_TRTR,ROC_AUC_Relativo_pct
0,TRTR,Datos reales,Real Test,0.7656,0.0000,100.0000
1,TSTR,Gaussian Copula V2.1,Real Test,0.7267,0.0389,94.9156
2,TSTS,Gaussian Copula V2.1,Synthetic Test V2.1,0.6218,0.1438,81.2185


Aquí mantenemos la misma precaución metodológica:

94.92 % no significa que V2.1 conserve el 94.92 % de la información del dataset real.

Significa únicamente que su ROC-AUC TSTR equivale al 94.92 % del ROC-AUC obtenido en TRTR.

#### 6.3.3 Diferencia TSTR–TSTS

In [ ]:
#### También quiero conservar explícitamente esta métrica porque será interesante cuando comparemos CTGAN.
# =========================================================
# 6.3.3 Gap entre utilidad externa e interna
# =========================================================

gap_tstr_tsts_gc = (
    AUC_TSTR_GC_V21 -
    AUC_TSTS_GC_V21
)

print(
    f"Gap TSTR - TSTS Gaussian Copula V2.1: "
    f"{gap_tstr_tsts_gc:.6f}"
)

Gap TSTR - TSTS Gaussian Copula V2.1: 0.104865


Esto nos da otra dimensión de comparación.

Imagina que posteriormente CTGAN obtiene:

TSTR = 0.735
TSTS = 0.710

No solo habría mejorado TSTR.

También habría reducido muchísimo la diferencia entre el comportamiento sintético y su capacidad de transferencia.

Ese sería un resultado metodológicamente muy interesante.

#### 6.3.4 Tabla maestra de comparación

In [ ]:
# =========================================================
# 6.3.4 Tabla maestra de sintetizadores
# =========================================================

resultados_sintetizadores = pd.DataFrame({

    "Sintetizador": [
        "Gaussian Copula V2.1"
    ],

    "TSTR_ROC_AUC": [
        AUC_TSTR_GC_V21
    ],

    "TSTS_ROC_AUC": [
        AUC_TSTS_GC_V21
    ]
})

resultados_sintetizadores["Gap_TRTR_TSTR"] = (
    AUC_TRTR_BASELINE -
    resultados_sintetizadores["TSTR_ROC_AUC"]
)

resultados_sintetizadores["Gap_TRTR_TSTS"] = (
    AUC_TRTR_BASELINE -
    resultados_sintetizadores["TSTS_ROC_AUC"]
)

resultados_sintetizadores["Gap_TSTR_TSTS"] = (
    resultados_sintetizadores["TSTR_ROC_AUC"] -
    resultados_sintetizadores["TSTS_ROC_AUC"]
)

resultados_sintetizadores["TSTR_Relativo_pct"] = (
    resultados_sintetizadores["TSTR_ROC_AUC"] /
    AUC_TRTR_BASELINE
) * 100

resultados_sintetizadores.round(4)

,Sintetizador,TSTR_ROC_AUC,TSTS_ROC_AUC,Gap_TRTR_TSTR,Gap_TRTR_TSTS,Gap_TSTR_TSTS,TSTR_Relativo_pct
0,Gaussian Copula V2.1,0.7267,0.6218,0.0389,0.1438,0.1049,94.9156


#### 6.3.5 Criterios de comparación

Los sintetizadores serán evaluados bajo un protocolo experimental común.

El criterio predictivo principal será **TSTR ROC-AUC**, dado que permite determinar si un modelo entrenado exclusivamente con datos sintéticos conserva capacidad de generalización sobre observaciones reales.

Como criterio complementario se utilizará **TSTS ROC-AUC**, que permitirá estudiar la utilidad interna del dataset sintético para escenarios en los que el usuario únicamente dispone de información sintética.

El escenario **TRTR** permanecerá fijo como baseline real.

Por tanto, un sintetizador será considerado predictivamente superior a Gaussian Copula V2.1 si:

1. Reduce el gap TSTR respecto al baseline TRTR.
2. Mantiene o mejora la utilidad interna observada mediante TSTS.
3. Conserva adecuadamente las distribuciones y relaciones estadísticas relevantes.
4. No introduce inconsistencias estructurales o semánticas significativas.

La selección final no dependerá exclusivamente de una única métrica, sino del equilibrio entre fidelidad estadística y utilidad predictiva.

**NOTA** Un punto metodológico que quiero corregir antes de CTGAN

No usaría TSTR - TSTS como criterio de calidad por sí mismo. Lo calculamos y lo observamos, pero no queremos simplemente minimizarlo.

Por ejemplo, un sintetizador con:

TSTR = 0.55
TSTS = 0.54
gap  = 0.01

tendría un gap excelente, pero sería pésimo predictivamente.

Por eso nuestras prioridades quedan:

1.º TSTR → criterio predictivo principal
2.º Fidelidad estadística/predictiva → criterio estructural
3.º TSTS → utilidad interna complementaria
4.º Gap TSTR-TSTS → diagnóstico, no objetivo de optimización

### 6.4 CTGAN

#### 6.4.1 Justificación de la selección de CTGAN

Tras establecer Gaussian Copula V2.1 como benchmark sintético, se evalúa un segundo enfoque de generación con el objetivo de determinar si un modelo generativo más flexible puede conservar mejor las dependencias multivariantes y la utilidad predictiva observada en los datos reales.

Se selecciona **CTGAN (Conditional Tabular GAN)** como segundo sintetizador debido a que está específicamente diseñado para la generación de datos tabulares que contienen variables continuas y categóricas, así como distribuciones complejas y potencialmente desbalanceadas.

La elección también responde a las limitaciones identificadas durante la optimización de Gaussian Copula. Aunque la versión V2.1 mejoró la conservación de relaciones críticas con TARGET y alcanzó un ROC-AUC TSTR de 0.726674, todavía se observó una atenuación de determinadas dependencias multivariantes y una utilidad interna inferior al baseline real (TSTS = 0.621809).

A diferencia del enfoque basado en cópulas, CTGAN utiliza una arquitectura generativa adversarial orientada a datos tabulares, proporcionando mayor flexibilidad para representar relaciones no lineales y distribuciones complejas.

Por tanto, el objetivo del experimento no consiste en asumir que CTGAN será superior, sino en comprobar empíricamente si su mayor capacidad de representación permite:

1. conservar adecuadamente las distribuciones estadísticas de las variables;
2. preservar mejor las relaciones predictivas con TARGET;
3. mejorar el rendimiento TSTR respecto a Gaussian Copula V2.1;
4. mejorar la utilidad interna medida mediante TSTS.

La evaluación se realizará utilizando exactamente el mismo protocolo experimental definido para Gaussian Copula V2.1, manteniendo constantes el conjunto real de evaluación, el preprocesamiento predictivo, el modelo LightGBM diagnóstico y sus hiperparámetros.

#### 6.4.2 Estrategia de preparación para CTGAN

Aquí tenemos una decisión importante.

En Gaussian Copula aprendimos varias cosas, pero no todas las modificaciones de V2.1 deben copiarse automáticamente a CTGAN.

Yo las dividiría así:

| Decisión aprendida                        | CTGAN             | Motivo                                                                            |
| ----------------------------------------- | ----------------- | --------------------------------------------------------------------------------- |
| No sintetizar variables derivadas         | ✅ Sí              | Es independiente del sintetizador                                                 |
| Reconstruir ratios después                | ✅ Sí              | Mantiene consistencia matemática                                                  |
| Corregir tipos semánticos                 | ✅ Sí              | CNT_CHILDREN, CNT_FAM_MEMBERS, etc. son conceptualmente discretas                 |
| `SK_ID_CURR` como ID                      | ✅ Sí              | No contiene señal que queramos aprender                                           |
| Forzar proporción TARGET                  | ⚠️ Evaluar        | CTGAN ya dispone de mecanismos condicionales                                      |
| Generar TARGET=0 y TARGET=1 separadamente | ❌ No inicialmente | Estaríamos imponiendo a CTGAN una solución diseñada para corregir Gaussian Copula |
| Manipular correlaciones                   | ❌ No              | Invalidaría la comparación                                                        |



Esto es fundamental para que el experimento sea limpio.

¿Por qué no generar inmediatamente por TARGET?

Porque queremos comprobar primero qué es capaz de aprender CTGAN por sí mismo.

Gaussian Copula V2.1 necesitó generación condicionada porque habíamos diagnosticado que la relación TARGET-X se debilitaba mucho.

Pero CTGAN es precisamente un modelo diseñado para aprender distribuciones tabulares complejas y utiliza mecanismos condicionales durante su entrenamiento.

Si desde el comienzo hacemos:

TARGET = 0 → generar 282682

TARGET = 1 → generar 24825

estaríamos trasladando una corrección específica de Gaussian Copula a CTGAN sin comprobar si CTGAN la necesita.

Eso reduciría el valor del experimento.

**NOTA** Para mantener la comparabilidad con Gaussian Copula V2.1, CTGAN será entrenado utilizando el mismo conjunto de variables base empleado durante la optimización anterior.

Las variables derivadas determinísticamente no serán sintetizadas directamente. En su lugar, serán reconstruidas después de la generación a partir de las variables base correspondientes. Esta decisión evita exigir al sintetizador que aprenda relaciones matemáticas que pueden preservarse exactamente mediante reglas deterministas.

Asimismo, se conservará la clasificación semántica de las variables identificada durante el diagnóstico anterior, diferenciando adecuadamente variables categóricas, numéricas e identificadores.

Sin embargo, no se trasladarán automáticamente a CTGAN todas las estrategias específicas utilizadas para corregir Gaussian Copula. En particular, inicialmente no se realizará generación separada por clase TARGET. Esto permitirá evaluar la capacidad propia de CTGAN para representar el desbalance de clases y las dependencias entre TARGET y las variables predictoras.

En caso de identificarse posteriormente una pérdida significativa de estas relaciones, cualquier estrategia adicional será documentada como una nueva iteración experimental.

#### 6.4.3 Preparación del dataset base

In [ ]:
# =========================================================
# 6.4.3 Preparación del dataset base para CTGAN
# =========================================================

derived_cols = [
    "AGE",
    "EMPLOYMENT_YEARS",
    "CREDIT_INCOME_RATIO",
    "ANNUITY_INCOME_RATIO",
    "CREDIT_GOODS_RATIO",
    "INCOME_PER_FAMILY",
    "CHILDREN_RATIO",
    "CREDIT_TERM_EST"
]

df_base_ctgan = df_real.drop(columns=derived_cols).copy()

print("Dataset real completo:", df_real.shape)
print("Dataset base CTGAN:", df_base_ctgan.shape)

print("\nVariables excluidas de la síntesis:")
for col in derived_cols:
    print("-", col)

Dataset real completo: (307507, 41)
Dataset base CTGAN: (307507, 33)

Variables excluidas de la síntesis:
- AGE
- EMPLOYMENT_YEARS
- CREDIT_INCOME_RATIO
- ANNUITY_INCOME_RATIO
- CREDIT_GOODS_RATIO
- INCOME_PER_FAMILY
- CHILDREN_RATIO
- CREDIT_TERM_EST


#### 6.4.4 Verificación de las variables

In [ ]:
#Antes de crear metadata
print("Columnas utilizadas por CTGAN:")
print(df_base_ctgan.columns.tolist())

print("\nNúmero de variables:")
print(df_base_ctgan.shape[1])

print("\nTARGET:")
print(df_base_ctgan["TARGET"].value_counts())
print()
print(df_base_ctgan["TARGET"].value_counts(normalize=True))

Columnas utilizadas por CTGAN:
['TARGET', 'SK_ID_CURR', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'AMT_REQ_CREDIT_BUREAU_YEAR', 'OBS_30_CNT_SOCIAL_CIRCLE', 'DEF_30_CNT_SOCIAL_CIRCLE', 'OBS_60_CNT_SOCIAL_CIRCLE', 'DEF_60_CNT_SOCIAL_CIRCLE', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'DAYS_BIRTH', 'CODE_GENDER', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS', 'NAME_FAMILY_STATUS', 'NAME_EDUCATION_TYPE', 'NAME_HOUSING_TYPE', 'DAYS_EMPLOYED', 'NAME_INCOME_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 'FLAG_OWN_CAR', 'OWN_CAR_AGE', 'FLAG_OWN_REALTY', 'NAME_CONTRACT_TYPE', 'NAME_TYPE_SUITE', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START', 'HAS_CAR_AGE']

Número de variables:
33

TARGET:
TARGET
0    282682
1     24825
Name: count, dtype: int64

TARGET
0    0.91927
1    0.08073
Name: proportion, dtype: float64


#### 6.4.5 Metadata de CTGAN

In [ ]:
%pip install sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.4/215.4 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 103.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.5/75.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.7/213.7 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 10.1 MB/s eta 0:00:00


In [ ]:
# Aquí sí reutilizamos la experiencia de V2.1.
from sdv.metadata import SingleTableMetadata

metadata_ctgan = SingleTableMetadata()

metadata_ctgan.detect_from_dataframe(
    data=df_base_ctgan
)

In [ ]:
#Y corregimos explícitamente los tipos relevantes:
metadata_ctgan.update_column(
    column_name="TARGET",
    sdtype="categorical"
)

metadata_ctgan.update_column(
    column_name="SK_ID_CURR",
    sdtype="id"
)

categorical_count_cols = [
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",
    "DEF_30_CNT_SOCIAL_CIRCLE",
    "DEF_60_CNT_SOCIAL_CIRCLE"
]

for col in categorical_count_cols:
    metadata_ctgan.update_column(
        column_name=col,
        sdtype="categorical"
    )

Las dos variables:

OBS_30_CNT_SOCIAL_CIRCLE
OBS_60_CNT_SOCIAL_CIRCLE

las dejamos numéricas, igual que en V2.1.

In [ ]:
#Comprobacion
metadata_dict = metadata_ctgan.to_dict()

for col in [
    "TARGET",
    "SK_ID_CURR",
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",
    "DEF_30_CNT_SOCIAL_CIRCLE",
    "DEF_60_CNT_SOCIAL_CIRCLE",
    "OBS_30_CNT_SOCIAL_CIRCLE",
    "OBS_60_CNT_SOCIAL_CIRCLE"
]:
    print(
        f"{col:30s} -> "
        f"{metadata_dict['columns'][col]['sdtype']}"
    )

TARGET                         -> categorical
SK_ID_CURR                     -> id
CNT_CHILDREN                   -> categorical
CNT_FAM_MEMBERS                -> categorical
DEF_30_CNT_SOCIAL_CIRCLE       -> categorical
DEF_60_CNT_SOCIAL_CIRCLE       -> categorical
OBS_30_CNT_SOCIAL_CIRCLE       -> numerical
OBS_60_CNT_SOCIAL_CIRCLE       -> numerical


**Validación de la preparación**

El dataset base para CTGAN contiene 307 507 observaciones y 33 variables, manteniendo la distribución original de TARGET (91.93 % no incumplimiento y 8.07 % incumplimiento).

La configuración de metadata conserva el tratamiento semántico definido durante el diagnóstico de Gaussian Copula V2.1. Las variables derivadas quedan excluidas del proceso generativo y serán reconstruidas posteriormente.

Por tanto, el dataset se considera preparado para el entrenamiento de CTGAN.

#### 6.4.6 Configuración y entrenamiento de CTGAN

**NOTA** Se configura CTGAN utilizando el dataset base de 33 variables definido anteriormente.

A diferencia de Gaussian Copula, CTGAN utiliza una arquitectura generativa adversarial, por lo que su entrenamiento requiere un proceso iterativo basado en épocas.

Para la primera configuración experimental se establecen 300 épocas. El objetivo no es realizar todavía una optimización exhaustiva de hiperparámetros, sino obtener una primera versión representativa del sintetizador que pueda evaluarse bajo el mismo protocolo utilizado con Gaussian Copula V2.1.

Cualquier modificación posterior de la configuración deberá estar motivada por los resultados obtenidos durante la fase de evaluación, siguiendo el carácter iterativo de CRISP-DM.

In [ ]:
#Crear CTGAN
from sdv.single_table import CTGANSynthesizer

ctgan = CTGANSynthesizer(
    metadata=metadata_ctgan,
    epochs=300,
    verbose=True
)

ctgan

In [ ]:
#Antes del entrenamiento, se fija la semilla:
import random
import numpy as np
import torch

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA disponible: True
GPU: Tesla T4


CTGAN tiene generador y discriminador compitiendo entre sí. Por eso sus losses pueden fluctuar y un loss menor tampoco implica automáticamente mejores datos sintéticos.

Nuestra evaluación real vendrá después con:

Distribuciones
        ↓
Relaciones estructurales
        ↓
TARGET ↔ variables críticas
        ↓
TSTR
        ↓
TSTS

Eso es bastante más importante que mirar solamente el loss de la GAN.

In [ ]:
#Entrenamiento
import time

inicio = time.time()

ctgan.fit(df_base_ctgan)

fin = time.time()

tiempo_entrenamiento = fin - inicio

print(
    f"\nTiempo de entrenamiento: "
    f"{tiempo_entrenamiento / 60:.2f} minutos"
)

KeyboardInterrupt: 

#### 6.4.7 Guardar el modelo

Yo ya lo denominaría internamente CTGAN V1, porque si el diagnóstico posterior identifica una limitación tendremos trazabilidad:

Gaussian Copula
V1 → V2 → V2.1

CTGAN
V1 → ¿V1.1/V2? solamente si el diagnóstico lo justifica

Eso encaja muy bien con la lógica iterativa de CRISP-DM.

In [ ]:
ctgan.save(
    filepath="/content/drive/MyDrive/Maestria VIU/Tesis/Notebooks/TFM/06_Synthetic_Model_Comparison/ctgan_v1.pkl"
)

print("CTGAN V1 guardado correctamente.")

Conste computacion diferencia de los sintetizadores

| Método               |  Entrenamiento |
| -------------------- | -------------: |
| Gaussian Copula V2.1 |    mucho menor |
| CTGAN V1             | **155.50 min** |

Luego podremos incluirlo como criterio secundario en la comparación final.

¿Los losses finales parecen problemáticos?

Terminaste aproximadamente en:

Generator Loss      = -1.09
Discriminator Loss  = +0.12

No veo aquí una señal que nos obligue a descartar el entrenamiento.

Pero tampoco podemos concluir:

"CTGAN entrenó bien porque Generator Loss = -1.09."

En GANs, los losses no tienen una interpretación equivalente a una función de pérdida supervisada convencional. Lo que realmente decidirá si este entrenamiento fue útil será la calidad de las muestras que genere.

Por eso no entrenaría más epochs ahora. Ya invertiste unas 2 h 36 min y tenemos un CTGAN V1 perfectamente válido para evaluar.

Si V1 resulta malo, primero diagnosticamos por qué antes de gastar otras 2–3 horas cambiando epochs.


**NOTA**
Resultado del entrenamiento

CTGAN V1 fue entrenado durante 300 épocas utilizando aceleración GPU (Tesla T4).

El entrenamiento requirió aproximadamente **155.50 minutos**, evidenciando un coste computacional considerablemente superior al enfoque Gaussian Copula.

Los valores observados al finalizar el entrenamiento fueron aproximadamente:

- Generator Loss: -1.09
- Discriminator Loss: 0.12

Estos valores se registran únicamente como información del proceso de entrenamiento. La calidad del sintetizador no será determinada directamente a partir de las funciones de pérdida de la GAN, sino mediante la evaluación estadística y predictiva del dataset generado.

El modelo entrenado fue almacenado como `ctgan_v1.pkl` para garantizar la reproducibilidad del experimento y evitar la necesidad de repetir el entrenamiento.

In [ ]:
#LEctura del CTGAN entrenado
import pickle
ruta_archivo = '/content/drive/MyDrive/Maestria VIU/Tesis/Notebooks/TFM/06_Synthetic_Model_Comparison/ctgan_v1.pkl'

# Leer el archivo pickle en modo binario ('rb')
with open(ruta_archivo, 'rb') as archivo:
    ctgan = pickle.load(archivo)

# Confirmar que se cargó correctamente
print("¡Archivo cargado con éxito!")
print("Tipo de objeto:", type(ctgan))

¡Archivo cargado con éxito!
Tipo de objeto: <class 'sdv.single_table.ctgan.CTGANSynthesizer'>


#### 6.4.8 Generación del dataset sintético CTGAN V1



Una vez entrenado CTGAN, se genera un dataset sintético con el mismo número de observaciones que el dataset real.

En esta primera versión no se condicionará manualmente la generación respecto a TARGET. El objetivo es evaluar la capacidad propia de CTGAN para reproducir tanto el desbalance de la variable objetivo como las dependencias existentes entre TARGET y las variables predictoras.

Esta estrategia permite obtener una evaluación inicial del sintetizador sin introducir correcciones específicas derivadas de los resultados obtenidos previamente con Gaussian Copula.

Posteriormente se reconstruirán de forma determinista las ocho variables derivadas excluidas del entrenamiento.

##### 6.4.8.1 Generación de las 307,507 observaciones

In [ ]:
import time

n_synthetic = len(df_real)

inicio = time.time()

df_ctgan_v1_base = ctgan.sample(
    num_rows=n_synthetic
)

fin = time.time()

tiempo_generacion_ctgan = fin - inicio

print("Shape generado:", df_ctgan_v1_base.shape)
print(
    f"Tiempo de generación: "
    f"{tiempo_generacion_ctgan / 60:.2f} minutos"
)

Shape generado: (307507, 33)
Tiempo de generación: 0.50 minutos


##### 6.4.8.2 Primera inspección

In [ ]:
print(df_ctgan_v1_base.head())

print("\nShape:")
print(df_ctgan_v1_base.shape)

print("\nNulos:")
print(df_ctgan_v1_base.isnull().sum().sum())

print("\nColumnas:")
print(df_ctgan_v1_base.columns.tolist())

   TARGET  SK_ID_CURR  AMT_INCOME_TOTAL  AMT_CREDIT  AMT_ANNUITY  \
0       1    13460034         89912.727    241815.5       7240.8   
1       0     2493192        311392.240    307330.2      11155.1   
2       0     8589238         89558.471    287795.1      31712.8   
3       0    10313411        183391.973    528017.8      36103.3   
4       0     2525172        230233.188   1240624.8      44265.1   

   AMT_GOODS_PRICE  AMT_REQ_CREDIT_BUREAU_YEAR  OBS_30_CNT_SOCIAL_CIRCLE  \
0         141139.8                         1.0                       2.0   
1         172630.0                         2.0                       0.0   
2         287015.8                         1.0                       4.0   
3         458812.7                         1.0                       5.0   
4        1130113.4                         0.0                       7.0   

   DEF_30_CNT_SOCIAL_CIRCLE  OBS_60_CNT_SOCIAL_CIRCLE  \
0                       2.0                       2.0   
1                   

##### 6.4.8.3 ¿CTGAN aprendió el desbalance de TARGET?

In [ ]:
target_real = (
    df_real["TARGET"]
    .value_counts(normalize=True)
    .sort_index()
)

target_ctgan = (
    df_ctgan_v1_base["TARGET"]
    .value_counts(normalize=True)
    .sort_index()
)

comparacion_target = pd.DataFrame({
    "Real": target_real,
    "CTGAN_V1": target_ctgan
})

comparacion_target["Diferencia_abs"] = (
    comparacion_target["Real"] -
    comparacion_target["CTGAN_V1"]
).abs()

comparacion_target

,Real,CTGAN_V1,Diferencia_abs
TARGET,,,
0,0.91927,0.924916,0.005645
1,0.08073,0.075084,0.005645


In [ ]:
print("TARGET CTGAN V1 - frecuencias:")
print(df_ctgan_v1_base["TARGET"].value_counts())

print("\nTARGET CTGAN V1 - proporciones:")
print(
    df_ctgan_v1_base["TARGET"]
    .value_counts(normalize=True)
)

TARGET CTGAN V1 - frecuencias:
TARGET
0    284418
1     23089
Name: count, dtype: int64

TARGET CTGAN V1 - proporciones:
TARGET
0    0.924916
1    0.075084
Name: proportion, dtype: float64


##### 6.4.8.4 Reconstrucción de variables derivadas

In [ ]:
# Ahora aplicamos exactamente las mismas reglas utilizadas para Gaussian Copula.
df_ctgan_v1 = df_ctgan_v1_base.copy()

# Edad
df_ctgan_v1["AGE"] = (
    -df_ctgan_v1["DAYS_BIRTH"] / 365.25
)

# Antigüedad laboral
df_ctgan_v1["EMPLOYMENT_YEARS"] = (
    -df_ctgan_v1["DAYS_EMPLOYED"] / 365.25
)

# Ratios financieros
df_ctgan_v1["CREDIT_INCOME_RATIO"] = (
    df_ctgan_v1["AMT_CREDIT"] /
    df_ctgan_v1["AMT_INCOME_TOTAL"]
)

df_ctgan_v1["ANNUITY_INCOME_RATIO"] = (
    df_ctgan_v1["AMT_ANNUITY"] /
    df_ctgan_v1["AMT_INCOME_TOTAL"]
)

df_ctgan_v1["CREDIT_GOODS_RATIO"] = (
    df_ctgan_v1["AMT_CREDIT"] /
    df_ctgan_v1["AMT_GOODS_PRICE"]
)

df_ctgan_v1["INCOME_PER_FAMILY"] = (
    df_ctgan_v1["AMT_INCOME_TOTAL"] /
    df_ctgan_v1["CNT_FAM_MEMBERS"]
)

df_ctgan_v1["CHILDREN_RATIO"] = (
    df_ctgan_v1["CNT_CHILDREN"] /
    df_ctgan_v1["CNT_FAM_MEMBERS"]
)

df_ctgan_v1["CREDIT_TERM_EST"] = (
    df_ctgan_v1["AMT_CREDIT"] /
    df_ctgan_v1["AMT_ANNUITY"]
)

##### 6.4.8.5 Control de infinitos

In [ ]:
# Como tenemos divisiones, no debemos asumir que CTGAN nunca generará un denominador igual a cero.
derived_cols = [
    "AGE",
    "EMPLOYMENT_YEARS",
    "CREDIT_INCOME_RATIO",
    "ANNUITY_INCOME_RATIO",
    "CREDIT_GOODS_RATIO",
    "INCOME_PER_FAMILY",
    "CHILDREN_RATIO",
    "CREDIT_TERM_EST"
]

for col in derived_cols:
    n_nan = df_ctgan_v1[col].isna().sum()
    n_inf = np.isinf(df_ctgan_v1[col]).sum()

    print(
        f"{col:25s} | "
        f"NaN: {n_nan:6d} | "
        f"Inf: {n_inf:6d}"
    )

AGE                       | NaN:      0 | Inf:      0
EMPLOYMENT_YEARS          | NaN:      0 | Inf:      0
CREDIT_INCOME_RATIO       | NaN:      0 | Inf:      0
ANNUITY_INCOME_RATIO      | NaN:      0 | Inf:      0
CREDIT_GOODS_RATIO        | NaN:      0 | Inf:      0
INCOME_PER_FAMILY         | NaN:      0 | Inf:      0
CHILDREN_RATIO            | NaN:      0 | Inf:      0
CREDIT_TERM_EST           | NaN:      0 | Inf:      0


##### 6.4.8.6 Restaurar el mismo orden de columnas

In [ ]:
df_ctgan_v1 = df_ctgan_v1[df_real.columns]

print("Real:", df_real.shape)
print("CTGAN V1:", df_ctgan_v1.shape)

print(
    "¿Mismas columnas y mismo orden?:",
    df_ctgan_v1.columns.tolist()
    == df_real.columns.tolist()
)

Real: (307507, 41)
CTGAN V1: (307507, 41)
¿Mismas columnas y mismo orden?: True


##### 6.4.8.7 Validación estructural básica

In [ ]:
validacion_estructura = pd.DataFrame({
    "Dataset": [
        "Real",
        "CTGAN V1"
    ],
    "Filas": [
        len(df_real),
        len(df_ctgan_v1)
    ],
    "Columnas": [
        df_real.shape[1],
        df_ctgan_v1.shape[1]
    ],
    "Nulos": [
        df_real.isna().sum().sum(),
        df_ctgan_v1.isna().sum().sum()
    ]
})

validacion_estructura

,Dataset,Filas,Columnas,Nulos
0,Real,307507,41,0
1,CTGAN V1,307507,41,0


In [ ]:
print(
    "Duplicados CTGAN V1:",
    df_ctgan_v1.duplicated().sum()
)

print(
    "TARGET únicos:",
    sorted(df_ctgan_v1["TARGET"].unique())
)

Duplicados CTGAN V1: 0
TARGET únicos: [np.int64(0), np.int64(1)]


**ORDEN**

CTGAN entrenado
      ↓
Generación V1             ← estamos aquí
      ↓
Validación estructural
      ↓
Fidelidad estadística
      ↓
Relaciones TARGET-X
      ↓
Dependencias críticas
      ↓
TSTR
      ↓
TSTS
      ↓
Comparación Gaussian V2.1 vs CTGAN V1

**Resultado de la generación de CTGAN V1**

CTGAN V1 generó 307 507 observaciones sintéticas. Tras reconstruir las ocho variables derivadas, el dataset resultante contiene las mismas 41 variables y mantiene el mismo orden estructural que el dataset real.

No se detectaron valores nulos, infinitos ni registros completamente duplicados.

CTGAN reprodujo además de forma aproximada el desbalance original de TARGET sin utilizar generación condicionada explícitamente. La proporción de incumplimiento fue del 8.073 % en los datos reales y del 7.514 % en CTGAN V1, lo que representa una diferencia absoluta de aproximadamente 0.559 puntos porcentuales.

Dado que la distribución de TARGET se conserva razonablemente y no se han identificado inconsistencias estructurales en esta etapa, no se aplicarán correcciones adicionales antes de evaluar la fidelidad estadística y predictiva del dataset.

#### 6.4.9 Validación estadística de CTGAN V1

Aquí quiero que repliquemos deliberadamente parte del diagnóstico que utilizamos con Gaussian Copula. Así tendremos comparabilidad directa:

distribuciones → relaciones estructurales → TARGET-X → utilidad predictiva.

##### 6.4.9.1 Fidelidad de variables críticas

In [ ]:
# =========================================================
# 6.4.9.1 Comparación descriptiva de variables críticas
# =========================================================

critical_numeric = [
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "CREDIT_GOODS_RATIO",
    "CREDIT_TERM_EST"
]

rows = []

for col in critical_numeric:

    rows.append({
        "Variable": col,

        "Media_Real": df_real[col].mean(),
        "Media_CTGAN": df_ctgan_v1[col].mean(),

        "Std_Real": df_real[col].std(),
        "Std_CTGAN": df_ctgan_v1[col].std(),

        "Mediana_Real": df_real[col].median(),
        "Mediana_CTGAN": df_ctgan_v1[col].median()
    })

comparacion_descriptiva = pd.DataFrame(rows)

comparacion_descriptiva.round(4)

,Variable,Media_Real,Media_CTGAN,Std_Real,Std_CTGAN,Mediana_Real,Mediana_CTGAN
0,EXT_SOURCE_1,0.5043,0.4974,0.1394,0.1397,0.5060,0.5060
1,EXT_SOURCE_2,0.5145,0.5225,0.1909,0.1683,0.5660,0.5639
2,EXT_SOURCE_3,0.5157,0.5305,0.1747,0.1848,0.5353,0.5365
3,DAYS_BIRTH,-16037.0273,-15419.4658,4363.9824,4235.6525,-15750.0000,-14613.0000
4,DAYS_EMPLOYED,-2251.5822,-2123.4761,2136.1601,1869.5481,-1648.0000,-1650.0000
5,AMT_CREDIT,599028.5967,622932.9482,402492.6019,424009.3982,513531.0000,540506.1000
6,AMT_ANNUITY,27108.5807,26751.2708,14493.5221,14940.9267,24903.0000,25100.6000
7,AMT_GOODS_PRICE,538317.8090,502979.0601,369289.7661,350589.2714,450000.0000,450024.5000
8,CREDIT_GOODS_RATIO,1.1225,1.3999,0.1255,0.9475,1.1188,1.1974
9,CREDIT_TERM_EST,21.6124,24.0730,7.8242,12.6627,20.0000,21.8432


No vamos a decidir calidad solamente por medias y desviaciones, pero nos permite detectar deformaciones evidentes.

##### 6.4.9.2 Variables discretas problemáticas en Gaussian Copula

In [ ]:
#Recordemos que Gaussian V2 tenía problemas muy grandes con CNT_CHILDREN y CNT_FAM_MEMBERS. Veamos qué hizo CTGAN.
def comparar_distribucion(df_real, df_synth, columna):

    real = (
        df_real[columna]
        .value_counts(normalize=True)
        .sort_index()
    )

    synth = (
        df_synth[columna]
        .value_counts(normalize=True)
        .sort_index()
    )

    comp = pd.concat(
        [real, synth],
        axis=1
    )

    comp.columns = [
        "Real",
        "CTGAN_V1"
    ]

    comp["Diferencia_abs"] = (
        comp["Real"] -
        comp["CTGAN_V1"]
    ).abs()

    return comp


comparar_distribucion(
    df_real,
    df_ctgan_v1,
    "CNT_CHILDREN"
)

,Real,CTGAN_V1,Diferencia_abs
CNT_CHILDREN,,,
0,0.700371,0.625212,0.075159
1,0.198753,0.212951,0.014198
2,0.086983,0.094492,0.007509
3,0.012088,0.024269,0.012182
4,0.001395,0.011515,0.010120
5,0.000273,0.007977,0.007704
6,0.000068,0.005775,0.005707
7,0.000023,0.003623,0.003600
8,0.000007,0.001652,0.001645


In [ ]:
comparar_distribucion(
    df_real,
    df_ctgan_v1,
    "CNT_FAM_MEMBERS"
)

,Real,CTGAN_V1,Diferencia_abs
CNT_FAM_MEMBERS,,,
1.0,0.220636,0.240388,0.019752
2.0,0.514970,0.421467,0.093504
3.0,0.171053,0.177967,0.006914
4.0,0.080310,0.092531,0.012221
5.0,0.011310,0.027687,0.016377
6.0,0.001327,0.010845,0.009518
7.0,0.000263,0.007561,0.007297
8.0,0.000065,0.004699,0.004634
9.0,0.000020,0.003119,0.003099


In [ ]:
comparar_distribucion(
    df_real,
    df_ctgan_v1,
    "DEF_30_CNT_SOCIAL_CIRCLE"
)

,Real,CTGAN_V1,Diferencia_abs
DEF_30_CNT_SOCIAL_CIRCLE,,,
0.0,0.885648,0.800349,0.085299
1.0,0.092118,0.109497,0.017378
2.0,0.017310,0.037417,0.020107
3.0,0.003876,0.022747,0.018871
4.0,0.000819,0.011479,0.010660
5.0,0.000182,0.009801,0.009619
6.0,0.000036,0.004377,0.004341
7.0,0.000003,0.001522,0.001519
8.0,0.000003,0.001398,0.001395


In [ ]:
comparar_distribucion(
    df_real,
    df_ctgan_v1,
    "DEF_60_CNT_SOCIAL_CIRCLE"
)

,Real,CTGAN_V1,Diferencia_abs
DEF_60_CNT_SOCIAL_CIRCLE,,,
0.0,0.916207,0.848582,0.067624
1.0,0.071023,0.085614,0.014592
2.0,0.010309,0.025651,0.015343
3.0,0.001945,0.015027,0.013083
4.0,0.000436,0.013193,0.012757
5.0,0.000065,0.005961,0.005896
6.0,0.000010,0.003151,0.003141
7.0,0.000003,0.001210,0.001206
24.0,0.000003,0.001610,0.001606


Esto nos permitirá comparar posteriormente:

Real
    ↓
Gaussian V2.1
    ↓
CTGAN V1

##### 6.4.9.3 Dependencias estructurales críticas

In [ ]:
# =========================================================
# 6.4.9.3 Dependencias estructurales
# =========================================================

structural_pairs = [
    (
        "CNT_CHILDREN",
        "CNT_FAM_MEMBERS"
    ),
    (
        "DEF_30_CNT_SOCIAL_CIRCLE",
        "DEF_60_CNT_SOCIAL_CIRCLE"
    )
]

rows = []

for var1, var2 in structural_pairs:

    corr_real = df_real[
        [var1, var2]
    ].corr(method="spearman").iloc[0, 1]

    corr_ctgan = df_ctgan_v1[
        [var1, var2]
    ].corr(method="spearman").iloc[0, 1]

    rows.append({
        "Relacion": f"{var1} ↔ {var2}",
        "Real": corr_real,
        "CTGAN_V1": corr_ctgan,
        "Error_abs_CTGAN": abs(
            corr_real - corr_ctgan
        )
    })

comparacion_estructural_ctgan = pd.DataFrame(rows)

comparacion_estructural_ctgan.round(6)

,Relacion,Real,CTGAN_V1,Error_abs_CTGAN
0,CNT_CHILDREN ↔ CNT_FAM_MEMBERS,0.811554,0.791256,0.020298
1,DEF_30_CNT_SOCIAL_CIRCLE ↔ DEF_60_CNT_SOCIAL_C...,0.845149,0.761643,0.083507


Para Gaussian Copula V2.1 habíamos obtenido aproximadamente:

| Relación          |   Real | Gaussian V2.1 |
| ----------------- | -----: | ------------: |
| CHILDREN ↔ FAMILY | 0.8792 |        0.4784 |
| DEF_30 ↔ DEF_60   | 0.8605 |        0.0593 |


La segunda relación era especialmente mala.

Por tanto, si CTGAN nos devuelve algo como:

DEF_30 ↔ DEF_60 = 0.7

sería un resultado muy relevante a favor de su capacidad multivariante.

##### 6.4.9.4 Relaciones críticas con TARGET

In [ ]:
#Esta es probablemente la comprobación estadística más importante antes de TSTR.
#Usaremos exactamente las siete variables que ya seguimos durante V2 → V2.1:
critical_target_vars = [
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "EXT_SOURCE_1",
    "DAYS_BIRTH",
    "CREDIT_GOODS_RATIO",
    "DAYS_EMPLOYED",
    "CREDIT_TERM_EST"
]

rows = []

for col in critical_target_vars:

    corr_real = df_real[
        ["TARGET", col]
    ].corr(method="spearman").iloc[0, 1]

    corr_ctgan = df_ctgan_v1[
        ["TARGET", col]
    ].corr(method="spearman").iloc[0, 1]

    rows.append({
        "Variable": col,
        "Real": corr_real,
        "CTGAN_V1": corr_ctgan,
        "Error_abs_CTGAN": abs(
            corr_real - corr_ctgan
        )
    })

target_corr_ctgan = pd.DataFrame(rows)

target_corr_ctgan.round(6)

,Variable,Real,CTGAN_V1,Error_abs_CTGAN
0,EXT_SOURCE_2,-0.147133,-0.093200,0.053933
1,EXT_SOURCE_3,-0.141501,-0.127644,0.013856
2,EXT_SOURCE_1,-0.085389,-0.114066,0.028676
3,DAYS_BIRTH,0.078331,0.127840,0.049509
4,CREDIT_GOODS_RATIO,0.066684,-0.001018,0.067702
5,DAYS_EMPLOYED,0.075036,0.075180,0.000144
6,CREDIT_TERM_EST,-0.030262,-0.005587,0.024676


In [ ]:
#Y calculamos el mismo MAE que utilizamos anteriormente:
mae_target_ctgan = (
    target_corr_ctgan["Error_abs_CTGAN"].mean()
)

print(
    f"MAE correlaciones críticas CTGAN V1: "
    f"{mae_target_ctgan:.6f}"
)

MAE correlaciones críticas CTGAN V1: 0.034071


Nuestros valores anteriores eran:

Gaussian V2       MAE = 0.066496
Gaussian V2.1     MAE = 0.045950
CTGAN V1          MAE = ???

Si CTGAN obtiene, por ejemplo:

0.030

sería evidencia de una mejor preservación de las asociaciones críticas con TARGET.

Si obtiene:

0.070

significaría que, a pesar de ser un modelo más sofisticado, está conservando peor esas relaciones.

Y eso es exactamente lo que queremos descubrir empíricamente.

**La hipótesis antes de TSTR**
Dada la mejor conservación global de determinadas relaciones multivariantes y la reducción del MAE de las asociaciones críticas con TARGET, se espera que CTGAN V1 pueda alcanzar una utilidad TSTR competitiva o superior a Gaussian Copula V2.1. Sin embargo, las distorsiones detectadas en algunas distribuciones y relaciones derivadas podrían limitar dicha mejora.

_______________________________________________________________________


**Diagnóstico estadístico de CTGAN V1**

La evaluación de CTGAN V1 muestra un comportamiento mixto en términos de fidelidad estadística.

Por un lado, determinadas distribuciones marginales presentan desviaciones respecto al dataset real. Estas diferencias son especialmente visibles en algunas variables discretas de baja frecuencia y en variables derivadas como CREDIT_GOODS_RATIO, cuya dispersión aumenta considerablemente como consecuencia de diferencias en la relación conjunta entre sus variables base.

Por otro lado, CTGAN presenta una conservación notablemente superior de determinadas dependencias multivariantes. La correlación entre CNT_CHILDREN y CNT_FAM_MEMBERS se mantiene próxima a la observada en los datos reales, al igual que la relación entre DEF_30_CNT_SOCIAL_CIRCLE y DEF_60_CNT_SOCIAL_CIRCLE.

Respecto a las asociaciones entre TARGET y las variables predictivamente relevantes analizadas, CTGAN V1 obtiene un error absoluto medio (MAE) de 0.034591. Este resultado es inferior al observado anteriormente para Gaussian Copula V2.1, aunque la mejora no es uniforme entre todas las variables.

En particular, CTGAN reproduce con elevada precisión determinadas asociaciones, como DAYS_EMPLOYED, mientras que atenúa considerablemente otras, como CREDIT_GOODS_RATIO, y sobreestima algunas relaciones como DAYS_BIRTH.

Estos resultados sugieren que CTGAN posee mayor capacidad para representar determinadas dependencias multivariantes complejas, aunque esta flexibilidad no garantiza una reproducción exacta de todas las distribuciones marginales ni de todas las relaciones predictivas.

A partir de este diagnóstico se plantea la hipótesis de que la mejor conservación global de determinadas relaciones multivariantes podría traducirse en una utilidad TSTR competitiva o superior a Gaussian Copula V2.1. Esta hipótesis será contrastada mediante evaluación predictiva sobre el mismo conjunto real de prueba.


#### 6.4.10 Evaluación TSTR de CTGAN V1

Ahora sí llegamos a la prueba principal.

No vamos a cambiar absolutamente nada del LightGBM.

La comparación debe ser:

TRTR Real                  0.765600
Gaussian V2.1 → Real       0.726674
CTGAN V1 → Real            ???

In [ ]:
#Preparación - Utilizamos TARGET como objetivo y eliminamos el identificador
X_ctgan = df_ctgan_v1.drop(
    columns=["TARGET", "SK_ID_CURR"]
)

y_ctgan = df_ctgan_v1["TARGET"]

print("X CTGAN:", X_ctgan.shape)
print("y CTGAN:", y_ctgan.shape)

print("\nDistribución TARGET:")
print(y_ctgan.value_counts(normalize=True))

X CTGAN: (307507, 39)
y CTGAN: (307507,)

Distribución TARGET:
TARGET
0    0.924916
1    0.075084
Name: proportion, dtype: float64


In [ ]:
#Preprocesamiento - Muy importante: el encoder se ajusta exclusivamente con CTGAN.
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_cols_ctgan = (
    X_ctgan
    .select_dtypes(include=["object", "category"])
    .columns
    .tolist()
)

print(
    "Variables categóricas:",
    len(categorical_cols_ctgan)
)

preprocessor_ctgan = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_cols_ctgan
        )
    ],
    remainder="passthrough"
)

Variables categóricas: 12


In [ ]:
#Ajustamos solamente el syntetic training
X_ctgan_enc = preprocessor_ctgan.fit_transform(
    X_ctgan
)

In [ ]:
#Leer el X_test_real que usamos con gaussian copula
ruta = "/content/drive/MyDrive/Maestria VIU/Tesis/Notebooks/TFM//05_Synthetic_Data_Optimization/X_test_real.parquet"
X_test_real = pd.read_parquet(ruta)

In [ ]:
#Y transformamos el mismo X_test_real que usamos para Gaussian Copula V2.1
X_test_real_ctgan_enc = (
    preprocessor_ctgan.transform(
        X_test_real.drop(
            columns=["SK_ID_CURR"],
            errors="ignore"
        )
    )
)

In [ ]:
#Comprobamos
print(
    "CTGAN train:",
    X_ctgan_enc.shape
)

print(
    "Real test:",
    X_test_real_ctgan_enc.shape
)

CTGAN train: (307507, 150)
Real test: (61502, 150)


In [ ]:
assert (
    X_ctgan_enc.shape[1]
    ==
    X_test_real_ctgan_enc.shape[1]
)

print(
    "Número de features compatible:",
    X_ctgan_enc.shape[1]
)

Número de features compatible: 150


In [ ]:
# Entrenar exactamente el mismo LightGBM
modelo_tstr_ctgan = crear_modelo_diagnostico()

modelo_tstr_ctgan.fit(
    X_ctgan_enc,
    y_ctgan
)

LGBMClassifier(learning_rate=0.05, n_estimators=200, n_jobs=-1, random_state=42,
               verbosity=-1)

In [ ]:
#Leer el y_test_real que usamos con gaussian copula
ruta = "/content/drive/MyDrive/Maestria VIU/Tesis/Notebooks/TFM//05_Synthetic_Data_Optimization/y_test_real.parquet"
y_test_real = pd.read_parquet(ruta)

In [ ]:
from sklearn.metrics import roc_auc_score

proba_tstr_ctgan = (
    modelo_tstr_ctgan.predict_proba(
        X_test_real_ctgan_enc
    )[:, 1]
)

auc_tstr_ctgan = roc_auc_score(
    y_test_real,
    proba_tstr_ctgan
)

print(
    f"TSTR ROC-AUC CTGAN V1: "
    f"{auc_tstr_ctgan:.6f}"
)

TSTR ROC-AUC CTGAN V1: 0.699521


In [ ]:
mejora_vs_gc = (
    auc_tstr_ctgan
    - AUC_TSTR_GC_V21
)

print(
    f"Diferencia CTGAN - Gaussian V2.1: "
    f"{mejora_vs_gc:+.6f}"
)

Diferencia CTGAN - Gaussian V2.1: -0.027153


Este resultado es especialmente valioso para el TFM porque contradice nuestra hipótesis inicial: CTGAN V1 conservó mejor varias dependencias y obtuvo un MAE de correlaciones críticas menor, pero eso no se tradujo en una mejor capacidad de transferencia predictiva.

Los resultados quedan así:

| Entrenamiento        | Test |      ROC-AUC | Gap vs TRTR |
| -------------------- | ---- | -----------: | ----------: |
| Real                 | Real | **0.765600** |           0 |
| Gaussian Copula V2.1 | Real | **0.726674** |    0.038926 |
| CTGAN V1             | Real | **0.700637** |    0.064963 |

CTGAN queda 0.026037 puntos de ROC-AUC por debajo de Gaussian Copula V2.1. Además, alcanza aproximadamente el 91.51% del ROC-AUC del baseline real, frente al 94.92% de Gaussian Copula V2.1.

Esto nos enseña algo metodológicamente importante: preservar mejor algunas correlaciones individuales o ciertas dependencias estructurales no garantiza preservar mejor la estructura predictiva conjunta necesaria para generalizar sobre datos reales.

**Interpretación del TSTR de CTGAN V1**

CTGAN V1 obtuvo un ROC-AUC TSTR de **0.700637**, frente a **0.726674** obtenido por Gaussian Copula V2.1 y **0.765600** del baseline TRTR.

Por tanto, CTGAN V1 presenta un gap de 0.064963 respecto al baseline real y un rendimiento 0.026037 inferior a Gaussian Copula V2.1.

Este resultado resulta especialmente relevante dado que el diagnóstico estadístico previo había mostrado una mejor conservación de determinadas dependencias multivariantes y un menor error medio en las correlaciones críticas con TARGET.

En consecuencia, la hipótesis de que dicha mejora estadística se traduciría directamente en una mayor utilidad TSTR no queda respaldada por los resultados.

La evidencia sugiere que la conservación de correlaciones bivariadas o de determinadas relaciones estructurales no es suficiente para garantizar la preservación de la estructura predictiva multivariante utilizada por el modelo de clasificación.

Asimismo, CTGAN V1 presenta distorsiones en determinadas distribuciones y relaciones derivadas, como CREDIT_GOODS_RATIO, que pueden contribuir a la pérdida de capacidad de transferencia hacia datos reales.

Gaussian Copula V2.1 continúa, por tanto, siendo el mejor sintetizador evaluado hasta el momento según el criterio predictivo principal TSTR.


**NOTA** Mayor fidelidad en determinadas relaciones estadísticas no implica necesariamente mayor utilidad predictiva.

Ese es un resultado bastante bueno científicamente porque evita una conclusión simplista de “CTGAN es más avanzado, entonces necesariamente genera mejores datos”

No hay contradicción. El MAE mide solamente un subconjunto concreto de relaciones bivariadas mediante Spearman. El ROC-AUC TSTR está midiendo si toda la estructura generada permite aprender patrones que luego funcionan en la población real.

Además, CTGAN mostró señales que podrían estar afectándolo:

categorías raras artificialmente sobrerrepresentadas;
CNT_CHILDREN y CNT_FAM_MEMBERS marginalmente más distorsionadas;
dispersión exagerada en CREDIT_GOODS_RATIO;
desaparición prácticamente completa de su asociación con TARGET;
algunas asociaciones con TARGET sobreestimadas en vez de reproducidas.

Por ahora debemos hablar de posibles explicaciones, no afirmar cuál de ellas causó la caída del AUC.

No optimicemos CTGAN todavía

Esto es importante.

Podríamos sentir la tentación de decir:

“Entonces condicionemos TARGET, cambiemos epochs y entrenemos CTGAN V2.”

Yo todavía no lo haría.

Primero tenemos que terminar el mismo protocolo:

TSTS de CTGAN V1.

Porque puede ocurrir algo muy interesante:

Caso A
TSTR = 0.7006
TSTS = 0.75

CTGAN generaría un universo sintético altamente predictivo internamente, pero diferente del real.

Caso B
TSTR = 0.7006
TSTS = 0.60

Entonces CTGAN también tendría baja utilidad interna.

Caso C
TSTR = 0.7006
TSTS ≈ 0.70

Tendríamos otra situación diferente: señal predictiva relativamente consistente entre su mundo sintético y el real, pero de menor calidad global.

Así que necesitamos TSTS antes de diagnosticar.

#### 6.4.11 Evaluación TSTS de CTGAN V1

Como complemento al escenario TSTR, se evalúa la utilidad predictiva interna del dataset generado mediante CTGAN V1.

Para ello, el dataset sintético se divide en conjuntos de entrenamiento y prueba utilizando una partición estratificada 80/20.

El preprocesamiento se ajusta exclusivamente utilizando el conjunto sintético de entrenamiento y posteriormente se aplica al conjunto sintético de prueba.

Se mantiene exactamente la misma configuración de LightGBM utilizada en los experimentos anteriores.

Este escenario permite analizar qué capacidad predictiva observaría un usuario que trabajara exclusivamente con el dataset sintético generado por CTGAN.

In [ ]:
from sklearn.model_selection import train_test_split

X_train_ctgan, X_test_ctgan, y_train_ctgan, y_test_ctgan = (
    train_test_split(
        X_ctgan,
        y_ctgan,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y_ctgan
    )
)

print("Train CTGAN:", X_train_ctgan.shape)
print("Test CTGAN:", X_test_ctgan.shape)

print("\nTARGET train:")
print(y_train_ctgan.value_counts(normalize=True))

print("\nTARGET test:")
print(y_test_ctgan.value_counts(normalize=True))

Train CTGAN: (246005, 39)
Test CTGAN: (61502, 39)

TARGET train:
TARGET
0    0.924916
1    0.075084
Name: proportion, dtype: float64

TARGET test:
TARGET
0    0.924913
1    0.075087
Name: proportion, dtype: float64


In [ ]:
#Encoder TSTS independiente
#No reutilizamos el encoder del TSTR. Ajustamos uno utilizando solamente X_train_ctgan.
categorical_cols_tsts_ctgan = (
    X_train_ctgan
    .select_dtypes(include=["object", "category"])
    .columns
    .tolist()
)

preprocessor_tsts_ctgan = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_cols_tsts_ctgan
        )
    ],
    remainder="passthrough"
)

In [ ]:
X_train_ctgan_enc = (
    preprocessor_tsts_ctgan.fit_transform(
        X_train_ctgan
    )
)

X_test_ctgan_enc = (
    preprocessor_tsts_ctgan.transform(
        X_test_ctgan
    )
)

print(
    "Synthetic train:",
    X_train_ctgan_enc.shape
)

print(
    "Synthetic test:",
    X_test_ctgan_enc.shape
)

assert (
    X_train_ctgan_enc.shape[1]
    ==
    X_test_ctgan_enc.shape[1]
)

Synthetic train: (246005, 150)
Synthetic test: (61502, 150)


In [ ]:
#LightGBM
modelo_tsts_ctgan = crear_modelo_diagnostico()

modelo_tsts_ctgan.fit(
    X_train_ctgan_enc,
    y_train_ctgan
)

LGBMClassifier(learning_rate=0.05, n_estimators=200, n_jobs=-1, random_state=42,
               verbosity=-1)

In [ ]:
proba_tsts_ctgan = (
    modelo_tsts_ctgan.predict_proba(
        X_test_ctgan_enc
    )[:, 1]
)

auc_tsts_ctgan = roc_auc_score(
    y_test_ctgan,
    proba_tsts_ctgan
)

print(
    f"TSTS ROC-AUC CTGAN V1: "
    f"{auc_tsts_ctgan:.6f}"
)

TSTS ROC-AUC CTGAN V1: 0.758667


In [ ]:
#Comparación provisional completa
comparacion_ctgan_gc = pd.DataFrame({
    "Sintetizador": [
        "Gaussian Copula V2.1",
        "CTGAN V1"
    ],

    "TSTR_ROC_AUC": [
        AUC_TSTR_GC_V21,
        auc_tstr_ctgan
    ],

    "TSTS_ROC_AUC": [
        AUC_TSTS_GC_V21,
        auc_tsts_ctgan
    ]
})

comparacion_ctgan_gc[
    "Gap_TRTR_TSTR"
] = (
    AUC_TRTR_BASELINE
    - comparacion_ctgan_gc["TSTR_ROC_AUC"]
)

comparacion_ctgan_gc[
    "Gap_TRTR_TSTS"
] = (
    AUC_TRTR_BASELINE
    - comparacion_ctgan_gc["TSTS_ROC_AUC"]
)

comparacion_ctgan_gc[
    "Gap_TSTR_TSTS"
] = (
    comparacion_ctgan_gc["TSTR_ROC_AUC"]
    - comparacion_ctgan_gc["TSTS_ROC_AUC"]
)

comparacion_ctgan_gc[
    "TSTR_Relativo_pct"
] = (
    comparacion_ctgan_gc["TSTR_ROC_AUC"]
    / AUC_TRTR_BASELINE
    * 100
)

comparacion_ctgan_gc.round(4)

,Sintetizador,TSTR_ROC_AUC,TSTS_ROC_AUC,Gap_TRTR_TSTR,Gap_TRTR_TSTS,Gap_TSTR_TSTS,TSTR_Relativo_pct
0,Gaussian Copula V2.1,0.7267,0.6218,0.0389,0.1438,0.1049,94.9156
1,CTGAN V1,0.6995,0.7587,0.0661,0.0069,-0.0591,91.3690


In [ ]:
print("auc_tstr_ctgan actual:", auc_tstr_ctgan)
print("AUC Gaussian:", AUC_TSTR_GC_V21)
print("AUC TRTR:", AUC_TRTR_BASELINE)

auc_tstr_ctgan actual: 0.6995208139413948
AUC Gaussian: 0.726674
AUC TRTR: 0.7656


In [ ]:
print(f"TSTR CTGAN almacenado: {auc_tstr_ctgan:.10f}")
print(f"TSTS CTGAN almacenado: {auc_tsts_ctgan:.10f}")

print(
    "Gap TSTR-TSTS:",
    auc_tstr_ctgan - auc_tsts_ctgan
)

TSTR CTGAN almacenado: 0.6995208139
TSTS CTGAN almacenado: 0.7586674989
Gap TSTR-TSTS: -0.05914668499552844


In [ ]:
# Resultados definitivos CTGAN V1

AUC_TSTR_CTGAN_V1 = 0.6995208139413948
AUC_TSTS_CTGAN_V1 = 0.7586674989

print(f"TSTR CTGAN V1: {AUC_TSTR_CTGAN_V1:.6f}")
print(f"TSTS CTGAN V1: {AUC_TSTS_CTGAN_V1:.6f}")

TSTR CTGAN V1: 0.699521
TSTS CTGAN V1: 0.758667


#### 6.4.12 Conclusiones de CTGAN V1

La evaluación de CTGAN V1 evidencia un comportamiento diferente al observado con Gaussian Copula V2.1.

Desde el punto de vista estructural, CTGAN fue capaz de generar 307 507 observaciones sin valores nulos, infinitos ni registros completamente duplicados. Asimismo, reprodujo razonablemente el desbalance de TARGET sin necesidad de utilizar generación condicionada explícita.

El análisis estadístico mostró resultados mixtos. CTGAN presentó desviaciones en determinadas distribuciones marginales y en algunas relaciones derivadas. Sin embargo, mostró una elevada capacidad para conservar determinadas dependencias multivariantes, destacando las relaciones entre CNT_CHILDREN y CNT_FAM_MEMBERS, así como entre DEF_30_CNT_SOCIAL_CIRCLE y DEF_60_CNT_SOCIAL_CIRCLE.

En las asociaciones críticas con TARGET, CTGAN V1 obtuvo un MAE de 0.034591, inferior al observado previamente para Gaussian Copula V2.1. No obstante, esta mayor fidelidad en el subconjunto de relaciones analizadas no se tradujo en una mayor capacidad de transferencia predictiva.

En el escenario TSTR, CTGAN V1 obtuvo un ROC-AUC de **0.699521**, inferior al **0.726674** alcanzado por Gaussian Copula V2.1. Por tanto, Gaussian Copula continúa siendo el método con mayor capacidad de transferencia hacia datos reales entre los sintetizadores evaluados hasta el momento.

Por el contrario, CTGAN V1 alcanzó un ROC-AUC TSTS de **0.758667**, muy próximo al baseline TRTR de **0.765600**. Este resultado indica una elevada utilidad predictiva interna del dataset sintético.

La diferencia entre ambos escenarios evidencia que una elevada capacidad predictiva dentro del entorno sintético no garantiza una capacidad equivalente de generalización hacia datos reales. En CTGAN, parte de la estructura predictiva aprendida parece ser específica de la distribución sintética generada.

Los resultados también muestran que una mejor conservación de determinadas correlaciones bivariadas o dependencias estructurales no implica necesariamente una mayor utilidad predictiva TSTR, dado que los modelos supervisados pueden explotar relaciones multivariantes y no lineales más complejas.

En conjunto, CTGAN V1 no supera a Gaussian Copula V2.1 según el criterio predictivo principal TSTR, aunque presenta una utilidad interna sustancialmente superior. Por ello, ambos sintetizadores representan comportamientos complementarios que deberán considerarse durante la comparación global.

In [ ]:
#Actualizamos nuestra tabla masestra
resultados_sintetizadores = pd.DataFrame({
    "Sintetizador": [
        "Gaussian Copula V2.1",
        "CTGAN V1"
    ],
    "TSTR_ROC_AUC": [
        AUC_TSTR_GC_V21,
        AUC_TSTR_CTGAN_V1
    ],
    "TSTS_ROC_AUC": [
        AUC_TSTS_GC_V21,
        AUC_TSTS_CTGAN_V1
    ]
})

resultados_sintetizadores["Gap_TRTR_TSTR"] = (
    AUC_TRTR_BASELINE
    - resultados_sintetizadores["TSTR_ROC_AUC"]
)

resultados_sintetizadores["Gap_TRTR_TSTS"] = (
    AUC_TRTR_BASELINE
    - resultados_sintetizadores["TSTS_ROC_AUC"]
)

resultados_sintetizadores["Gap_TSTR_TSTS"] = (
    resultados_sintetizadores["TSTR_ROC_AUC"]
    - resultados_sintetizadores["TSTS_ROC_AUC"]
)

resultados_sintetizadores["TSTR_Relativo_pct"] = (
    resultados_sintetizadores["TSTR_ROC_AUC"]
    / AUC_TRTR_BASELINE
    * 100
)

resultados_sintetizadores["TSTS_Relativo_pct"] = (
    resultados_sintetizadores["TSTS_ROC_AUC"]
    / AUC_TRTR_BASELINE
    * 100
)

resultados_sintetizadores.round(4)

,Sintetizador,TSTR_ROC_AUC,TSTS_ROC_AUC,Gap_TRTR_TSTR,Gap_TRTR_TSTS,Gap_TSTR_TSTS,TSTR_Relativo_pct,TSTS_Relativo_pct
0,Gaussian Copula V2.1,0.7267,0.6218,0.0389,0.1438,0.1049,94.9156,81.2185
1,CTGAN V1,0.6995,0.7587,0.0661,0.0069,-0.0591,91.3690,99.0945



Gaussian Copula
     ↓
Diagnóstico
     ↓
V1 → V2 → V2.1
     ↓
Benchmark sintético
           │
           ├──────────────┐
           ↓              ↓
     Gaussian V2.1     CTGAN V1
     TSTR 0.7267       TSTR 0.6995
     TSTS 0.6218       TSTS 0.7587
           │              │
           └──────┬───────┘
                  ↓
          Tercer sintetizador
                  ↓
          Comparación global
                  ↓
        Selección metodológica
        \

### 6.5 Tercer sintetizador: TVAE

Tenemos así tres familias diferentes:

Gaussian Copula → enfoque estadístico basado en distribuciones/dependencias.

CTGAN → red generativa adversarial.

TVAE → autoencoder variacional para datos tabulares.

Eso hace que la comparación tenga bastante más sentido que probar simplemente otra variante de GAN.

#### 6.5.1 Justificación de la selección de TVAE

Tras evaluar Gaussian Copula V2.1 y CTGAN V1, se incorpora un tercer método de generación sintética basado en una arquitectura diferente: **TVAE (Tabular Variational Autoencoder)**.

La selección de TVAE permite ampliar la comparación entre familias metodológicas distintas. Mientras Gaussian Copula utiliza un enfoque estadístico basado en distribuciones y dependencias, y CTGAN emplea entrenamiento generativo adversarial, TVAE utiliza un autoencoder variacional adaptado a datos tabulares.

La incorporación de este tercer enfoque resulta especialmente relevante debido al comportamiento observado en los experimentos anteriores.

Gaussian Copula V2.1 presentó la mayor capacidad de transferencia hacia datos reales hasta el momento, alcanzando un ROC-AUC TSTR de 0.726674, aunque su utilidad predictiva interna fue considerablemente menor (TSTS = 0.621809).

Por el contrario, CTGAN V1 alcanzó una elevada utilidad predictiva interna (TSTS = 0.758667), pero presentó una menor capacidad de transferencia hacia datos reales (TSTR = 0.699521).

Estos resultados muestran que la fidelidad estadística, la utilidad predictiva interna y la transferibilidad hacia datos reales no necesariamente evolucionan en la misma dirección.

TVAE se incorpora, por tanto, con el objetivo de analizar si una tercera arquitectura generativa presenta un equilibrio diferente entre estas dimensiones.

La hipótesis experimental no presupone que TVAE sea superior a los métodos anteriores. Su objetivo es determinar empíricamente si un enfoque basado en autoencoders variacionales puede conservar simultáneamente una estructura predictiva interna útil y una adecuada capacidad de transferencia hacia datos reales.

Al igual que en los experimentos anteriores, se mantendrá constante el protocolo de evaluación, incluyendo el conjunto de variables base, la reconstrucción determinista de variables derivadas, el conjunto real de prueba y la configuración del modelo LightGBM.

#### 6.5.2 Preparación de datos para TVAE

Para CTGAN ya construimos:

df_base_ctgan

con las mismas 33 variables base utilizadas por Gaussian Copula V2.1.

In [ ]:
# =========================================================
# 6.5.2 Preparación del dataset base para TVAE
# =========================================================

df_base_tvae = df_real.drop(
    columns=derived_cols
).copy()

print("Dataset real:", df_real.shape)
print("Dataset base TVAE:", df_base_tvae.shape)

print(
    "Mismo conjunto base que CTGAN:",
    df_base_tvae.columns.tolist()
    == df_base_ctgan.columns.tolist()
)

Dataset real: (307507, 41)
Dataset base TVAE: (307507, 33)
Mismo conjunto base que CTGAN: True


In [ ]:
#Verificamos la TARGET
print(
    df_base_tvae["TARGET"]
    .value_counts()
)

print()

print(
    df_base_tvae["TARGET"]
    .value_counts(normalize=True)
)

TARGET
0    282682
1     24825
Name: count, dtype: int64

TARGET
0    0.91927
1    0.08073
Name: proportion, dtype: float64


#### 6.5.3 Metadata de TVAE

In [ ]:
#Mantendremos exactamente el mismo criterio semántico utilizado con CTGAN.
from sdv.metadata import SingleTableMetadata

metadata_tvae = SingleTableMetadata()

metadata_tvae.detect_from_dataframe(
    data=df_base_tvae
)

In [ ]:
metadata_tvae.update_column(
    column_name="TARGET",
    sdtype="categorical"
)

metadata_tvae.update_column(
    column_name="SK_ID_CURR",
    sdtype="id"
)

categorical_count_cols = [
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",
    "DEF_30_CNT_SOCIAL_CIRCLE",
    "DEF_60_CNT_SOCIAL_CIRCLE"
]

for col in categorical_count_cols:
    metadata_tvae.update_column(
        column_name=col,
        sdtype="categorical"
    )

In [ ]:
#Comprobamos
metadata_tvae_dict = metadata_tvae.to_dict()

for col in [
    "TARGET",
    "SK_ID_CURR",
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",
    "DEF_30_CNT_SOCIAL_CIRCLE",
    "DEF_60_CNT_SOCIAL_CIRCLE",
    "OBS_30_CNT_SOCIAL_CIRCLE",
    "OBS_60_CNT_SOCIAL_CIRCLE"
]:
    print(
        f"{col:30s} -> "
        f"{metadata_tvae_dict['columns'][col]['sdtype']}"
    )

TARGET                         -> categorical
SK_ID_CURR                     -> id
CNT_CHILDREN                   -> categorical
CNT_FAM_MEMBERS                -> categorical
DEF_30_CNT_SOCIAL_CIRCLE       -> categorical
DEF_60_CNT_SOCIAL_CIRCLE       -> categorical
OBS_30_CNT_SOCIAL_CIRCLE       -> numerical
OBS_60_CNT_SOCIAL_CIRCLE       -> numerical


Esto es importante porque estamos controlando la preparación:

                  MISMAS 33 VARIABLES
                         │
          ┌──────────────┼──────────────┐
          ↓              ↓              ↓
   Gaussian Copula     CTGAN           TVAE
          ↓              ↓              ↓
     evaluación      evaluación      evaluación

Así reducimos factores de confusión.

#### 6.5.4 Configuración experimental de TVAE

Aquí haría algo parecido a CTGAN.

TVAE también es una red neuronal, así que no deberíamos empezar a optimizar hiperparámetros para intentar superar Gaussian Copula.

Nuestra primera versión será:

TVAE V1 — 300 epochs.

Esto además mantiene un presupuesto de épocas comparable con CTGAN V1, aunque una época de CTGAN y una de TVAE no tienen el mismo coste computacional ni deben interpretarse como equivalentes.

TVAE V1 será entrenado inicialmente durante 300 épocas.

Se utiliza este presupuesto como configuración experimental inicial y no como resultado de un proceso de optimización de hiperparámetros. De esta manera se mantiene el objetivo de comparar los sintetizadores bajo configuraciones representativas sin ajustar cada generador específicamente para maximizar su rendimiento predictivo.

Aunque CTGAN V1 también fue entrenado durante 300 épocas, este número no implica equivalencia de coste computacional entre ambas arquitecturas. Por ello, el tiempo de entrenamiento será registrado como una dimensión adicional de la comparación.

Cualquier optimización posterior de TVAE deberá estar motivada por el diagnóstico de los resultados obtenidos en esta primera versión.

#### 6.5.5 Comprobación de entorno antes de entrenar

In [ ]:
import sdv
import torch
import time
import random
import numpy as np

print("SDV:", sdv.__version__)
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

SDV: 1.38.3
PyTorch: 2.11.0+cu128
CUDA disponible: True
GPU: Tesla T4


In [ ]:
#Fijamos semillas
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

print("Random state:", RANDOM_STATE)

Random state: 42


#### 6.5.6 Crear TVAE V1

In [ ]:
from sdv.single_table import TVAESynthesizer

tvae = TVAESynthesizer(
    metadata=metadata_tvae,
    epochs=300,
    verbose=True
)

tvae

Hay una razón: CTGAN nos costó 155.50 minutos. Antes de lanzar otro entrenamiento neuronal largo, quiero comprobar dos cosas de tu entorno actual:

versión exacta de SDV;
que TVAESynthesizer se instancia correctamente con esos parámetros.

Así evitamos descubrir después de media hora que tu versión concreta de SDV tiene alguna diferencia.

In [ ]:
print(tvae)

In [ ]:
print(
    "TVAE creado correctamente:",
    isinstance(tvae, TVAESynthesizer)
)

TVAE creado correctamente: True


#### 6.5.7 Entrenamiento de TVAE V1

TVAE V1 se entrena utilizando el mismo dataset base de 33 variables empleado en los experimentos anteriores.

El entrenamiento se realizará durante 300 épocas utilizando aceleración GPU.

Al igual que en CTGAN, el objetivo de esta primera versión no es realizar una optimización exhaustiva de hiperparámetros, sino obtener una configuración representativa que pueda ser evaluada bajo el mismo protocolo estadístico y predictivo.

El tiempo de entrenamiento será registrado como una dimensión adicional de comparación entre sintetizadores.

In [ ]:
import time

inicio = time.time()

tvae.fit(df_base_tvae)

fin = time.time()

tiempo_entrenamiento_tvae = fin - inicio

print(
    f"\nTiempo de entrenamiento TVAE: "
    f"{tiempo_entrenamiento_tvae / 60:.2f} minutos"
)

KeyboardInterrupt: 

##### 6.5.7.1 Guardar TVAE V1

In [ ]:
tvae.save(
    filepath="/content/drive/MyDrive/Maestria VIU/Tesis/Notebooks/TFM/06_Synthetic_Model_Comparison/tvae_v1.pkl"
)

print("TVAE V1 guardado correctamente.")

TVAE V1 fue entrenado durante 300 épocas utilizando aceleración GPU mediante una Tesla T4.

El entrenamiento requirió aproximadamente **XX.XX minutos**.

Este tiempo se registra como parte del coste computacional del sintetizador y será comparado posteriormente con CTGAN V1 y Gaussian Copula V2.1.

El modelo entrenado fue almacenado como `tvae_v1.pkl` para garantizar la reproducibilidad del experimento.

In [ ]:
#LEctura TVAE
#LEctura del TVAE entrenado
import pickle
ruta_archivo = '/content/drive/MyDrive/Maestria VIU/Tesis/Notebooks/TFM/06_Synthetic_Model_Comparison/tvae_v1.pkl'

# Leer el archivo pickle en modo binario ('rb')
with open(ruta_archivo, 'rb') as archivo:
    tvae = pickle.load(archivo)

# Confirmar que se cargó correctamente
print("¡Archivo cargado con éxito!")
print("Tipo de objeto:", type(tvae))

¡Archivo cargado con éxito!
Tipo de objeto: <class 'sdv.single_table.ctgan.TVAESynthesizer'>


| Sintetizador | Épocas | GPU      |        Tiempo |
| ------------ | -----: | -------- | ------------: |
| CTGAN V1     |    300 | Tesla T4 |    155.50 min |
| TVAE V1      |    300 | Tesla T4 | **71.35 min** |


#### 6.5.8 Resultado del entrenamiento

TVAE V1 fue entrenado durante 300 épocas utilizando aceleración GPU mediante una Tesla T4.

El entrenamiento requirió aproximadamente **71.35 minutos**, un tiempo considerablemente inferior al registrado para CTGAN V1 bajo el mismo número de épocas.

El valor de pérdida observado al finalizar el entrenamiento fue aproximadamente **-17.67**. Este valor se registra únicamente como información del proceso de optimización y no será utilizado directamente como indicador de calidad del dataset sintético.

La calidad de TVAE V1 será evaluada posteriormente mediante fidelidad estadística, conservación de dependencias y utilidad predictiva bajo los escenarios TSTR y TSTS.

El modelo entrenado fue almacenado como `tvae_v1.pkl` para garantizar la reproducibilidad del experimento.

#### 6.5.9 Generación del dataset TVAE V1

Ahora sí repetimos exactamente la lógica de CTGAN: generación libre, sin imponer todavía la distribución de TARGET.

In [ ]:
import time

n_synthetic = len(df_real)

inicio = time.time()

df_tvae_v1_base = tvae.sample(
    num_rows=n_synthetic
)

fin = time.time()

tiempo_generacion_tvae = fin - inicio

print("Shape generado:", df_tvae_v1_base.shape)

print(
    f"Tiempo de generación TVAE: "
    f"{tiempo_generacion_tvae / 60:.2f} minutos"
)

Shape generado: (307507, 33)
Tiempo de generación TVAE: 0.18 minutos


##### 6.5.9.1 Validación estructural inicial

In [ ]:
print("Shape:")
print(df_tvae_v1_base.shape)

print("\nNulos:")
print(df_tvae_v1_base.isnull().sum().sum())

print("\nDuplicados:")
print(df_tvae_v1_base.duplicated().sum())

print("\nTARGET únicos:")
print(sorted(df_tvae_v1_base["TARGET"].unique()))

Shape:
(307507, 33)

Nulos:
0

Duplicados:
0

TARGET únicos:
[np.int64(0), np.int64(1)]


##### 6.5.9.2 Distribución de TARGET

In [ ]:
#Esto vuelve a ser importante porque TVAE también se está evaluando sin generación condicionada.
target_real = (
    df_real["TARGET"]
    .value_counts(normalize=True)
    .sort_index()
)

target_tvae = (
    df_tvae_v1_base["TARGET"]
    .value_counts(normalize=True)
    .sort_index()
)

comparacion_target_tvae = pd.DataFrame({
    "Real": target_real,
    "TVAE_V1": target_tvae
})

comparacion_target_tvae["Diferencia_abs"] = (
    comparacion_target_tvae["Real"]
    - comparacion_target_tvae["TVAE_V1"]
).abs()

comparacion_target_tvae

,Real,TVAE_V1,Diferencia_abs
TARGET,,,
0,0.91927,0.992631,0.073361
1,0.08073,0.007369,0.073361


In [ ]:
print("TARGET TVAE V1 - frecuencias:")
print(
    df_tvae_v1_base["TARGET"]
    .value_counts()
)

print("\nTARGET TVAE V1 - proporciones:")
print(
    df_tvae_v1_base["TARGET"]
    .value_counts(normalize=True)
)

TARGET TVAE V1 - frecuencias:
TARGET
0    305241
1      2266
Name: count, dtype: int64

TARGET TVAE V1 - proporciones:
TARGET
0    0.992631
1    0.007369
Name: proportion, dtype: float64


##### 6.5.9.3 Reconstrucción de variables derivadas

In [ ]:
df_tvae_v1 = df_tvae_v1_base.copy()

df_tvae_v1["AGE"] = (
    -df_tvae_v1["DAYS_BIRTH"] / 365.25
)

df_tvae_v1["EMPLOYMENT_YEARS"] = (
    -df_tvae_v1["DAYS_EMPLOYED"] / 365.25
)

df_tvae_v1["CREDIT_INCOME_RATIO"] = (
    df_tvae_v1["AMT_CREDIT"]
    / df_tvae_v1["AMT_INCOME_TOTAL"]
)

df_tvae_v1["ANNUITY_INCOME_RATIO"] = (
    df_tvae_v1["AMT_ANNUITY"]
    / df_tvae_v1["AMT_INCOME_TOTAL"]
)

df_tvae_v1["CREDIT_GOODS_RATIO"] = (
    df_tvae_v1["AMT_CREDIT"]
    / df_tvae_v1["AMT_GOODS_PRICE"]
)

df_tvae_v1["INCOME_PER_FAMILY"] = (
    df_tvae_v1["AMT_INCOME_TOTAL"]
    / df_tvae_v1["CNT_FAM_MEMBERS"]
)

df_tvae_v1["CHILDREN_RATIO"] = (
    df_tvae_v1["CNT_CHILDREN"]
    / df_tvae_v1["CNT_FAM_MEMBERS"]
)

df_tvae_v1["CREDIT_TERM_EST"] = (
    df_tvae_v1["AMT_CREDIT"]
    / df_tvae_v1["AMT_ANNUITY"]
)

##### 6.5.9.4 NaN e infinitos

In [ ]:
for col in derived_cols:

    n_nan = df_tvae_v1[col].isna().sum()
    n_inf = np.isinf(df_tvae_v1[col]).sum()

    print(
        f"{col:25s} | "
        f"NaN: {n_nan:6d} | "
        f"Inf: {n_inf:6d}"
    )

AGE                       | NaN:      0 | Inf:      0
EMPLOYMENT_YEARS          | NaN:      0 | Inf:      0
CREDIT_INCOME_RATIO       | NaN:      0 | Inf:      0
ANNUITY_INCOME_RATIO      | NaN:      0 | Inf:      0
CREDIT_GOODS_RATIO        | NaN:      0 | Inf:      0
INCOME_PER_FAMILY         | NaN:      0 | Inf:      0
CHILDREN_RATIO            | NaN:      0 | Inf:      0
CREDIT_TERM_EST           | NaN:      0 | Inf:      0


##### 6.5.9.5 Restaurar estructura final

In [ ]:
df_tvae_v1 = df_tvae_v1[
    df_real.columns
]

print("Real:", df_real.shape)
print("TVAE V1:", df_tvae_v1.shape)

print(
    "¿Mismas columnas y mismo orden?:",
    df_tvae_v1.columns.tolist()
    == df_real.columns.tolist()
)

Real: (307507, 41)
TVAE V1: (307507, 41)
¿Mismas columnas y mismo orden?: True


In [ ]:
print(
    "Duplicados TVAE V1:",
    df_tvae_v1.duplicated().sum()
)

print(
    "Nulos TVAE V1:",
    df_tvae_v1.isna().sum().sum()
)

Duplicados TVAE V1: 0
Nulos TVAE V1: 0


**Conclusiones**

TVAE V1 generó correctamente 307 507 observaciones y mantuvo la estructura de 41 variables del dataset final, sin valores nulos, infinitos ni registros completamente duplicados.

Las variables derivadas pudieron reconstruirse de forma determinista sin producir inconsistencias numéricas.

Sin embargo, se identificó una desviación importante en la distribución de la variable objetivo TARGET.

Mientras que la proporción de incumplimiento en los datos reales es del 8.073 %, TVAE V1 generó únicamente un 0.737 % de observaciones pertenecientes a la clase positiva.

En términos absolutos, el dataset real contiene 24 825 observaciones con TARGET=1, mientras que TVAE V1 generó únicamente 2 266.

Este resultado evidencia una limitada capacidad de TVAE V1 para reproducir de forma espontánea el desbalance original de la variable objetivo, mostrando una fuerte concentración hacia la clase mayoritaria.

No se aplicará todavía ninguna corrección o generación condicionada, dado que el objetivo de esta primera versión es evaluar el comportamiento natural del sintetizador bajo el mismo protocolo empleado para los demás métodos.

La influencia de esta desviación sobre la utilidad predictiva será evaluada posteriormente mediante los escenarios TSTR y TSTS.

#### 6.5.10 Validación estadística de TVAE V1

Ahora seguimos igual que con CTGAN. Primero queremos saber si, además del problema con TARGET, TVAE conserva las distribuciones y dependencias entre variables.

##### 6.5.10.1 Variables numéricas críticas

In [ ]:
variables_numericas_criticas = [
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "CREDIT_GOODS_RATIO",
    "CREDIT_TERM_EST"
]

filas = []

for col in variables_numericas_criticas:

    filas.append({
        "Variable": col,
        "Mean_Real": df_real[col].mean(),
        "Mean_TVAE": df_tvae_v1[col].mean(),
        "Std_Real": df_real[col].std(),
        "Std_TVAE": df_tvae_v1[col].std(),
        "Median_Real": df_real[col].median(),
        "Median_TVAE": df_tvae_v1[col].median()
    })

comparacion_numerica_tvae = pd.DataFrame(filas)

comparacion_numerica_tvae.round(4)

,Variable,Mean_Real,Mean_TVAE,Std_Real,Std_TVAE,Median_Real,Median_TVAE
0,EXT_SOURCE_1,0.5043,0.5070,0.1394,0.0361,0.5060,0.5060
1,EXT_SOURCE_2,0.5145,0.4902,0.1909,0.1776,0.5660,0.5594
2,EXT_SOURCE_3,0.5157,0.5705,0.1747,0.0819,0.5353,0.5660
3,DAYS_BIRTH,-16037.0273,-15892.4368,4363.9824,4197.6175,-15750.0000,-15872.0000
4,DAYS_EMPLOYED,-2251.5822,-1499.7739,2136.1601,787.2794,-1648.0000,-1629.0000
5,AMT_CREDIT,599028.5967,604833.2757,402492.6019,394673.7314,513531.0000,506940.2000
6,AMT_ANNUITY,27108.5807,25418.2497,14493.5221,9994.4176,24903.0000,24796.4000
7,AMT_GOODS_PRICE,538317.8090,543164.9022,369289.7661,365505.0280,450000.0000,450283.9000
8,CREDIT_GOODS_RATIO,1.1225,1.1496,0.1255,0.3078,1.1188,1.1193
9,CREDIT_TERM_EST,21.6124,22.4694,7.8242,8.4552,20.0000,22.0745


##### 6.5.10.2 Distribuciones discretas

In [ ]:
variables_discretas = [
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",
    "DEF_30_CNT_SOCIAL_CIRCLE",
    "DEF_60_CNT_SOCIAL_CIRCLE"
]

for col in variables_discretas:

    real_dist = (
        df_real[col]
        .value_counts(normalize=True)
        .sort_index()
    )

    tvae_dist = (
        df_tvae_v1[col]
        .value_counts(normalize=True)
        .sort_index()
    )

    comparacion = pd.concat(
        [
            real_dist.rename("Real"),
            tvae_dist.rename("TVAE_V1")
        ],
        axis=1
    ).fillna(0)

    comparacion["Diferencia_abs"] = (
        comparacion["Real"]
        - comparacion["TVAE_V1"]
    ).abs()

    print(f"\n===== {col} =====")
    display(comparacion)


===== CNT_CHILDREN =====


,Real,TVAE_V1,Diferencia_abs
CNT_CHILDREN,,,
0,0.700371,0.704517,0.004146
1,0.198753,0.204314,0.005561
2,0.086983,0.068542,0.018442
3,0.012088,0.022445,0.010357
4,0.001395,0.000179,0.001216
5,0.000273,0.000000,0.000273
6,0.000068,0.000003,0.000065
7,0.000023,0.000000,0.000023
8,0.000007,0.000000,0.000007



===== CNT_FAM_MEMBERS =====


,Real,TVAE_V1,Diferencia_abs
CNT_FAM_MEMBERS,,,
1.0,0.220636,0.279753,0.059117
2.0,0.514970,0.456907,0.058064
3.0,0.171053,0.180588,0.009535
4.0,0.080310,0.063020,0.017291
5.0,0.011310,0.019522,0.008211
6.0,0.001327,0.000205,0.001122
7.0,0.000263,0.000000,0.000263
8.0,0.000065,0.000003,0.000062
9.0,0.000020,0.000000,0.000020



===== DEF_30_CNT_SOCIAL_CIRCLE =====


,Real,TVAE_V1,Diferencia_abs
DEF_30_CNT_SOCIAL_CIRCLE,,,
0.0,0.885648,0.903228,0.017580
1.0,0.092118,0.089556,0.002563
2.0,0.017310,0.007040,0.010270
3.0,0.003876,0.000159,0.003717
4.0,0.000819,0.000013,0.000806
5.0,0.000182,0.000003,0.000179
6.0,0.000036,0.000000,0.000036
7.0,0.000003,0.000000,0.000003
8.0,0.000003,0.000000,0.000003



===== DEF_60_CNT_SOCIAL_CIRCLE =====


,Real,TVAE_V1,Diferencia_abs
DEF_60_CNT_SOCIAL_CIRCLE,,,
0.0,0.916207,0.945286,0.029079
1.0,0.071023,0.049992,0.021030
2.0,0.010309,0.004351,0.005958
3.0,0.001945,0.000364,0.001580
4.0,0.000436,0.000007,0.000429
5.0,0.000065,0.000000,0.000065
6.0,0.000010,0.000000,0.000010
7.0,0.000003,0.000000,0.000003
24.0,0.000003,0.000000,0.000003


Aquí quiero comprobar especialmente si TVAE repite el problema de CTGAN de sobrerrepresentar categorías extremadamente raras.

##### 6.5.10.3 Relaciones estructurales

In [ ]:
relaciones_estructurales = [
    ("CNT_CHILDREN", "CNT_FAM_MEMBERS"),
    (
        "DEF_30_CNT_SOCIAL_CIRCLE",
        "DEF_60_CNT_SOCIAL_CIRCLE"
    )
]

resultados_relaciones_tvae = []

for var1, var2 in relaciones_estructurales:

    corr_real = df_real[[var1, var2]].corr(
        method="spearman"
    ).iloc[0, 1]

    corr_tvae = df_tvae_v1[[var1, var2]].corr(
        method="spearman"
    ).iloc[0, 1]

    resultados_relaciones_tvae.append({
        "Relacion": f"{var1} ↔ {var2}",
        "Real": corr_real,
        "TVAE_V1": corr_tvae,
        "Error_abs": abs(corr_real - corr_tvae)
    })

relaciones_tvae_df = pd.DataFrame(
    resultados_relaciones_tvae
)

relaciones_tvae_df.round(6)

,Relacion,Real,TVAE_V1,Error_abs
0,CNT_CHILDREN ↔ CNT_FAM_MEMBERS,0.811554,0.795197,0.016357
1,DEF_30_CNT_SOCIAL_CIRCLE ↔ DEF_60_CNT_SOCIAL_C...,0.845149,0.737836,0.107313


Importante: aquí utilizamos nuevamente las correlaciones calculadas directamente con el df_real actual del notebook 06. Así evitamos mezclar los valores antiguos del notebook 05.

##### 6.5.10.4 Relaciones críticas con TARGET

In [ ]:
variables_target_criticas = [
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "EXT_SOURCE_1",
    "DAYS_BIRTH",
    "CREDIT_GOODS_RATIO",
    "DAYS_EMPLOYED",
    "CREDIT_TERM_EST"
]

resultados_target_tvae = []

for col in variables_target_criticas:

    corr_real = df_real[
        ["TARGET", col]
    ].corr(method="spearman").iloc[0, 1]

    corr_tvae = df_tvae_v1[
        ["TARGET", col]
    ].corr(method="spearman").iloc[0, 1]

    resultados_target_tvae.append({
        "Variable": col,
        "Real": corr_real,
        "TVAE_V1": corr_tvae,
        "Error_abs": abs(corr_real - corr_tvae)
    })

correlaciones_target_tvae = pd.DataFrame(
    resultados_target_tvae
)

correlaciones_target_tvae.round(6)

,Variable,Real,TVAE_V1,Error_abs
0,EXT_SOURCE_2,-0.147133,-0.072033,0.075100
1,EXT_SOURCE_3,-0.141501,-0.110851,0.030650
2,EXT_SOURCE_1,-0.085389,-0.010781,0.074609
3,DAYS_BIRTH,0.078331,0.026111,0.052220
4,CREDIT_GOODS_RATIO,0.066684,0.009675,0.057009
5,DAYS_EMPLOYED,0.075036,0.081026,0.005990
6,CREDIT_TERM_EST,-0.030262,-0.011798,0.018464


In [ ]:
#Calculamos MAE
mae_corr_target_tvae = (
    correlaciones_target_tvae["Error_abs"].mean()
)

print(
    f"MAE correlaciones críticas TARGET - TVAE V1: "
    f"{mae_corr_target_tvae:.6f}"
)

MAE correlaciones críticas TARGET - TVAE V1: 0.044863


Algo importante sobre este MAE

No debemos interpretar un MAE bajo como prueba suficiente de calidad.

CTGAN ya nos enseñó por qué:

conservó bastante bien ciertas relaciones;
obtuvo un MAE atractivo;
pero su TSTR terminó siendo peor que Gaussian Copula.

TVAE podría mostrarnos otra combinación completamente distinta.

| Dimensión                | TVAE V1                                                      |
| ------------------------ | ------------------------------------------------------------ |
| Estructura final         | Correcta                                                     |
| TARGET                   | Muy mal preservado                                           |
| Distribuciones numéricas | Mixtas                                                       |
| Variables discretas      | Mejor que CTGAN en raros extremos, pero con pérdida de colas |
| Relaciones estructurales | Bastante buenas                                              |
| Correlaciones con TARGET | Debilitadas                                                  |
| MAE TARGET               | `0.044863`                                                   |


En comparación, el MAE de correlaciones críticas queda muy cerca de Gaussian Copula V2.1 (0.045950) y peor que CTGAN V1 (0.034591). Pero ya sabemos que ese MAE por sí solo no predice TSTR.

Interpretación de las variables numéricas

TVAE reproduce bastante bien las medias de algunas variables financieras:

AMT_CREDIT
AMT_GOODS_PRICE
DAYS_BIRTH

Pero hay un patrón de reducción de variabilidad bastante evidente.

Por ejemplo:

EXT_SOURCE_1
Std real = 0.1394
Std TVAE = 0.0361

Eso es una compresión muy fuerte.

También:

EXT_SOURCE_3
Std real = 0.1747
Std TVAE = 0.0819

y:

DAYS_EMPLOYED
Std real = 2136.16
Std TVAE = 787.28

Esto sugiere que TVAE está generando observaciones demasiado concentradas alrededor de regiones centrales del espacio de datos.

Ese comportamiento también aparece en varias variables discretas.

**Distribuciones discretas**

Aquí TVAE tiene un comportamiento casi opuesto al de CTGAN.

CTGAN tendía a sobrerrepresentar categorías muy raras.

TVAE hace más bien esto:

conserva razonablemente las categorías frecuentes;
reduce bastante las categorías infrecuentes;
en algunos casos directamente deja de generarlas.

Por ejemplo CNT_CHILDREN:

5 hijos
Real = 0.000273
TVAE = 0

y muchas categorías mayores desaparecen completamente.

Eso no necesariamente es malo desde el punto de vista práctico, porque varias de esas categorías son extremadamente raras, pero sí indica pérdida de diversidad de cola.

En CNT_FAM_MEMBERS ocurre algo similar.

En DEF_30 y DEF_60, TVAE concentra todavía más masa en 0.

Así que el patrón general sería:

CTGAN expandía demasiado las colas raras; TVAE las comprime.

Ese contraste es muy interesante para tu comparación.

**Relaciones estructurales**

Aquí TVAE sale bastante bien:

CNT_CHILDREN ↔ CNT_FAM_MEMBERS

Real = 0.811554
TVAE = 0.795197
Error = 0.016357

Esto es excelente y prácticamente igual de bueno que CTGAN:

CTGAN error ≈ 0.018991

Para:

DEF_30 ↔ DEF_60

Real = 0.845149
TVAE = 0.737836
Error = 0.107313

aquí TVAE conserva una relación fuerte, aunque peor que CTGAN, cuyo error era aproximadamente 0.084309.

Conclusión:

TVAE conserva bastante bien dependencias estructurales importantes, especialmente las relacionadas con composición familiar.

**Relaciones con TARGET**

Aquí se ve una pérdida más clara de señal.

TVAE debilita bastante:

EXT_SOURCE_2
Real  = -0.147133
TVAE  = -0.072033
EXT_SOURCE_1
Real  = -0.085389
TVAE  = -0.010781
DAYS_BIRTH
Real  = 0.078331
TVAE  = 0.026111

y:

CREDIT_GOODS_RATIO
Real  = 0.066684
TVAE  = 0.009675

En cambio:

DAYS_EMPLOYED
Real = 0.075036
TVAE = 0.081026

se conserva muy bien.

La lectura más razonable es:

TVAE mantiene algunas señales concretas, pero reduce de forma marcada la intensidad de varias relaciones predictivas importantes.

Esto encaja bastante con el fuerte colapso de TARGET=1: si la clase positiva aparece muy poco, el sintetizador tiene menos capacidad efectiva para representar adecuadamente su estructura condicional.

Pero mantengamos cuidado metodológico: no afirmaría que la baja prevalencia causa directamente estas correlaciones debilitadas. Solo que ambos fenómenos aparecen conjuntamente.

**Conclusion**

La validación estadística de TVAE V1 muestra un comportamiento mixto.

En las variables numéricas, TVAE reproduce razonablemente determinadas medidas de tendencia central, especialmente en variables financieras como AMT_CREDIT y AMT_GOODS_PRICE. Sin embargo, se observa una reducción notable de la dispersión en algunas variables, destacando EXT_SOURCE_1, EXT_SOURCE_3 y DAYS_EMPLOYED.

Este comportamiento sugiere una tendencia del modelo a concentrar las observaciones sintéticas alrededor de regiones centrales de la distribución, reduciendo parte de la variabilidad presente en los datos reales.

En las variables discretas, TVAE conserva adecuadamente varias categorías frecuentes, pero tiende a infrarepresentar o eliminar categorías poco frecuentes. Este comportamiento contrasta con CTGAN V1, que había mostrado una tendencia a sobrerrepresentar determinadas categorías raras.

Respecto a las dependencias estructurales, TVAE preserva de forma satisfactoria la relación entre CNT_CHILDREN y CNT_FAM_MEMBERS, con un error absoluto de 0.016357 respecto al dataset real. La relación entre DEF_30_CNT_SOCIAL_CIRCLE y DEF_60_CNT_SOCIAL_CIRCLE también se mantiene elevada, aunque con una desviación mayor de 0.107313.

En las relaciones críticas con TARGET, TVAE V1 presenta un MAE de 0.044863. Varias asociaciones predictivas relevantes muestran una reducción considerable de intensidad, especialmente EXT_SOURCE_2, EXT_SOURCE_1, DAYS_BIRTH y CREDIT_GOODS_RATIO. Por el contrario, DAYS_EMPLOYED conserva una asociación muy próxima a la observada en los datos reales.

En conjunto, TVAE V1 conserva determinadas estructuras internas del dataset, pero presenta pérdida de variabilidad y debilitamiento de varias relaciones predictivas con TARGET.

Este resultado debe interpretarse conjuntamente con la fuerte desviación observada en la distribución de TARGET, donde la clase positiva representa únicamente el 0.737 % de los registros sintéticos frente al 8.073 % del dataset real.

La utilidad predictiva de esta representación será evaluada posteriormente mediante los escenarios TSTR y TSTS.

#### 6.5.11 TSTR de TVAE V1

In [ ]:
# =========================================================
# 6.5.11 TSTR - TVAE V1
# Train Synthetic, Test Real
# =========================================================

X_tvae = df_tvae_v1.drop(
    columns=["TARGET", "SK_ID_CURR"]
).copy()

y_tvae = df_tvae_v1["TARGET"].copy()

categorical_cols_tvae = (
    X_tvae
    .select_dtypes(include=["object", "category"])
    .columns
    .tolist()
)

preprocessor_tvae = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_cols_tvae
        )
    ],
    remainder="passthrough"
)

X_tvae_enc = preprocessor_tvae.fit_transform(X_tvae)

X_test_real_tvae_enc = preprocessor_tvae.transform(
    X_test_real
)

print(
    "TVAE train encoded:",
    X_tvae_enc.shape
)

print(
    "Real test encoded:",
    X_test_real_tvae_enc.shape
)

TVAE train encoded: (307507, 118)
Real test encoded: (61502, 118)


In [ ]:
modelo_tstr_tvae = crear_modelo_diagnostico()

modelo_tstr_tvae.fit(
    X_tvae_enc,
    y_tvae
)

proba_tstr_tvae = modelo_tstr_tvae.predict_proba(
    X_test_real_tvae_enc
)[:, 1]

auc_tstr_tvae = roc_auc_score(
    y_test_real,
    proba_tstr_tvae
)

print(
    f"TSTR ROC-AUC TVAE V1: "
    f"{auc_tstr_tvae:.6f}"
)

TSTR ROC-AUC TVAE V1: 0.660415


In [ ]:
print(
    "Diferencia TVAE - Gaussian V2.1:",
    auc_tstr_tvae - AUC_TSTR_GC_V21
)

print(
    "Diferencia TVAE - CTGAN V1:",
    auc_tstr_tvae - AUC_TSTR_CTGAN_V1
)

print(
    "Gap vs TRTR:",
    AUC_TRTR_BASELINE - auc_tstr_tvae
)

print(
    "TSTR relativo al baseline:",
    auc_tstr_tvae / AUC_TRTR_BASELINE * 100
)

Diferencia TVAE - Gaussian V2.1: -0.06625906011650162
Diferencia TVAE - CTGAN V1: -0.03910587405789634
Gap vs TRTR: 0.10518506011650153
TSTR relativo al baseline: 86.26109455113617


| Método               | TSTR ROC-AUC |  Gap vs TRTR | TSTR relativo |
| -------------------- | -----------: | -----------: | ------------: |
| Real → Real (TRTR)   | **0.765600** |            — |          100% |
| Gaussian Copula V2.1 | **0.726674** |     0.038926 |        94.92% |
| CTGAN V1             |     0.699521 |     0.066079 |        91.37% |
| TVAE V1              | **0.660415** | **0.105185** |    **86.26%** |


##### 6.5.11.1 Evaluación TSTR de TVAE V1

TVAE V1 obtuvo un ROC-AUC de **0.660415** en el escenario TSTR.

Este resultado se encuentra por debajo de Gaussian Copula V2.1 (0.726674) y CTGAN V1 (0.699521), con diferencias de -0.066259 y -0.039106 respectivamente.

Respecto al baseline TRTR de 0.765600, TVAE V1 presenta un gap de 0.105185 y alcanza aproximadamente el **86.26 % del ROC-AUC del baseline real**.

Por tanto, TVAE V1 presenta la menor capacidad de transferencia hacia datos reales entre los tres sintetizadores evaluados hasta el momento.

El resultado es consistente con diferentes desviaciones identificadas durante la validación estadística. TVAE V1 redujo considerablemente la representación de la clase minoritaria de TARGET, comprimió la dispersión de determinadas variables y debilitó varias asociaciones predictivas relevantes.

Asimismo, el espacio resultante después de la codificación contiene 118 variables, frente a las 150 observadas previamente con CTGAN V1. Esta reducción es coherente con la pérdida de determinadas categorías poco frecuentes en el dataset generado por TVAE.

Estos resultados no permiten atribuir la reducción del rendimiento TSTR a una única característica del proceso generativo. Sin embargo, conjuntamente muestran que la preservación parcial de determinadas dependencias estructurales no ha sido suficiente para mantener la capacidad de transferencia predictiva observada con los otros sintetizadores.

Gaussian Copula V2.1 continúa siendo, por tanto, el sintetizador con mayor utilidad TSTR entre los métodos evaluados.

Todavía nos falta TSTS de TVAE

Y no debemos saltárnoslo.

CTGAN nos enseñó precisamente por qué:

CTGAN TSTR = 0.6995
CTGAN TSTS = 0.7587

TVAE podría producir otra sorpresa.

Además, aquí tenemos una pregunta particularmente interesante:

¿Qué rendimiento observaría alguien que recibe únicamente el dataset TVAE, sin saber que la prevalencia real de default debería ser 8.07%?

Eso es exactamente lo que TSTS nos permite diagnosticar.

#### 6.5.12 TSTS de TVAE V1

In [ ]:
# =========================================================
# 6.5.12 TSTS - TVAE V1
# Train Synthetic, Test Synthetic
# =========================================================

from sklearn.model_selection import train_test_split

X_train_tvae, X_test_tvae, y_train_tvae, y_test_tvae = (
    train_test_split(
        X_tvae,
        y_tvae,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y_tvae
    )
)

print("Train TVAE:", X_train_tvae.shape)
print("Test TVAE:", X_test_tvae.shape)

print("\nTARGET train:")
print(y_train_tvae.value_counts())
print(y_train_tvae.value_counts(normalize=True))

print("\nTARGET test:")
print(y_test_tvae.value_counts())
print(y_test_tvae.value_counts(normalize=True))

Train TVAE: (246005, 39)
Test TVAE: (61502, 39)

TARGET train:
TARGET
0    244192
1      1813
Name: count, dtype: int64
TARGET
0    0.99263
1    0.00737
Name: proportion, dtype: float64

TARGET test:
TARGET
0    61049
1      453
Name: count, dtype: int64
TARGET
0    0.992634
1    0.007366
Name: proportion, dtype: float64


El stratify es especialmente importante ahora porque solamente tenemos 2,266 positivos.

Deberíamos obtener aproximadamente 453 positivos en el test sintético, así que sigue siendo posible calcular ROC-AUC.

Después hacemos el preprocessing solo con el training sintético

In [ ]:
categorical_cols_tsts_tvae = (
    X_train_tvae
    .select_dtypes(include=["object", "category"])
    .columns
    .tolist()
)

preprocessor_tsts_tvae = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_cols_tsts_tvae
        )
    ],
    remainder="passthrough"
)

X_train_tvae_enc = (
    preprocessor_tsts_tvae.fit_transform(X_train_tvae)
)

X_test_tvae_enc = (
    preprocessor_tsts_tvae.transform(X_test_tvae)
)

print(
    "TVAE synthetic train encoded:",
    X_train_tvae_enc.shape
)

print(
    "TVAE synthetic test encoded:",
    X_test_tvae_enc.shape
)

TVAE synthetic train encoded: (246005, 118)
TVAE synthetic test encoded: (61502, 118)


In [ ]:
#Entrenamos con LightGBM
modelo_tsts_tvae = crear_modelo_diagnostico()

modelo_tsts_tvae.fit(
    X_train_tvae_enc,
    y_train_tvae
)

proba_tsts_tvae = modelo_tsts_tvae.predict_proba(
    X_test_tvae_enc
)[:, 1]

auc_tsts_tvae = roc_auc_score(
    y_test_tvae,
    proba_tsts_tvae
)

print(
    f"TSTS ROC-AUC TVAE V1: "
    f"{auc_tsts_tvae:.6f}"
)

TSTS ROC-AUC TVAE V1: 0.990961


In [ ]:
print(
    f"TSTR TVAE V1: {auc_tstr_tvae:.6f}"
)

print(
    f"TSTS TVAE V1: {auc_tsts_tvae:.6f}"
)

print(
    f"Gap TSTR-TSTS: "
    f"{auc_tstr_tvae - auc_tsts_tvae:.6f}"
)

print(
    f"TSTS relativo al TRTR: "
    f"{auc_tsts_tvae / AUC_TRTR_BASELINE * 100:.2f}%"
)

TSTR TVAE V1: 0.660415
TSTS TVAE V1: 0.990961
Gap TSTR-TSTS: -0.330546
TSTS relativo al TRTR: 129.44%


| Escenario       |       ROC-AUC |
| --------------- | ------------: |
| TRTR Real       |  **0.765600** |
| TSTR TVAE V1    |  **0.660415** |
| TSTS TVAE V1    |  **0.990961** |
| Gap TSTR − TSTS | **−0.330546** |
| TSTR / TRTR     |    **86.26%** |
| TSTS / TRTR     |   **129.44%** |


| Sintetizador             |         TSTR |         TSTS | Gap TSTR−TSTS |
| ------------------------ | -----------: | -----------: | ------------: |
| **Gaussian Copula V2.1** | **0.726674** |     0.621809 |     +0.104865 |
| **CTGAN V1**             |     0.699521 |     0.758667 |     −0.059147 |
| **TVAE V1**              |     0.660415 | **0.990961** | **−0.330546** |


Tenemos tres comportamientos generativos diferentes.

Gaussian Copula V2.1

Su dataset sintético es internamente menos predictivo:

TSTS = 0.6218

pero el entrenamiento con esos datos transfiere mejor al real:

TSTR = 0.7267

CTGAN V1

Produce una estructura interna bastante predictiva:

TSTS = 0.7587

pero su transferencia baja:

TSTR = 0.6995

TVAE V1

Produce una estructura sintética casi perfectamente discriminativa:

TSTS = 0.9910

pero es el que peor transfiere:

TSTR = 0.6604

Por tanto, con estos experimentos podemos defender empíricamente algo bastante relevante:

La capacidad predictiva observada dentro de un dataset sintético no constituye, por sí sola, evidencia suficiente de que dicho dataset preserve la estructura predictiva necesaria para sustituir datos reales durante el entrenamiento.

Para nuestro problema, TSTR es mucho más importante para decidir sustitución de datos reales.

##### 6.5.12.1 Evaluación TSTS de TVAE V1

TVAE V1 obtuvo un ROC-AUC de **0.990961** en el escenario TSTS, considerablemente superior al TSTR de **0.660415**.

La diferencia TSTR-TSTS es de **-0.330546**, constituyendo la mayor divergencia entre utilidad predictiva interna y capacidad de transferencia observada entre los sintetizadores evaluados.

El elevado ROC-AUC TSTS indica que TARGET resulta altamente discriminable dentro de la distribución sintética generada por TVAE. Sin embargo, este comportamiento no se mantiene cuando el modelo entrenado con dichos datos se evalúa sobre observaciones reales.

Por tanto, el elevado rendimiento TSTS no debe interpretarse como evidencia de superioridad frente al dataset real ni como una medida directa de fidelidad. Aunque el cociente entre TSTS y TRTR alcanza el 129.44 %, ambos valores corresponden a escenarios de evaluación con distribuciones diferentes.

La combinación de un TSTS de 0.990961 y un TSTR de 0.660415 evidencia una fuerte divergencia entre la utilidad predictiva interna del dataset sintético y su capacidad de transferencia hacia datos reales.

Este comportamiento debe considerarse conjuntamente con las desviaciones identificadas durante la validación estadística. TVAE V1 generó únicamente un 0.737 % de observaciones pertenecientes a la clase positiva, frente al 8.073 % observado en los datos reales, además de presentar reducción de variabilidad y debilitamiento de diversas relaciones críticas con TARGET.

Los resultados son compatibles con la existencia de una estructura predictiva sintética altamente separable que no se encuentra adecuadamente alineada con la estructura predictiva del dataset real. No obstante, el experimento no permite atribuir esta divergencia a un único mecanismo generativo.

En consecuencia, TVAE V1 presenta una elevada utilidad predictiva interna, pero la menor capacidad de transferencia TSTR de los tres sintetizadores evaluados.

#### 6.5.13 Conclusiones de TVAE V1

La evaluación de TVAE V1 muestra que el sintetizador es capaz de generar un dataset estructuralmente válido, sin valores nulos, infinitos ni registros completamente duplicados, manteniendo además determinadas dependencias relevantes entre variables.

TVAE presentó un coste computacional inferior al de CTGAN V1 en el entorno experimental utilizado, requiriendo aproximadamente 71.35 minutos de entrenamiento frente a 155.50 minutos para CTGAN, ambos configurados con 300 épocas.

Sin embargo, se identificaron limitaciones importantes. TVAE redujo considerablemente la representación de la clase minoritaria de TARGET, desde un 8.073 % en los datos reales hasta un 0.737 % en los datos sintéticos. Asimismo, mostró compresión de la variabilidad en determinadas variables y pérdida de categorías poco frecuentes.

En términos de fidelidad predictiva bivariada, el MAE de las correlaciones críticas con TARGET fue de 0.044863. Aunque determinadas dependencias estructurales fueron preservadas satisfactoriamente, estos resultados no se tradujeron en una elevada capacidad de transferencia.

TVAE V1 obtuvo un ROC-AUC TSTR de **0.660415**, inferior a CTGAN V1 (0.699521) y Gaussian Copula V2.1 (0.726674).

En contraste, alcanzó un ROC-AUC TSTS de **0.990961**, mostrando que la estructura sintética generada permite una discriminación prácticamente perfecta de TARGET dentro de su propia distribución.

La fuerte divergencia entre TSTS y TSTR demuestra que una elevada utilidad predictiva interna no garantiza que las relaciones aprendidas mediante datos sintéticos sean transferibles hacia datos reales.

TVAE V1 queda, por tanto, como el sintetizador con menor capacidad TSTR entre los métodos evaluados, aunque su comportamiento constituye un caso especialmente relevante para analizar las diferencias entre fidelidad estadística, utilidad predictiva interna y transferibilidad.

#### 6.5.14 Control experimental de la distribución de TARGET

La comparación inicial entre Gaussian Copula V2.1, CTGAN V1 y TVAE V1 presenta una diferencia metodológica relevante.

Gaussian Copula V2.1 utiliza generación condicionada por TARGET, reproduciendo de forma explícita la prevalencia observada en el dataset real. Por el contrario, CTGAN V1 y TVAE V1 fueron evaluados inicialmente mediante generación no condicionada, permitiendo que cada sintetizador reprodujera espontáneamente la distribución de la variable objetivo.

Esta diferencia resulta especialmente relevante debido a que CTGAN V1 reprodujo aproximadamente un 7.51 % de observaciones positivas, mientras que TVAE V1 generó únicamente un 0.74 %, frente al 8.07 % presente en los datos reales.

Por ello, se incorpora un experimento de control en el que CTGAN y TVAE serán utilizados para generar nuevas muestras condicionadas por TARGET, manteniendo exactamente el número de observaciones positivas y negativas del dataset real.

Los modelos generativos no serán reentrenados. Únicamente se modifica la estrategia de muestreo, permitiendo aislar parcialmente el efecto asociado a la prevalencia marginal de TARGET.

Las nuevas muestras se denominarán CTGAN V1-C y TVAE V1-C, donde C identifica el uso de muestreo condicionado.

El objetivo del experimento es determinar si las diferencias de utilidad predictiva observadas entre los sintetizadores se mantienen cuando la distribución marginal de TARGET se encuentra controlada.

##### 6.5.14.1 Definir cantidades de cada clase

In [ ]:
target_counts_real = (
    df_real["TARGET"]
    .value_counts()
    .sort_index()
)

n_target_0 = int(target_counts_real.loc[0])
n_target_1 = int(target_counts_real.loc[1])

print("TARGET=0:", n_target_0)
print("TARGET=1:", n_target_1)
print("Total:", n_target_0 + n_target_1)

TARGET=0: 282682
TARGET=1: 24825
Total: 307507


##### 6.5.14.2 Comprobar Conditional Sampling

In [ ]:
from sdv.sampling import Condition

In [ ]:
condition_target_0 = Condition(
    num_rows=n_target_0,
    column_values={
        "TARGET": 0
    }
)

condition_target_1 = Condition(
    num_rows=n_target_1,
    column_values={
        "TARGET": 1
    }
)

print(condition_target_0)
print(condition_target_1)

##### 6.5.14.3 Generación condicionada de CTGAN V1-C

CTGAN ya aprendió espontáneamente una prevalencia muy cercana a la real. Por tanto, es nuestro caso de control más limpio.

Además, antes de pedir 307 mil filas, hagamos una pequeña prueba. El muestreo condicional en modelos neuronales puede ser más costoso que sample().

In [ ]:
test_condition_ctgan = Condition(
    num_rows=1000,
    column_values={
        "TARGET": 1
    }
)

inicio = time.time()

df_test_ctgan_cond = ctgan.sample_from_conditions(
    conditions=[test_condition_ctgan]
)

fin = time.time()

print("Shape:", df_test_ctgan_cond.shape)

print("\nTARGET:")
print(df_test_ctgan_cond["TARGET"].value_counts())

print(
    f"\nTiempo prueba: {fin - inicio:.2f} segundos"
)

Sampling conditions: 100%|██████████| 1000/1000 [00:01<00:00, 833.25it/s]

Shape: (1000, 33)

TARGET:
TARGET
1    1000
Name: count, dtype: int64

Tiempo prueba: 1.20 segundos


In [ ]:
inicio = time.time()

df_ctgan_v1c_base = ctgan.sample_from_conditions(
    conditions=[
        condition_target_0,
        condition_target_1
    ]
)

fin = time.time()

tiempo_generacion_ctgan_v1c = fin - inicio

print("Shape:", df_ctgan_v1c_base.shape)

print("\nTARGET:")
print(
    df_ctgan_v1c_base["TARGET"]
    .value_counts()
    .sort_index()
)

print("\nTARGET proporciones:")
print(
    df_ctgan_v1c_base["TARGET"]
    .value_counts(normalize=True)
    .sort_index()
)

print(
    f"\nTiempo generación condicionada CTGAN: "
    f"{tiempo_generacion_ctgan_v1c / 60:.2f} minutos"
)

Sampling conditions: 100%|██████████| 307507/307507 [00:34<00:00, 9018.39it/s]

Shape: (307507, 33)

TARGET:
TARGET
0    282682
1     24825
Name: count, dtype: int64

TARGET proporciones:
TARGET
0    0.91927
1    0.08073
Name: proportion, dtype: float64

Tiempo generación condicionada CTGAN: 0.57 minutos


##### 6.5.14.4 Mezclar las observaciones

In [ ]:
#Aunque SDV haya devuelto las condiciones agrupadas, no queremos un dataset ordenado por TARGET:
df_ctgan_v1c_base = (
    df_ctgan_v1c_base
    .sample(
        frac=1,
        random_state=RANDOM_STATE
    )
    .reset_index(drop=True)
)

print(
    df_ctgan_v1c_base["TARGET"]
    .value_counts()
    .sort_index()
)

TARGET
0    282682
1     24825
Name: count, dtype: int64


##### 6.5.14.5 Reconstruir las variables derivadas

In [ ]:
#Creemos una función única:
def reconstruir_variables_derivadas(df_base):
    df = df_base.copy()

    df["AGE"] = -df["DAYS_BIRTH"] / 365.25

    df["EMPLOYMENT_YEARS"] = (
        -df["DAYS_EMPLOYED"] / 365.25
    )

    df["CREDIT_INCOME_RATIO"] = (
        df["AMT_CREDIT"]
        / df["AMT_INCOME_TOTAL"]
    )

    df["ANNUITY_INCOME_RATIO"] = (
        df["AMT_ANNUITY"]
        / df["AMT_INCOME_TOTAL"]
    )

    df["CREDIT_GOODS_RATIO"] = (
        df["AMT_CREDIT"]
        / df["AMT_GOODS_PRICE"]
    )

    df["INCOME_PER_FAMILY"] = (
        df["AMT_INCOME_TOTAL"]
        / df["CNT_FAM_MEMBERS"]
    )

    df["CHILDREN_RATIO"] = (
        df["CNT_CHILDREN"]
        / df["CNT_FAM_MEMBERS"]
    )

    df["CREDIT_TERM_EST"] = (
        df["AMT_CREDIT"]
        / df["AMT_ANNUITY"]
    )

    return df

In [ ]:
df_ctgan_v1c = reconstruir_variables_derivadas(
    df_ctgan_v1c_base
)

df_ctgan_v1c = df_ctgan_v1c[
    df_real.columns
]

print("CTGAN V1-C:", df_ctgan_v1c.shape)

print(
    "Mismas columnas:",
    df_ctgan_v1c.columns.tolist()
    == df_real.columns.tolist()
)

CTGAN V1-C: (307507, 41)
Mismas columnas: True


##### 6.5.14.6 Validación estructural

In [ ]:
print(
    "Nulos:",
    df_ctgan_v1c.isna().sum().sum()
)

print(
    "Duplicados:",
    df_ctgan_v1c.duplicated().sum()
)

for col in derived_cols:

    print(
        f"{col:25s} | "
        f"NaN: {df_ctgan_v1c[col].isna().sum():6d} | "
        f"Inf: {np.isinf(df_ctgan_v1c[col]).sum():6d}"
    )

Nulos: 0
Duplicados: 0
AGE                       | NaN:      0 | Inf:      0
EMPLOYMENT_YEARS          | NaN:      0 | Inf:      0
CREDIT_INCOME_RATIO       | NaN:      0 | Inf:      0
ANNUITY_INCOME_RATIO      | NaN:      0 | Inf:      0
CREDIT_GOODS_RATIO        | NaN:      0 | Inf:      0
INCOME_PER_FAMILY         | NaN:      0 | Inf:      0
CHILDREN_RATIO            | NaN:      0 | Inf:      0
CREDIT_TERM_EST           | NaN:      0 | Inf:      0


##### 6.5.14.7 TSTR de CTGAN V1-C

In [ ]:
# =========================================================
# 6.5.14.7 TSTR - CTGAN V1-C
# Train Synthetic Conditional, Test Real
# =========================================================

X_ctgan_v1c = df_ctgan_v1c.drop(
    columns=["TARGET", "SK_ID_CURR"]
).copy()

y_ctgan_v1c = df_ctgan_v1c["TARGET"].copy()

print("X CTGAN V1-C:", X_ctgan_v1c.shape)
print("y CTGAN V1-C:", y_ctgan_v1c.shape)

print("\nTARGET:")
print(
    y_ctgan_v1c
    .value_counts()
    .sort_index()
)

X CTGAN V1-C: (307507, 39)
y CTGAN V1-C: (307507,)

TARGET:
TARGET
0    282682
1     24825
Name: count, dtype: int64


In [ ]:
#Ahora construimos el encoder solo sobre CTGAN V1-C:
categorical_cols_ctgan_v1c = (
    X_ctgan_v1c
    .select_dtypes(include=["object", "category"])
    .columns
    .tolist()
)

preprocessor_ctgan_v1c = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_cols_ctgan_v1c
        )
    ],
    remainder="passthrough"
)

X_ctgan_v1c_enc = (
    preprocessor_ctgan_v1c.fit_transform(
        X_ctgan_v1c
    )
)

X_test_real_ctgan_v1c_enc = (
    preprocessor_ctgan_v1c.transform(
        X_test_real
    )
)

print(
    "CTGAN V1-C train encoded:",
    X_ctgan_v1c_enc.shape
)

print(
    "Real test encoded:",
    X_test_real_ctgan_v1c_enc.shape
)

print(
    "Features compatibles:",
    X_ctgan_v1c_enc.shape[1]
    == X_test_real_ctgan_v1c_enc.shape[1]
)

CTGAN V1-C train encoded: (307507, 150)
Real test encoded: (61502, 150)
Features compatibles: True


In [ ]:
#Aplicamos LightGBM
modelo_tstr_ctgan_v1c = crear_modelo_diagnostico()

modelo_tstr_ctgan_v1c.fit(
    X_ctgan_v1c_enc,
    y_ctgan_v1c
)

proba_tstr_ctgan_v1c = (
    modelo_tstr_ctgan_v1c.predict_proba(
        X_test_real_ctgan_v1c_enc
    )[:, 1]
)

auc_tstr_ctgan_v1c = roc_auc_score(
    y_test_real,
    proba_tstr_ctgan_v1c
)

print(
    f"TSTR ROC-AUC CTGAN V1-C: "
    f"{auc_tstr_ctgan_v1c:.6f}"
)

TSTR ROC-AUC CTGAN V1-C: 0.700643


In [ ]:
#Comparación crítica: V1 vs V1-C
comparacion_ctgan_cond = pd.DataFrame({
    "Version": [
        "CTGAN V1",
        "CTGAN V1-C"
    ],
    "TARGET_1_pct": [
        df_ctgan_v1["TARGET"].mean() * 100,
        df_ctgan_v1c["TARGET"].mean() * 100
    ],
    "TSTR_ROC_AUC": [
        AUC_TSTR_CTGAN_V1,
        auc_tstr_ctgan_v1c
    ]
})

comparacion_ctgan_cond["Gap_vs_TRTR"] = (
    AUC_TRTR_BASELINE
    - comparacion_ctgan_cond["TSTR_ROC_AUC"]
)

comparacion_ctgan_cond["TSTR_Relativo_pct"] = (
    comparacion_ctgan_cond["TSTR_ROC_AUC"]
    / AUC_TRTR_BASELINE
    * 100
)

comparacion_ctgan_cond.round(6)

,Version,TARGET_1_pct,TSTR_ROC_AUC,Gap_vs_TRTR,TSTR_Relativo_pct
0,CTGAN V1,7.508447,0.699521,0.066079,91.368967
1,CTGAN V1-C,8.072987,0.700643,0.064957,91.515518


In [ ]:
delta_tstr_ctgan_cond = (
    auc_tstr_ctgan_v1c
    - AUC_TSTR_CTGAN_V1
)

print(
    f"CTGAN V1 TSTR:   "
    f"{AUC_TSTR_CTGAN_V1:.6f}"
)

print(
    f"CTGAN V1-C TSTR: "
    f"{auc_tstr_ctgan_v1c:.6f}"
)

print(
    f"Cambio absoluto TSTR: "
    f"{delta_tstr_ctgan_cond:+.6f}"
)

print(
    f"Diferencia V1-C vs Gaussian V2.1: "
    f"{auc_tstr_ctgan_v1c - AUC_TSTR_GC_V21:+.6f}"
)

CTGAN V1 TSTR:   0.699521
CTGAN V1-C TSTR: 0.700643
Cambio absoluto TSTR: +0.001122
Diferencia V1-C vs Gaussian V2.1: -0.026031


##### 6.5.14.8 TSTS de CTGAN V1-C

In [ ]:
#Necesitamos repetir TSTS porque ahora la distribución sintética cambió.
X_train_ctgan_v1c, X_test_ctgan_v1c, \
y_train_ctgan_v1c, y_test_ctgan_v1c = train_test_split(
    X_ctgan_v1c,
    y_ctgan_v1c,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_ctgan_v1c
)

print(
    "Train:",
    X_train_ctgan_v1c.shape
)

print(
    "Test:",
    X_test_ctgan_v1c.shape
)

print("\nTARGET train:")
print(
    y_train_ctgan_v1c
    .value_counts()
    .sort_index()
)

print("\nTARGET test:")
print(
    y_test_ctgan_v1c
    .value_counts()
    .sort_index()
)

Train: (246005, 39)
Test: (61502, 39)

TARGET train:
TARGET
0    226145
1     19860
Name: count, dtype: int64

TARGET test:
TARGET
0    56537
1     4965
Name: count, dtype: int64


In [ ]:
categorical_cols_tsts_ctgan_v1c = (
    X_train_ctgan_v1c
    .select_dtypes(include=["object", "category"])
    .columns
    .tolist()
)

preprocessor_tsts_ctgan_v1c = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_cols_tsts_ctgan_v1c
        )
    ],
    remainder="passthrough"
)

X_train_ctgan_v1c_enc = (
    preprocessor_tsts_ctgan_v1c.fit_transform(
        X_train_ctgan_v1c
    )
)

X_test_ctgan_v1c_enc = (
    preprocessor_tsts_ctgan_v1c.transform(
        X_test_ctgan_v1c
    )
)

print(
    "Synthetic train encoded:",
    X_train_ctgan_v1c_enc.shape
)

print(
    "Synthetic test encoded:",
    X_test_ctgan_v1c_enc.shape
)

Synthetic train encoded: (246005, 150)
Synthetic test encoded: (61502, 150)


In [ ]:
modelo_tsts_ctgan_v1c = crear_modelo_diagnostico()

modelo_tsts_ctgan_v1c.fit(
    X_train_ctgan_v1c_enc,
    y_train_ctgan_v1c
)

proba_tsts_ctgan_v1c = (
    modelo_tsts_ctgan_v1c.predict_proba(
        X_test_ctgan_v1c_enc
    )[:, 1]
)

auc_tsts_ctgan_v1c = roc_auc_score(
    y_test_ctgan_v1c,
    proba_tsts_ctgan_v1c
)

print(
    f"TSTS ROC-AUC CTGAN V1-C: "
    f"{auc_tsts_ctgan_v1c:.6f}"
)

TSTS ROC-AUC CTGAN V1-C: 0.775625


In [ ]:
print("===== CTGAN V1-C =====")

print(
    f"TSTR: {auc_tstr_ctgan_v1c:.6f}"
)

print(
    f"TSTS: {auc_tsts_ctgan_v1c:.6f}"
)

print(
    f"Gap TSTR-TSTS: "
    f"{auc_tstr_ctgan_v1c - auc_tsts_ctgan_v1c:.6f}"
)

print(
    f"TSTR relativo TRTR: "
    f"{auc_tstr_ctgan_v1c / AUC_TRTR_BASELINE * 100:.2f}%"
)

print(
    f"TSTS relativo TRTR: "
    f"{auc_tsts_ctgan_v1c / AUC_TRTR_BASELINE * 100:.2f}%"
)

===== CTGAN V1-C =====
TSTR: 0.700643
TSTS: 0.775625
Gap TSTR-TSTS: -0.074982
TSTR relativo TRTR: 91.52%
TSTS relativo TRTR: 101.31%


| Versión              |   TARGET=1 |         TSTR |         TSTS |
| -------------------- | ---------: | -----------: | -----------: |
| CTGAN V1             |     7.508% |     0.699521 |     0.758667 |
| **CTGAN V1-C**       | **8.073%** | **0.700643** | **0.775625** |
| Gaussian Copula V2.1 |     8.073% | **0.726674** |     0.621809 |


Es una mejora de apenas 0.0011 puntos ROC-AUC.

Eso es importante porque CTGAN pasó de una prevalencia espontánea de aproximadamente 7.51% a exactamente 8.073%, pero su capacidad de transferencia prácticamente permaneció igual.

Por tanto, para CTGAN tenemos evidencia experimental de que la pequeña diferencia inicial en la prevalencia marginal de TARGET no explica la diferencia de rendimiento frente a Gaussian Copula V2.1.

Gaussian sigue por encima:

0.726674 − 0.700643 = 0.026031

Y ahora ambos tienen exactamente la misma distribución marginal de TARGET.

| Métrica | CTGAN V1 | CTGAN V1-C |        Cambio |
| ------- | -------: | ---------: | ------------: |
| TSTR    | 0.699521 |   0.700643 | **+0.001122** |
| TSTS    | 0.758667 |   0.775625 | **+0.016958** |


El condicionamiento prácticamente no afecta la transferencia hacia datos reales, pero aumenta más claramente la capacidad predictiva dentro del propio entorno sintético.

Consecuentemente:

V1: gap TSTR−TSTS ≈ −0.05915
V1-C: gap TSTR−TSTS = −0.07498

Es decir, la divergencia entre utilidad interna y transferibilidad incluso aumenta ligeramente.

Esto vuelve a reforzar nuestra conclusión anterior:

Mejorar la capacidad discriminativa dentro del dataset sintético no implica necesariamente mejorar su capacidad de sustituir datos reales durante el entrenamiento.

**Conslusion**

El muestreo condicionado permitió generar CTGAN V1-C reproduciendo exactamente la distribución marginal de TARGET observada en los datos reales, con 282,682 observaciones de la clase 0 y 24,825 observaciones de la clase 1.

CTGAN V1-C obtuvo un ROC-AUC TSTR de **0.700643**, frente al **0.699521** obtenido por CTGAN V1 mediante generación no condicionada. Por tanto, controlar explícitamente la prevalencia de TARGET produjo una mejora absoluta de únicamente **0.001122** puntos de ROC-AUC.

Este resultado indica que, en el caso de CTGAN, la pequeña diferencia existente entre la prevalencia positiva generada espontáneamente (7.51 %) y la prevalencia real (8.07 %) no explica de forma relevante la diferencia de transferibilidad observada frente a Gaussian Copula V2.1.

Incluso después de controlar TARGET, Gaussian Copula V2.1 mantiene un TSTR superior (0.726674 frente a 0.700643), con una diferencia de 0.026031.

En el escenario TSTS, CTGAN V1-C alcanzó un ROC-AUC de **0.775625**, superior al 0.758667 obtenido por CTGAN V1. El condicionamiento produjo, por tanto, un incremento mayor de la utilidad predictiva interna que de la capacidad de transferencia hacia datos reales.

El gap TSTR-TSTS de CTGAN V1-C fue de **-0.074982**, evidenciando que una mayor capacidad discriminativa dentro de la distribución sintética no implica necesariamente una mejora equivalente de transferibilidad.

Estos resultados permiten controlar una diferencia metodológica presente en la comparación inicial y proporcionan evidencia de que la ventaja TSTR observada en Gaussian Copula V2.1 frente a CTGAN no puede atribuirse únicamente al control explícito de la distribución marginal de TARGET.

##### 6.5.14.9 Prueba de conditional sampling de TVAE

In [ ]:
test_condition_tvae = Condition(
    num_rows=1000,
    column_values={
        "TARGET": 1
    }
)

inicio = time.time()

df_test_tvae_cond = tvae.sample_from_conditions(
    conditions=[test_condition_tvae]
)

fin = time.time()

print("Shape:", df_test_tvae_cond.shape)

print("\nTARGET:")
print(
    df_test_tvae_cond["TARGET"]
    .value_counts()
)

print(
    f"\nTiempo prueba: "
    f"{fin - inicio:.2f} segundos"
)

Sampling conditions: 100%|██████████| 1000/1000 [00:07<00:00, 129.78it/s]

Shape: (1000, 33)

TARGET:
TARGET
1    1000
Name: count, dtype: int64

Tiempo prueba: 7.71 segundos


##### 6.5.14.10 Generación de TVAE V1-C

In [ ]:
inicio = time.time()

df_tvae_v1c_base = tvae.sample_from_conditions(
    conditions=[
        condition_target_0,
        condition_target_1
    ]
)

fin = time.time()

tiempo_generacion_tvae_v1c = fin - inicio

print(
    "Shape:",
    df_tvae_v1c_base.shape
)

print("\nTARGET:")
print(
    df_tvae_v1c_base["TARGET"]
    .value_counts()
    .sort_index()
)

print("\nTARGET proporciones:")
print(
    df_tvae_v1c_base["TARGET"]
    .value_counts(normalize=True)
    .sort_index()
)

print(
    f"\nTiempo generación condicionada TVAE: "
    f"{tiempo_generacion_tvae_v1c / 60:.2f} minutos"
)

Sampling conditions: 100%|██████████| 307507/307507 [02:05<00:00, 2452.52it/s]

Shape: (307507, 33)

TARGET:
TARGET
0    282682
1     24825
Name: count, dtype: int64

TARGET proporciones:
TARGET
0    0.91927
1    0.08073
Name: proportion, dtype: float64

Tiempo generación condicionada TVAE: 2.09 minutos


In [ ]:
#Mezclamos
df_tvae_v1c_base = (
    df_tvae_v1c_base
    .sample(
        frac=1,
        random_state=RANDOM_STATE
    )
    .reset_index(drop=True)
)

In [ ]:
df_tvae_v1c = reconstruir_variables_derivadas(
    df_tvae_v1c_base
)

df_tvae_v1c = df_tvae_v1c[
    df_real.columns
]

print(
    "TVAE V1-C:",
    df_tvae_v1c.shape
)

print(
    "Mismas columnas:",
    df_tvae_v1c.columns.tolist()
    == df_real.columns.tolist()
)

print(
    "Nulos:",
    df_tvae_v1c.isna().sum().sum()
)

print(
    "Duplicados:",
    df_tvae_v1c.duplicated().sum()
)

for col in derived_cols:
    print(
        f"{col:25s} | "
        f"NaN: {df_tvae_v1c[col].isna().sum():6d} | "
        f"Inf: {np.isinf(df_tvae_v1c[col]).sum():6d}"
    )

TVAE V1-C: (307507, 41)
Mismas columnas: True
Nulos: 0
Duplicados: 0
AGE                       | NaN:      0 | Inf:      0
EMPLOYMENT_YEARS          | NaN:      0 | Inf:      0
CREDIT_INCOME_RATIO       | NaN:      0 | Inf:      0
ANNUITY_INCOME_RATIO      | NaN:      0 | Inf:      0
CREDIT_GOODS_RATIO        | NaN:      0 | Inf:      0
INCOME_PER_FAMILY         | NaN:      0 | Inf:      0
CHILDREN_RATIO            | NaN:      0 | Inf:      0
CREDIT_TERM_EST           | NaN:      0 | Inf:      0


##### 6.5.14.11 TSTR de TVAE V1-C

In [ ]:
# =========================================================
# 6.5.14.11 TSTR - TVAE V1-C
# Train Synthetic Conditional, Test Real
# =========================================================

X_tvae_v1c = df_tvae_v1c.drop(
    columns=["TARGET", "SK_ID_CURR"]
).copy()

y_tvae_v1c = df_tvae_v1c["TARGET"].copy()

print(
    "X TVAE V1-C:",
    X_tvae_v1c.shape
)

print(
    "y TVAE V1-C:",
    y_tvae_v1c.shape
)

print("\nTARGET:")
print(
    y_tvae_v1c
    .value_counts()
    .sort_index()
)

X TVAE V1-C: (307507, 39)
y TVAE V1-C: (307507,)

TARGET:
TARGET
0    282682
1     24825
Name: count, dtype: int64


In [ ]:
#el encoder se ajusta únicamente al training sintético
categorical_cols_tvae_v1c = (
    X_tvae_v1c
    .select_dtypes(include=["object", "category"])
    .columns
    .tolist()
)

preprocessor_tvae_v1c = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_cols_tvae_v1c
        )
    ],
    remainder="passthrough"
)

X_tvae_v1c_enc = (
    preprocessor_tvae_v1c.fit_transform(
        X_tvae_v1c
    )
)

X_test_real_tvae_v1c_enc = (
    preprocessor_tvae_v1c.transform(
        X_test_real
    )
)

print(
    "TVAE V1-C train encoded:",
    X_tvae_v1c_enc.shape
)

print(
    "Real test encoded:",
    X_test_real_tvae_v1c_enc.shape
)

print(
    "Features compatibles:",
    X_tvae_v1c_enc.shape[1]
    == X_test_real_tvae_v1c_enc.shape[1]
)

TVAE V1-C train encoded: (307507, 123)
Real test encoded: (61502, 123)
Features compatibles: True


In [ ]:
#Aplicamos LighGBM
modelo_tstr_tvae_v1c = crear_modelo_diagnostico()

modelo_tstr_tvae_v1c.fit(
    X_tvae_v1c_enc,
    y_tvae_v1c
)

proba_tstr_tvae_v1c = (
    modelo_tstr_tvae_v1c.predict_proba(
        X_test_real_tvae_v1c_enc
    )[:, 1]
)

auc_tstr_tvae_v1c = roc_auc_score(
    y_test_real,
    proba_tstr_tvae_v1c
)

print(
    f"TSTR ROC-AUC TVAE V1-C: "
    f"{auc_tstr_tvae_v1c:.6f}"
)

TSTR ROC-AUC TVAE V1-C: 0.670177


##### 6.5.14.12 Comparación TVAE V1 vs V1-C

In [ ]:
comparacion_tvae_cond = pd.DataFrame({
    "Version": [
        "TVAE V1",
        "TVAE V1-C"
    ],
    "TARGET_1_pct": [
        df_tvae_v1["TARGET"].mean() * 100,
        df_tvae_v1c["TARGET"].mean() * 100
    ],
    "TSTR_ROC_AUC": [
        auc_tstr_tvae,
        auc_tstr_tvae_v1c
    ]
})

comparacion_tvae_cond["Gap_vs_TRTR"] = (
    AUC_TRTR_BASELINE
    - comparacion_tvae_cond["TSTR_ROC_AUC"]
)

comparacion_tvae_cond["TSTR_Relativo_pct"] = (
    comparacion_tvae_cond["TSTR_ROC_AUC"]
    / AUC_TRTR_BASELINE
    * 100
)

comparacion_tvae_cond.round(6)

,Version,TARGET_1_pct,TSTR_ROC_AUC,Gap_vs_TRTR,TSTR_Relativo_pct
0,TVAE V1,0.736894,0.660415,0.105185,86.261095
1,TVAE V1-C,8.072987,0.670177,0.095423,87.536227


In [ ]:
delta_tstr_tvae_cond = (
    auc_tstr_tvae_v1c
    - auc_tstr_tvae
)

print(
    f"TVAE V1 TSTR:   "
    f"{auc_tstr_tvae:.6f}"
)

print(
    f"TVAE V1-C TSTR: "
    f"{auc_tstr_tvae_v1c:.6f}"
)

print(
    f"Cambio absoluto TSTR: "
    f"{delta_tstr_tvae_cond:+.6f}"
)

print(
    f"Diferencia V1-C vs Gaussian V2.1: "
    f"{auc_tstr_tvae_v1c - AUC_TSTR_GC_V21:+.6f}"
)

print(
    f"Diferencia V1-C vs CTGAN V1-C: "
    f"{auc_tstr_tvae_v1c - auc_tstr_ctgan_v1c:+.6f}"
)

TVAE V1 TSTR:   0.660415
TVAE V1-C TSTR: 0.670177
Cambio absoluto TSTR: +0.009762
Diferencia V1-C vs Gaussian V2.1: -0.056497
Diferencia V1-C vs CTGAN V1-C: -0.030465


##### 6.5.14.13 TSTS de TVAE V1-C

In [ ]:
X_train_tvae_v1c, X_test_tvae_v1c, \
y_train_tvae_v1c, y_test_tvae_v1c = train_test_split(
    X_tvae_v1c,
    y_tvae_v1c,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_tvae_v1c
)

print(
    "Train:",
    X_train_tvae_v1c.shape
)

print(
    "Test:",
    X_test_tvae_v1c.shape
)

print("\nTARGET train:")
print(
    y_train_tvae_v1c
    .value_counts()
    .sort_index()
)

print("\nTARGET test:")
print(
    y_test_tvae_v1c
    .value_counts()
    .sort_index()
)

Train: (246005, 39)
Test: (61502, 39)

TARGET train:
TARGET
0    226145
1     19860
Name: count, dtype: int64

TARGET test:
TARGET
0    56537
1     4965
Name: count, dtype: int64


In [ ]:
categorical_cols_tsts_tvae_v1c = (
    X_train_tvae_v1c
    .select_dtypes(include=["object", "category"])
    .columns
    .tolist()
)

preprocessor_tsts_tvae_v1c = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_cols_tsts_tvae_v1c
        )
    ],
    remainder="passthrough"
)

X_train_tvae_v1c_enc = (
    preprocessor_tsts_tvae_v1c.fit_transform(
        X_train_tvae_v1c
    )
)

X_test_tvae_v1c_enc = (
    preprocessor_tsts_tvae_v1c.transform(
        X_test_tvae_v1c
    )
)

print(
    "Synthetic train encoded:",
    X_train_tvae_v1c_enc.shape
)

print(
    "Synthetic test encoded:",
    X_test_tvae_v1c_enc.shape
)

Synthetic train encoded: (246005, 122)
Synthetic test encoded: (61502, 122)


In [ ]:
modelo_tsts_tvae_v1c = crear_modelo_diagnostico()

modelo_tsts_tvae_v1c.fit(
    X_train_tvae_v1c_enc,
    y_train_tvae_v1c
)

proba_tsts_tvae_v1c = (
    modelo_tsts_tvae_v1c.predict_proba(
        X_test_tvae_v1c_enc
    )[:, 1]
)

auc_tsts_tvae_v1c = roc_auc_score(
    y_test_tvae_v1c,
    proba_tsts_tvae_v1c
)

print(
    f"TSTS ROC-AUC TVAE V1-C: "
    f"{auc_tsts_tvae_v1c:.6f}"
)

TSTS ROC-AUC TVAE V1-C: 0.993802


In [ ]:
print("===== TVAE V1-C =====")

print(
    f"TSTR: {auc_tstr_tvae_v1c:.6f}"
)

print(
    f"TSTS: {auc_tsts_tvae_v1c:.6f}"
)

print(
    f"Gap TSTR-TSTS: "
    f"{auc_tstr_tvae_v1c - auc_tsts_tvae_v1c:.6f}"
)

print(
    f"TSTR relativo TRTR: "
    f"{auc_tstr_tvae_v1c / AUC_TRTR_BASELINE * 100:.2f}%"
)

print(
    f"TSTS relativo TRTR: "
    f"{auc_tsts_tvae_v1c / AUC_TRTR_BASELINE * 100:.2f}%"
)

===== TVAE V1-C =====
TSTR: 0.670177
TSTS: 0.993802
Gap TSTR-TSTS: -0.323625
TSTR relativo TRTR: 87.54%
TSTS relativo TRTR: 129.81%


| Método condicionado      | TARGET=1 |         TSTR |     TSTS | Gap TSTR−TSTS |
| ------------------------ | -------: | -----------: | -------: | ------------: |
| **Gaussian Copula V2.1** |   8.073% | **0.726674** | 0.621809 |     +0.104865 |
| **CTGAN V1-C**           |   8.073% | **0.700643** | 0.775625 |     −0.074982 |
| **TVAE V1-C**            |   8.073% | **0.670177** | 0.993802 |     −0.323625 |


Por TSTR, nuestro ranking controlado queda:

Gaussian Copula V2.1 > CTGAN V1-C > TVAE V1-C

Y esto es mucho más defendible que nuestra comparación anterior.

TVAE pasó de:

0.737% positivos → 8.073% positivos

y aun así solamente consiguió:

TSTR = 0.670177

Por tanto, corregir la prevalencia marginal no solucionó el problema principal de transferibilidad de TVAE.

Continúa:

0.056497 por debajo de Gaussian.
0.030465 por debajo de CTGAN condicionado.

Esto proporciona evidencia mucho más sólida de que las limitaciones de TVAE no pueden explicarse únicamente por su colapso inicial de la distribución de TARGET.

| TVAE |         TSTR |         TSTS |
| ---- | -----------: | -----------: |
| V1   |     0.660415 |     0.990961 |
| V1-C | **0.670177** | **0.993802** |


el enorme desacoplamiento permanece:

TSTR = 0.6702

frente a

TSTS = 0.9938

Gap:

−0.323625

Esto hace muy difícil defender TVAE como sustituto para entrenamiento sobre datos reales, al menos con la configuración evaluada.

TVAE genera un universo sintético en el cual TARGET es casi perfectamente discriminable, pero las relaciones que permiten esa discriminación no se transfieren suficientemente al real.

Antes teníamos:

TVAE V1 → 118

Ahora:

TVAE V1-C → 123

Eso indica que el muestreo condicionado produjo algunas categorías que no aparecieron en la generación espontánea.

Pero todavía estamos lejos de las:

150 de CTGAN V1-C.

No debemos interpretar directamente 123 < 150 como “peor calidad”, porque el número de columnas one-hot depende de las categorías presentes. Pero sí es coherente con nuestro diagnóstico anterior de que TVAE representa un espacio categórico menos diverso.

Además, que el TSTS use 122 y el TSTR 123 tampoco es extraño: en TSTS el encoder se ajusta solamente al 80% sintético de entrenamiento y alguna categoría extremadamente rara puede haber quedado únicamente en el 20% de test.

**¿Estaba bien comparar CTGAN y TVAE contra Gaussian Copula V2.1 si a Gaussian V2.1 le habíamos controlado TARGET?**

La comparación inicial era útil pero metodológicamente asimétrica.

Por eso realizamos el experimento adicional.

Y después de controlar TARGET en los tres:

|          | Gaussian V2.1 | CTGAN V1-C | TVAE V1-C |
| -------- | ------------: | ---------: | --------: |
| TARGET=1 |        8.073% |     8.073% |    8.073% |
| TSTR     |  **0.726674** |   0.700643 |  0.670177 |


la conclusión no cambia.

Eso es excelente para la robustez de nuestro análisis.

Ahora podemos decir:

La mayor utilidad TSTR de Gaussian Copula V2.1 frente a CTGAN y TVAE se mantiene incluso cuando se controla explícitamente la distribución marginal de TARGET.

Mucho más sólido que lo que podíamos afirmar antes.

Pero todavía evitaría:

“Gaussian Copula es universalmente mejor que CTGAN y TVAE.”

Porque evaluamos configuraciones concretas, no todo el espacio de hiperparámetros de cada arquitectura.

**Comparacion Natural vs condicionada**

| Sintetizador         | Estrategia   | TARGET=1 |         TSTR |         TSTS |
| -------------------- | ------------ | -------: | -----------: | -----------: |
| Gaussian Copula V2.1 | Condicionada |   8.073% | **0.726674** |     0.621809 |
| CTGAN V1             | Natural      |   7.508% |     0.699521 |     0.758667 |
| CTGAN V1-C           | Condicionada |   8.073% | **0.700643** | **0.775625** |
| TVAE V1              | Natural      |   0.737% |     0.660415 |     0.990961 |
| TVAE V1-C            | Condicionada |   8.073% | **0.670177** | **0.993802** |


Esto permite separar dos capacidades distintas:

Capacidad generativa espontánea: qué tan bien reproduce cada sintetizador TARGET sin intervención.

Utilidad bajo prevalencia controlada: qué ocurre cuando eliminamos la diferencia marginal de TARGET como factor de comparación.

**Consulsion Experimental TARGET controlada**


La comparación inicial entre los sintetizadores presentaba una diferencia metodológica relevante: Gaussian Copula V2.1 utilizaba muestreo condicionado por TARGET, mientras que CTGAN V1 y TVAE V1 habían sido evaluados inicialmente mediante generación espontánea.

Para controlar esta diferencia se generaron CTGAN V1-C y TVAE V1-C utilizando los mismos modelos previamente entrenados, pero condicionando el muestreo para reproducir exactamente la distribución de TARGET observada en el dataset real: 91.927 % para la clase 0 y 8.073 % para la clase 1.

En CTGAN, el condicionamiento produjo un cambio reducido en la capacidad de transferencia. El ROC-AUC TSTR pasó de 0.699521 a 0.700643, una mejora absoluta de 0.001122. Por tanto, la pequeña desviación inicial de la prevalencia de TARGET no explica de forma relevante la diferencia de rendimiento frente a Gaussian Copula V2.1.

En TVAE, el efecto fue mayor. El sintetizador había generado espontáneamente únicamente un 0.737 % de observaciones positivas. Al controlar TARGET hasta alcanzar el 8.073 %, el TSTR aumentó de 0.660415 a 0.670177, una mejora absoluta de 0.009762. Sin embargo, este incremento no fue suficiente para alcanzar el rendimiento de CTGAN V1-C ni de Gaussian Copula V2.1.

Bajo una distribución marginal de TARGET controlada, los resultados TSTR fueron 0.726674 para Gaussian Copula V2.1, 0.700643 para CTGAN V1-C y 0.670177 para TVAE V1-C.

Por tanto, el ranking de transferibilidad observado inicialmente se mantiene después de controlar la prevalencia de la variable objetivo:

**Gaussian Copula V2.1 > CTGAN V1-C > TVAE V1-C.**

Los resultados muestran que la distribución marginal de TARGET influye en distinta medida sobre los sintetizadores, especialmente en TVAE, pero no resulta suficiente para explicar las diferencias de transferibilidad observadas.

Asimismo, CTGAN V1-C obtuvo un TSTS de 0.775625 y TVAE V1-C un TSTS de 0.993802, mientras que sus respectivos TSTR fueron considerablemente inferiores. Esto refuerza la necesidad de diferenciar entre utilidad predictiva interna y capacidad de transferencia hacia datos reales.

En consecuencia, para la comparación global se utilizarán principalmente las versiones con TARGET controlado, mientras que las versiones de generación espontánea se conservarán como diagnóstico de la capacidad de cada sintetizador para reproducir autónomamente la distribución de la variable objetivo.

<h1><bold>Consideración sobre la comparabilidad entre versiones</bold></h1>

Las versiones desarrolladas para cada sintetizador no presentan una secuencia experimental completamente simétrica.

Gaussian Copula fue utilizado inicialmente como método base y sometido a varias iteraciones de diagnóstico y optimización (V1, V2 y V2.1), mientras que CTGAN y TVAE fueron incorporados posteriormente como sintetizadores alternativos y evaluados inicialmente mediante generación espontánea.

Por esta razón, las versiones V1 de CTGAN y TVAE no deben interpretarse como equivalentes directos de Gaussian Copula V1 o V2.

La comparación principal entre arquitecturas se realiza utilizando un experimento controlado sobre la distribución marginal de TARGET:

- Gaussian Copula V2.1
- CTGAN V1-C
- TVAE V1-C

En los tres casos se reproduce la misma prevalencia de TARGET observada en el dataset real, permitiendo reducir una fuente relevante de asimetría experimental.

Las versiones previas de Gaussian Copula se conservan como parte del proceso iterativo de optimización y diagnóstico dentro del enfoque CRISP-DM, mientras que las versiones espontáneas de CTGAN y TVAE se utilizan como evidencia adicional sobre la capacidad autónoma de cada sintetizador para reproducir la variable objetivo.

### 6.6 Comparación global de sintetizadores

Una vez evaluadas las tres familias de generación sintética consideradas en el estudio —Gaussian Copula, CTGAN y TVAE—, se realiza una comparación global de sus resultados.

La comparación considera diferentes dimensiones de calidad, dado que la similitud estadística de un dataset sintético no implica necesariamente la conservación de su utilidad predictiva.

Se analizarán las siguientes dimensiones:

1. Fidelidad estadística de variables relevantes.
2. Preservación de dependencias estructurales.
3. Conservación de relaciones entre variables predictivas y TARGET.
4. Utilidad predictiva mediante TSTR y TSTS.
5. Coste computacional.
6. Evaluación conjunta de los resultados.

Gaussian Copula fue sometido previamente a un proceso iterativo de diagnóstico y optimización (V1, V2 y V2.1), mientras que CTGAN y TVAE fueron incorporados posteriormente como familias generativas alternativas. Por ello, sus secuencias de versiones no representan experimentos completamente simétricos.

Para reducir una fuente relevante de asimetría en la comparación principal, se utilizan las versiones con distribución marginal de TARGET controlada: Gaussian Copula V2.1, CTGAN V1-C y TVAE V1-C.

En los tres datasets, TARGET reproduce la prevalencia observada en los datos reales, permitiendo centrar la comparación en la capacidad de cada proceso generativo para preservar la estructura estadística y predictiva del problema.

#### 6.6.1 Armonización de las métricas estadísticas

Durante las etapas anteriores se identificaron pequeñas diferencias entre determinadas métricas calculadas en distintas iteraciones del proceso experimental.

Estas diferencias pueden estar asociadas a cambios realizados durante la preparación de los datos, tratamiento semántico de variables o procedimientos empleados en las diferentes etapas del análisis.

Para garantizar una comparación homogénea, las métricas estadísticas de Gaussian Copula V2.1, CTGAN V1-C y TVAE V1-C se recalculan en esta sección utilizando:

- el mismo dataset real de referencia;
- las mismas variables;
- las mismas funciones de cálculo;
- el mismo coeficiente de correlación;
- y el mismo procedimiento experimental.

Los valores obtenidos en esta sección constituyen la referencia común utilizada en la comparación global de sintetizadores.

##### 6.6.1.1 Verificación de datasets

In [ ]:
datasets_comparacion = {
    "Real": df_real,
    "Gaussian Copula V2.1": df_gc_v21,
    "CTGAN V1-C": df_ctgan_v1c,
    "TVAE V1-C": df_tvae_v1c
}

for nombre, df in datasets_comparacion.items():

    print(f"\n{nombre}")
    print("Shape:", df.shape)
    print("Nulos:", df.isna().sum().sum())
    print("Duplicados:", df.duplicated().sum())

    print(
        "TARGET=1:",
        round(df["TARGET"].mean() * 100, 4),
        "%"
    )


Real
Shape: (307507, 41)
Nulos: 0
Duplicados: 0
TARGET=1: 8.073 %

Gaussian Copula V2.1
Shape: (307507, 41)
Nulos: 0
Duplicados: 0
TARGET=1: 8.073 %

CTGAN V1-C
Shape: (307507, 41)
Nulos: 0
Duplicados: 0
TARGET=1: 8.073 %

TVAE V1-C
Shape: (307507, 41)
Nulos: 0
Duplicados: 0
TARGET=1: 8.073 %


##### 6.6.1.2 Una única función para correlaciones

In [ ]:
#Usaremos explícitamente Spearman.
def correlacion_spearman(df, var1, var2):
    return df[[var1, var2]].corr(
        method="spearman"
    ).iloc[0, 1]

In [ ]:
#Primero comprobamos las dos relaciones estructurales que nos causaron la discrepancia:
relaciones_estructurales = [
    (
        "CNT_CHILDREN",
        "CNT_FAM_MEMBERS"
    ),
    (
        "DEF_30_CNT_SOCIAL_CIRCLE",
        "DEF_60_CNT_SOCIAL_CIRCLE"
    )
]

resultados_estructurales = []

for var1, var2 in relaciones_estructurales:

    for nombre, df in datasets_comparacion.items():

        corr = correlacion_spearman(
            df,
            var1,
            var2
        )

        resultados_estructurales.append({
            "Relacion": f"{var1} ↔ {var2}",
            "Dataset": nombre,
            "Spearman": corr
        })

df_corr_estructurales = pd.DataFrame(
    resultados_estructurales
)

df_corr_estructurales.round(6)

,Relacion,Dataset,Spearman
0,CNT_CHILDREN ↔ CNT_FAM_MEMBERS,Real,0.811554
1,CNT_CHILDREN ↔ CNT_FAM_MEMBERS,Gaussian Copula V2.1,0.448137
2,CNT_CHILDREN ↔ CNT_FAM_MEMBERS,CTGAN V1-C,0.790882
3,CNT_CHILDREN ↔ CNT_FAM_MEMBERS,TVAE V1-C,0.795944
4,DEF_30_CNT_SOCIAL_CIRCLE ↔ DEF_60_CNT_SOCIAL_C...,Real,0.845149
5,DEF_30_CNT_SOCIAL_CIRCLE ↔ DEF_60_CNT_SOCIAL_C...,Gaussian Copula V2.1,0.072378
6,DEF_30_CNT_SOCIAL_CIRCLE ↔ DEF_60_CNT_SOCIAL_C...,CTGAN V1-C,0.762225
7,DEF_30_CNT_SOCIAL_CIRCLE ↔ DEF_60_CNT_SOCIAL_C...,TVAE V1-C,0.747427


In [ ]:
tabla_estructurales = (
    df_corr_estructurales
    .pivot(
        index="Relacion",
        columns="Dataset",
        values="Spearman"
    )
)

tabla_estructurales.round(6)

Dataset,CTGAN V1-C,Gaussian Copula V2.1,Real,TVAE V1-C
Relacion,,,,
CNT_CHILDREN ↔ CNT_FAM_MEMBERS,0.790882,0.448137,0.811554,0.795944
DEF_30_CNT_SOCIAL_CIRCLE ↔ DEF_60_CNT_SOCIAL_CIRCLE,0.762225,0.072378,0.845149,0.747427


##### 6.6.1.3 Error absoluto respecto al real

In [ ]:
#Esto es incluso más útil que mirar solamente las correlaciones
for sintetizador in [
    "Gaussian Copula V2.1",
    "CTGAN V1-C",
    "TVAE V1-C"
]:
    tabla_estructurales[
        f"Error {sintetizador}"
    ] = abs(
        tabla_estructurales[sintetizador]
        - tabla_estructurales["Real"]
    )

tabla_estructurales.round(6)

Dataset,CTGAN V1-C,Gaussian Copula V2.1,Real,TVAE V1-C,Error Gaussian Copula V2.1,Error CTGAN V1-C,Error TVAE V1-C
Relacion,,,,,,,
CNT_CHILDREN ↔ CNT_FAM_MEMBERS,0.790882,0.448137,0.811554,0.795944,0.363417,0.020672,0.015610
DEF_30_CNT_SOCIAL_CIRCLE ↔ DEF_60_CNT_SOCIAL_CIRCLE,0.762225,0.072378,0.845149,0.747427,0.772771,0.082925,0.097722


##### 6.6.1.4 Relaciones críticas con TARGET

In [ ]:
#Vamos a utilizar exactamente las siete que venimos siguiendo
variables_criticas = [
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "EXT_SOURCE_1",
    "DAYS_BIRTH",
    "CREDIT_GOODS_RATIO",
    "DAYS_EMPLOYED",
    "CREDIT_TERM_EST"
]

In [ ]:
resultados_target = []

for variable in variables_criticas:

    for nombre, df in datasets_comparacion.items():

        corr = correlacion_spearman(
            df,
            variable,
            "TARGET"
        )

        resultados_target.append({
            "Variable": variable,
            "Dataset": nombre,
            "Spearman_TARGET": corr
        })

df_corr_target = pd.DataFrame(
    resultados_target
)

tabla_target = (
    df_corr_target
    .pivot(
        index="Variable",
        columns="Dataset",
        values="Spearman_TARGET"
    )
)

tabla_target.round(6)

Dataset,CTGAN V1-C,Gaussian Copula V2.1,Real,TVAE V1-C
Variable,,,,
CREDIT_GOODS_RATIO,-0.002051,0.034936,0.066684,0.041553
CREDIT_TERM_EST,-0.005514,-0.011080,-0.030262,-0.030026
DAYS_BIRTH,0.134817,0.036745,0.078331,0.079388
DAYS_EMPLOYED,0.080472,0.036845,0.075036,0.256500
EXT_SOURCE_1,-0.119185,-0.051703,-0.085389,-0.047889
EXT_SOURCE_2,-0.095178,-0.081960,-0.147133,-0.221671
EXT_SOURCE_3,-0.135870,-0.078247,-0.141501,-0.357425


In [ ]:
sintetizadores_controlados = [
    "Gaussian Copula V2.1",
    "CTGAN V1-C",
    "TVAE V1-C"
]

for sintetizador in sintetizadores_controlados:

    tabla_target[
        f"Error {sintetizador}"
    ] = abs(
        tabla_target[sintetizador]
        - tabla_target["Real"]
    )

tabla_target.round(6)

Dataset,CTGAN V1-C,Gaussian Copula V2.1,Real,TVAE V1-C,Error Gaussian Copula V2.1,Error CTGAN V1-C,Error TVAE V1-C
Variable,,,,,,,
CREDIT_GOODS_RATIO,-0.002051,0.034936,0.066684,0.041553,0.031748,0.068735,0.025131
CREDIT_TERM_EST,-0.005514,-0.011080,-0.030262,-0.030026,0.019182,0.024749,0.000236
DAYS_BIRTH,0.134817,0.036745,0.078331,0.079388,0.041586,0.056486,0.001057
DAYS_EMPLOYED,0.080472,0.036845,0.075036,0.256500,0.038191,0.005436,0.181464
EXT_SOURCE_1,-0.119185,-0.051703,-0.085389,-0.047889,0.033687,0.033795,0.037501
EXT_SOURCE_2,-0.095178,-0.081960,-0.147133,-0.221671,0.065173,0.051955,0.074537
EXT_SOURCE_3,-0.135870,-0.078247,-0.141501,-0.357425,0.063254,0.005631,0.215924


##### 6.6.1.5 MAE común de relaciones TARGET-X

In [ ]:
mae_target = {}

for sintetizador in sintetizadores_controlados:

    errores = abs(
        tabla_target[sintetizador]
        - tabla_target["Real"]
    )

    mae_target[sintetizador] = errores.mean()

df_mae_target = (
    pd.DataFrame.from_dict(
        mae_target,
        orient="index",
        columns=["MAE_TARGET_X"]
    )
    .sort_values("MAE_TARGET_X")
)

df_mae_target.round(6)

,MAE_TARGET_X
CTGAN V1-C,0.035255
Gaussian Copula V2.1,0.041832
TVAE V1-C,0.076550


menor MAE = mejor preservación de estas relaciones bivariadas seleccionadas.

Pero cuidado: no convertiremos este MAE en nuestra métrica principal de calidad.

Ya aprendimos con CTGAN que una buena preservación de correlaciones seleccionadas no garantiza el mejor TSTR.

##### 6.6.1.6 Tabla de control de TARGET

In [ ]:
tabla_target_dist = pd.DataFrame({
    nombre: {
        "TARGET_0_pct":
            (df["TARGET"] == 0).mean() * 100,

        "TARGET_1_pct":
            (df["TARGET"] == 1).mean() * 100
    }
    for nombre, df in datasets_comparacion.items()
}).T

tabla_target_dist.round(6)

,TARGET_0_pct,TARGET_1_pct
Real,91.927013,8.072987
Gaussian Copula V2.1,91.927013,8.072987
CTGAN V1-C,91.927013,8.072987
TVAE V1-C,91.927013,8.072987


1. La armonización resolvió la discrepancia

Todos los datasets están ahora bajo exactamente el mismo marco:

| Dataset              |   Filas | Variables | Nulos | Duplicados | TARGET=1 |
| -------------------- | ------: | --------: | ----: | ---------: | -------: |
| Real                 | 307,507 |        41 |     0 |          0 |   8.073% |
| Gaussian Copula V2.1 | 307,507 |        41 |     0 |          0 |   8.073% |
| CTGAN V1-C           | 307,507 |        41 |     0 |          0 |   8.073% |
| TVAE V1-C            | 307,507 |        41 |     0 |          0 |   8.073% |


Esto es exactamente lo que queríamos antes de construir la comparación final.

A partir de ahora, yo descartaría para las tablas finales los antiguos valores estadísticos del Notebook 05 y utilizaría estos valores recalculados en Notebook 06.

Los anteriores siguen siendo válidos como parte histórica del diagnóstico que llevó a V2.1, pero no como números para comparar directamente las tres arquitecturas.

2. Dependencias estructurales: Gaussian queda claramente atrás

Los resultados armonizados son:

| Relación          |     Real |  GC V2.1 |   CTGAN V1-C |    TVAE V1-C |
| ----------------- | -------: | -------: | -----------: | -----------: |
| CHILDREN ↔ FAMILY | 0.811554 | 0.448137 | **0.790882** | **0.795944** |
| DEF30 ↔ DEF60     | 0.845149 | 0.072378 | **0.762225** |     0.747427 |


Errores:

| Relación          |           GC |        CTGAN |         TVAE |
| ----------------- | -----------: | -----------: | -----------: |
| CHILDREN ↔ FAMILY | **0.363417** |     0.020672 | **0.015610** |
| DEF30 ↔ DEF60     | **0.772771** | **0.082925** |     0.097722 |


Esto es bastante contundente.

Gaussian Copula V2.1 conserva muy mal esas dos dependencias estructurales, especialmente:

DEF30 ↔ DEF60

Real:

0.845149

Gaussian:

0.072378

Prácticamente pierde la asociación.

En cambio CTGAN y TVAE conservan bastante bien las dos estructuras.

Esto también confirma algo que habíamos sospechado desde Notebook 05: mejorar Gaussian V2.1 ayudó a su utilidad predictiva, pero no solucionó todas sus limitaciones multivariantes.

3. Relaciones con TARGET: aquí aparece una historia distinta

El MAE armonizado queda:

| Sintetizador         | MAE TARGET-X ↓ |
| -------------------- | -------------: |
| **CTGAN V1-C**       |   **0.035255** |
| Gaussian Copula V2.1 |   **0.041832** |
| TVAE V1-C            |   **0.076550** |


El menor MAE para las relaciones TARGET-X corresponde a CTGAN V1-C.

Y esto es muy interesante porque el ranking TSTR es:

| Sintetizador      |       TSTR ↑ |
| ----------------- | -----------: |
| **Gaussian V2.1** | **0.726674** |
| CTGAN V1-C        |     0.700643 |
| TVAE V1-C         |     0.670177 |


Por tanto:

El sintetizador con menor error medio en las correlaciones bivariadas seleccionadas con TARGET no es el sintetizador con mayor TSTR.

Esto es un hallazgo metodológico importante de tu experimento.

Significa que preservar siete correlaciones individuales razonablemente bien no garantiza preservar toda la estructura multivariante que utiliza LightGBM para generalizar hacia datos reales.

Y eso justifica perfectamente que no seleccionemos el sintetizador solamente por correlaciones.

4. TVAE tiene un comportamiento especialmente revelador

Mira algunas relaciones:

| Variable        |      Real |     TVAE V1-C |
| --------------- | --------: | ------------: |
| DAYS_BIRTH      |  0.078331 |  **0.079388** |
| CREDIT_TERM_EST | −0.030262 | **−0.030026** |


Son prácticamente perfectas.

Pero luego:

| Variable      |      Real |     TVAE V1-C |
| ------------- | --------: | ------------: |
| DAYS_EMPLOYED |  0.075036 |  **0.256500** |
| EXT_SOURCE_2  | −0.147133 | **−0.221671** |
| EXT_SOURCE_3  | −0.141501 | **−0.357425** |


Aquí TVAE amplifica enormemente algunas asociaciones.

Especialmente:

EXT_SOURCE_3: -0.1415 → -0.3574

Eso es muy interesante considerando:

TSTS = 0.993802

Sin afirmar causalidad, ahora sí tenemos evidencia estadística compatible con nuestra interpretación anterior: TVAE V1-C presenta algunas relaciones con TARGET considerablemente más intensas que en el dataset real.

Eso puede contribuir a explicar por qué TARGET resulta extremadamente discriminable dentro del entorno TVAE.

Pero mantendría “puede contribuir”, no “explica”, porque estamos observando correlaciones bivariadas y el LightGBM trabaja multivariadamente.

5. CTGAN también tiene una anomalía interesante

CTGAN conserva extraordinariamente:

EXT_SOURCE_3

Real:

−0.141501

CTGAN:

−0.135870

Error:

0.005631

Y DAYS_EMPLOYED:

Real 0.075036

CTGAN 0.080472

Error:

0.005436

Pero falla bastante en:

CREDIT_GOODS_RATIO

Real 0.066684

CTGAN −0.002051

Aquí incluso prácticamente desaparece y cambia ligeramente de signo.

También sobrerrepresenta:

DAYS_BIRTH

Real 0.078331

CTGAN 0.134817.

Por eso el MAE resume, pero no debemos ocultar los comportamientos individuales.

6. Y Gaussian nos da otra lección

Gaussian tiene:

peor fidelidad estructural que los modelos neuronales.

Tiene:

MAE TARGET-X intermedio.

Pero tiene:

mejor TSTR.

Eso podría parecer contradictorio, pero no lo es.

Lo que estamos midiendo son dimensiones diferentes:

Fidelidad estructural seleccionada
→ dos pares concretos.

MAE TARGET-X
→ siete relaciones bivariadas concretas.

TSTR
→ capacidad conjunta de todas las variables sintéticas para entrenar un modelo que generalice hacia observaciones reales.

Por eso TSTR es especialmente relevante para nuestra pregunta de negocio:

¿Puede el dataset sintético sustituir al real como fuente de entrenamiento?

**Resultados de la armonización**

La recalculación homogénea de las métricas permitió establecer una única referencia estadística para la comparación de Gaussian Copula V2.1, CTGAN V1-C y TVAE V1-C.

Los cuatro datasets presentan 307,507 registros y 41 variables, sin valores nulos ni registros completamente duplicados. Asimismo, las tres versiones sintéticas controladas reproducen exactamente la prevalencia de TARGET observada en el dataset real (8.073 % para la clase positiva).

En las dependencias estructurales seleccionadas, CTGAN V1-C y TVAE V1-C presentan una conservación considerablemente superior a Gaussian Copula V2.1.

Para la relación CNT_CHILDREN–CNT_FAM_MEMBERS, el valor real de Spearman es 0.811554, frente a 0.448137 en Gaussian Copula, 0.790882 en CTGAN y 0.795944 en TVAE.

La diferencia resulta todavía mayor en DEF_30_CNT_SOCIAL_CIRCLE–DEF_60_CNT_SOCIAL_CIRCLE, cuya correlación real es 0.845149. Gaussian Copula V2.1 obtiene únicamente 0.072378, mientras que CTGAN V1-C y TVAE V1-C alcanzan 0.762225 y 0.747427 respectivamente.

Respecto a las siete relaciones críticas seleccionadas con TARGET, CTGAN V1-C presenta el menor error absoluto medio (MAE = 0.035255), seguido de Gaussian Copula V2.1 (0.041832) y TVAE V1-C (0.076550).

TVAE muestra desviaciones especialmente relevantes en determinadas asociaciones. Por ejemplo, EXT_SOURCE_3 presenta una correlación de -0.357425 frente a -0.141501 en los datos reales, mientras que DAYS_EMPLOYED aumenta desde 0.075036 hasta 0.256500. Estas relaciones amplificadas son compatibles con la elevada discriminabilidad interna observada posteriormente mediante TSTS, aunque no permiten establecer por sí solas una relación causal.

Los resultados evidencian que las distintas dimensiones de fidelidad no producen un ranking único. CTGAN y TVAE preservan mejor determinadas dependencias estructurales, CTGAN obtiene el menor MAE en las relaciones seleccionadas con TARGET y Gaussian Copula V2.1 mantiene, sin embargo, la mayor capacidad TSTR.

Por tanto, la selección del sintetizador no debe basarse exclusivamente en similitud estadística bivariada, sino considerar conjuntamente fidelidad estadística, preservación estructural y utilidad predictiva.

#### 6.6.2 Comparación de fidelidad estadística

Hasta ahora evaluamos relaciones. Ahora debemos comparar las distribuciones marginales de las variables.

Y para evitar una sección gigantesca con 41 variables, utilizaría las variables críticas que ya justificamos predictivamente.

In [ ]:
variables_numericas_fidelidad = [
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "CREDIT_GOODS_RATIO",
    "CREDIT_TERM_EST"
]

Para una comparación formal entre distribuciones numéricas utilizaría Kolmogorov-Smirnov (KS):

0 → distribuciones muy similares.
valores mayores → mayor divergencia.

No usaría el p-value como criterio principal porque con ~307 mil observaciones prácticamente cualquier diferencia pequeña puede resultar estadísticamente significativa. Nos interesa el estadístico KS como medida de distancia.

La fidelidad marginal de las variables numéricas se evalúa mediante el estadístico de Kolmogorov-Smirnov (KS), complementando las medidas descriptivas utilizadas durante el análisis individual de cada sintetizador.

El estadístico KS cuantifica la máxima diferencia entre las funciones de distribución acumulada de dos muestras. Valores próximos a cero indican una mayor similitud entre las distribuciones.

Debido al elevado tamaño muestral del estudio, el análisis se centra en la magnitud del estadístico KS y no en su p-value, dado que diferencias pequeñas pueden resultar estadísticamente significativas con muestras de gran tamaño.


In [ ]:
from scipy.stats import ks_2samp

variables_numericas_fidelidad = [
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "CREDIT_GOODS_RATIO",
    "CREDIT_TERM_EST"
]

resultados_ks = []

for variable in variables_numericas_fidelidad:

    real_values = df_real[variable].dropna()

    for sintetizador in sintetizadores_controlados:

        synth_values = (
            datasets_comparacion[sintetizador][variable]
            .dropna()
        )

        ks_stat = ks_2samp(
            real_values,
            synth_values
        ).statistic

        resultados_ks.append({
            "Variable": variable,
            "Sintetizador": sintetizador,
            "KS": ks_stat
        })

df_ks = pd.DataFrame(resultados_ks)

tabla_ks = df_ks.pivot(
    index="Variable",
    columns="Sintetizador",
    values="KS"
)

tabla_ks.round(6)

Sintetizador,CTGAN V1-C,Gaussian Copula V2.1,TVAE V1-C
Variable,,,
AMT_ANNUITY,0.053651,0.020663,0.088983
AMT_CREDIT,0.051218,0.053635,0.032770
AMT_GOODS_PRICE,0.105477,0.075299,0.057095
CREDIT_GOODS_RATIO,0.322253,0.187007,0.251575
CREDIT_TERM_EST,0.121269,0.078749,0.118384
DAYS_BIRTH,0.090096,0.046669,0.068298
DAYS_EMPLOYED,0.106264,0.141805,0.225875
EXT_SOURCE_1,0.294575,0.306396,0.299697
EXT_SOURCE_2,0.063013,0.077384,0.156569


In [ ]:
resumen_ks = pd.DataFrame({
    "KS_Medio": tabla_ks.mean(),
    "KS_Mediana": tabla_ks.median(),
    "KS_Maximo": tabla_ks.max()
}).sort_values("KS_Medio")

resumen_ks.round(6)

,KS_Medio,KS_Mediana,KS_Maximo
Sintetizador,,,
Gaussian Copula V2.1,0.113289,0.078067,0.306396
CTGAN V1-C,0.140601,0.105871,0.322253
TVAE V1-C,0.160429,0.137477,0.305047


1. Resultado global de KS

Recordemos que menor KS = mayor similitud entre la distribución real y sintética.

| Sintetizador             |   KS medio ↓ | KS mediana ↓ |  KS máximo ↓ |
| ------------------------ | -----------: | -----------: | -----------: |
| **Gaussian Copula V2.1** | **0.113289** | **0.078067** |     0.306396 |
| CTGAN V1-C               |     0.140601 |     0.105871 |     0.322253 |
| TVAE V1-C                |     0.160429 |     0.137477 | **0.305047** |


Por KS medio y mediana, el ranking es:

Gaussian Copula V2.1 > CTGAN V1-C > TVAE V1-C

Curiosamente, es exactamente el mismo orden que TSTR:

0.726674 > 0.700643 > 0.670177

No debemos afirmar que el mejor KS causa el mejor TSTR, pero sí podemos señalar que ambos criterios producen aquí el mismo orden.

2. Gaussian no gana en todas las variables

Esto es importante para no sobreinterpretar el promedio.

Gaussian obtiene el menor KS en:

AMT_ANNUITY: 0.020663
CREDIT_GOODS_RATIO: 0.187007
CREDIT_TERM_EST: 0.078749
DAYS_BIRTH: 0.046669
EXT_SOURCE_3: 0.145288

TVAE gana en:

AMT_CREDIT: 0.032770
AMT_GOODS_PRICE: 0.057095

CTGAN gana en:

DAYS_EMPLOYED: 0.106264
EXT_SOURCE_1: 0.294575
EXT_SOURCE_2: 0.063013

Así que cada sintetizador gana en ciertas variables, pero Gaussian presenta el mejor comportamiento agregado.

3. Hay variables difíciles para los tres

Especialmente EXT_SOURCE_1:

|          |       KS |
| -------- | -------: |
| CTGAN    | 0.294575 |
| Gaussian | 0.306396 |
| TVAE     | 0.299697 |


Los tres tienen un KS relativamente elevado.

Esto es interesante porque EXT_SOURCE_1 había aparecido como una variable predictivamente importante. Por tanto, aunque Gaussian tenga el mejor resultado agregado, ninguno de los tres reproduce perfectamente todas las distribuciones relevantes.

También tenemos dificultades con CREDIT_GOODS_RATIO:

GC: 0.187007
TVAE: 0.251575
CTGAN: 0.322253

Y recordemos que esta variable es reconstruida determinísticamente. Por eso su desviación no significa que la hayamos sintetizado incorrectamente de manera directa: refleja diferencias en la distribución conjunta de sus variables base, AMT_CREDIT y AMT_GOODS_PRICE.

Eso es metodológicamente muy interesante.

4. El caso de TVAE sigue siendo coherente con el diagnóstico anterior

TVAE presenta el peor KS medio:

0.160429

y además algunos errores elevados:

DAYS_EMPLOYED: 0.225875
EXT_SOURCE_1: 0.299697
EXT_SOURCE_3: 0.305047

Esto complementa perfectamente lo que vimos antes:

TARGET controlado correctamente;
algunas dependencias estructurales muy buenas;
algunas correlaciones TARGET-X fuertemente amplificadas;
TSTS extremadamente alto: 0.993802;
pero TSTR relativamente bajo: 0.670177.

Cada vez tenemos más evidencia de que TVAE genera un dataset internamente muy discriminativo pero con una representación estadística que se aleja del real en dimensiones relevantes.

5. Gaussian presenta un perfil muy particular

Ahora podemos caracterizarlo mejor:

| Dimensión                              | GC V2.1     |
| -------------------------------------- | ----------- |
| Fidelidad marginal numérica            | 🥇 Mejor    |
| Relaciones estructurales seleccionadas | 🔴 Peor     |
| MAE TARGET-X                           | 🥈 Segundo  |
| TSTR                                   | 🥇 Mejor    |
| TSTS                                   | 🔴 Más bajo |


Esto es justamente por lo que no debemos utilizar una única métrica de calidad sintética.

Gaussian conserva mejor las distribuciones marginales y consigue mejor transferencia hacia el real, pero pierde fuertemente algunas dependencias estructurales.

CTGAN parece más equilibrado en estructura y relaciones TARGET-X.

TVAE conserva muy bien algunas estructuras pero presenta mayores desviaciones marginales y un fuerte desacoplamiento TSTS/TSTR.

Esto va a hacer que nuestra 6.7 Selección final tenga bastante sustancia.

##### 6.6.2.1 Fidelidad marginal de variables numéricas

La comparación mediante el estadístico de Kolmogorov-Smirnov muestra diferencias relevantes entre los tres métodos de generación sintética.

Gaussian Copula V2.1 presenta el menor KS medio (0.113289) y la menor mediana (0.078067), seguido de CTGAN V1-C, con un KS medio de 0.140601, y TVAE V1-C, con 0.160429. Por tanto, considerando conjuntamente las diez variables numéricas analizadas, Gaussian Copula V2.1 presenta la mayor similitud marginal con las distribuciones reales.

No obstante, ningún sintetizador domina individualmente en todas las variables. Gaussian Copula obtiene la menor distancia en AMT_ANNUITY, CREDIT_GOODS_RATIO, CREDIT_TERM_EST, DAYS_BIRTH y EXT_SOURCE_3; CTGAN presenta los menores valores en DAYS_EMPLOYED, EXT_SOURCE_1 y EXT_SOURCE_2; mientras que TVAE obtiene los mejores resultados en AMT_CREDIT y AMT_GOODS_PRICE.

Algunas variables presentan desviaciones relevantes para los tres métodos. EXT_SOURCE_1 registra valores KS cercanos o superiores a 0.29 en todos los sintetizadores, evidenciando una mayor dificultad para reproducir su distribución.

También resulta relevante el comportamiento de variables derivadas como CREDIT_GOODS_RATIO y CREDIT_TERM_EST. Estas variables no fueron sintetizadas directamente, sino reconstruidas determinísticamente a partir de variables base. Por ello, sus diferencias respecto al dataset real reflejan principalmente desviaciones en las distribuciones conjuntas de las variables utilizadas para su construcción.

En términos agregados, el ranking obtenido mediante KS medio es:

**Gaussian Copula V2.1 > CTGAN V1-C > TVAE V1-C.**

Este orden coincide con el observado mediante TSTR. Sin embargo, esta coincidencia no implica una relación causal entre fidelidad marginal y capacidad de transferencia, dado que TSTR depende de la estructura multivariante completa utilizada por el modelo predictivo.

##### 6.6.2.2 Fidelidad de variables categóricas/discretas

Nos falta la otra mitad de la fidelidad estadística. Para la comparación de las distribuciones categóricas se utiliza
Total Variation Distance (TVD), dado que KS no resulta apropiada
para variables categóricas nominales.

Propongo utilizar Total Variation Distance (TVD), que compara las distribuciones de probabilidades de las categorías:

$$ TVD(P,Q)=\frac{1}{2}\sum_i |P_i-Q_i| $$

Interpretación:

0 = distribuciones idénticas;
cuanto menor, mejor;
valores mayores representan mayor diferencia entre las proporciones categóricas.

La comparación de las variables categóricas y discretas se realiza mediante Total Variation Distance (TVD).

Esta métrica cuantifica la diferencia global entre dos distribuciones discretas considerando las probabilidades asociadas a todas sus categorías. Un valor de cero representa distribuciones idénticas, mientras que valores mayores indican una mayor divergencia.

A diferencia del análisis numérico mediante KS, TVD permite comparar directamente las proporciones observadas en las diferentes categorías sin depender del orden numérico asignado a estas.

In [ ]:
#las categóricas/discretas relevantes que ya hemos venido observando
variables_categoricas_fidelidad = [
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",
    "DEF_30_CNT_SOCIAL_CIRCLE",
    "DEF_60_CNT_SOCIAL_CIRCLE"
]

In [ ]:
categorical_object_cols = (
    df_real
    .select_dtypes(include=["object", "category"])
    .columns
    .tolist()
)

variables_categoricas_fidelidad = list(dict.fromkeys(
    categorical_object_cols + [
        "CNT_CHILDREN",
        "CNT_FAM_MEMBERS",
        "DEF_30_CNT_SOCIAL_CIRCLE",
        "DEF_60_CNT_SOCIAL_CIRCLE"
    ]
))

print("Número de variables:", len(variables_categoricas_fidelidad))
print(variables_categoricas_fidelidad)

Número de variables: 16
['CODE_GENDER', 'NAME_FAMILY_STATUS', 'NAME_EDUCATION_TYPE', 'NAME_HOUSING_TYPE', 'NAME_INCOME_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_CONTRACT_TYPE', 'NAME_TYPE_SUITE', 'WEEKDAY_APPR_PROCESS_START', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS', 'DEF_30_CNT_SOCIAL_CIRCLE', 'DEF_60_CNT_SOCIAL_CIRCLE']


In [ ]:
#Aplicamos TVD
def calcular_tvd(real, synthetic):

    categorias = real.dropna().unique().tolist()

    categorias_synth = synthetic.dropna().unique().tolist()

    categorias = list(
        set(categorias) | set(categorias_synth)
    )

    p_real = (
        real.value_counts(normalize=True)
        .reindex(categorias, fill_value=0)
    )

    p_synth = (
        synthetic.value_counts(normalize=True)
        .reindex(categorias, fill_value=0)
    )

    return 0.5 * abs(p_real - p_synth).sum()

In [ ]:
resultados_tvd = []

for variable in variables_categoricas_fidelidad:

    for sintetizador in sintetizadores_controlados:

        tvd = calcular_tvd(
            df_real[variable],
            datasets_comparacion[sintetizador][variable]
        )

        resultados_tvd.append({
            "Variable": variable,
            "Sintetizador": sintetizador,
            "TVD": tvd
        })

df_tvd = pd.DataFrame(resultados_tvd)

tabla_tvd = df_tvd.pivot(
    index="Variable",
    columns="Sintetizador",
    values="TVD"
)

tabla_tvd.round(6)

Sintetizador,CTGAN V1-C,Gaussian Copula V2.1,TVAE V1-C
Variable,,,
CNT_CHILDREN,0.076005,0.004189,0.019603
CNT_FAM_MEMBERS,0.094398,0.002172,0.075774
CODE_GENDER,0.131002,0.014822,0.155089
DEF_30_CNT_SOCIAL_CIRCLE,0.085904,0.002354,0.018686
DEF_60_CNT_SOCIAL_CIRCLE,0.066720,0.001454,0.028334
FLAG_OWN_CAR,0.012253,0.004364,0.018549
FLAG_OWN_REALTY,0.051131,0.002475,0.014374
NAME_CONTRACT_TYPE,0.010036,0.002267,0.027655
NAME_EDUCATION_TYPE,0.027486,0.008923,0.118843


In [ ]:
resumen_tvd = pd.DataFrame({
    "TVD_Medio": tabla_tvd.mean(),
    "TVD_Mediana": tabla_tvd.median(),
    "TVD_Maximo": tabla_tvd.max()
}).sort_values("TVD_Medio")

resumen_tvd.round(6)

,TVD_Medio,TVD_Mediana,TVD_Maximo
Sintetizador,,,
Gaussian Copula V2.1,0.004933,0.003182,0.019161
CTGAN V1-C,0.082966,0.071363,0.188828
TVAE V1-C,0.115458,0.091528,0.384235


1. Resultado global de TVD

Recordemos: menor TVD = distribución categórica sintética más parecida a la real.

Sintetizador	TVD medio ↓	TVD mediana ↓	TVD máximo ↓
Gaussian Copula V2.1	0.004933	0.003182	0.019161
CTGAN V1-C	0.082966	0.071363	0.188828
TVAE V1-C	0.115458	0.091528	0.384235

Aquí la diferencia no es pequeña.

Gaussian obtiene un TVD medio de apenas:

0.004933

frente a:

CTGAN: 0.082966
TVAE: 0.115458

Es decir, Gaussian reproduce con enorme precisión las proporciones marginales de las categorías evaluadas.

2. Pero aquí tenemos uno de los resultados más importantes del TFM

Compara esto:

Distribuciones marginales

Para CNT_CHILDREN:

Gaussian TVD = 0.004189

Para CNT_FAM_MEMBERS:

Gaussian TVD = 0.002172

Son excelentes.

Pero cuando anteriormente medimos su relación conjunta:

CNT_CHILDREN ↔ CNT_FAM_MEMBERS

obtuvimos:

Real = 0.811554
Gaussian = 0.448137

Error = 0.363417.

Y el contraste es todavía más extremo para:

DEF_30_CNT_SOCIAL_CIRCLE

TVD Gaussian:

0.002354

DEF_60_CNT_SOCIAL_CIRCLE

TVD Gaussian:

0.001454

Individualmente, sus distribuciones son prácticamente idénticas a las reales.

Pero su relación conjunta:

Real = 0.845149
Gaussian = 0.072378

Error:

0.772771

Este ejemplo es excelente para demostrar empíricamente una de las ideas centrales que venimos defendiendo:

Preservar las distribuciones marginales de las variables no implica preservar su estructura de dependencia.

En tu experimento tenemos una demostración muy clara de ello.

3. CTGAN muestra prácticamente el comportamiento opuesto

Por ejemplo:

CNT_CHILDREN

TVD:

0.076005

CNT_FAM_MEMBERS

TVD:

0.094398

Mucho peores que Gaussian marginalmente.

Pero:

CNT_CHILDREN ↔ CNT_FAM_MEMBERS

CTGAN:

0.790882

Real:

0.811554

Error solamente:

0.020672

Y:

DEF30 ↔ DEF60

CTGAN:

0.762225

frente al real:

0.845149

Error:

0.082925

Por tanto, CTGAN reproduce peor las proporciones individuales, pero preserva mucho mejor determinadas relaciones entre variables.

Eso muestra por qué necesitamos analizar ambas cosas.

4. TVAE vuelve a presentar un comportamiento irregular

TVAE obtiene algunos resultados marginales bastante buenos:

FLAG_OWN_REALTY: 0.014374
FLAG_OWN_CAR: 0.018549
DEF_30: 0.018686
CNT_CHILDREN: 0.019603

Pero tiene desviaciones muy grandes en otras variables:

NAME_INCOME_TYPE: 0.253441
WEEKDAY_APPR_PROCESS_START: 0.384235
ORGANIZATION_TYPE: 0.196987
NAME_TYPE_SUITE: 0.187612
OCCUPATION_TYPE: 0.184871

Por eso su TVD medio termina siendo el peor:

0.115458.

Esto también es consistente con lo que habíamos observado sobre pérdida o distorsión de categorías en TVAE.

##### 6.6.2.3 Fidelidad marginal de variables categóricas y discretas

La comparación mediante Total Variation Distance evidencia diferencias importantes entre los tres métodos.

Gaussian Copula V2.1 presenta la mayor fidelidad marginal categórica, con un TVD medio de 0.004933 y una mediana de 0.003182. CTGAN V1-C obtiene un TVD medio de 0.082966, mientras que TVAE V1-C alcanza 0.115458.

Gaussian Copula presenta valores reducidos de TVD en las 16 variables analizadas, con un máximo de únicamente 0.019161. Esto indica que las proporciones marginales de las categorías generadas son muy próximas a las observadas en el dataset real.

Sin embargo, este resultado debe analizarse conjuntamente con las dependencias estructurales. Por ejemplo, Gaussian Copula presenta TVD de 0.004189 y 0.002172 para CNT_CHILDREN y CNT_FAM_MEMBERS respectivamente, pero la correlación entre ambas variables disminuye desde 0.811554 en los datos reales hasta 0.448137 en los sintéticos.

El contraste resulta todavía mayor para DEF_30_CNT_SOCIAL_CIRCLE y DEF_60_CNT_SOCIAL_CIRCLE. Sus TVD individuales en Gaussian Copula son 0.002354 y 0.001454, mientras que la correlación entre ambas variables disminuye desde 0.845149 hasta 0.072378.

CTGAN presenta el comportamiento contrario en estas relaciones: sus distribuciones marginales son menos precisas, pero conserva considerablemente mejor determinadas dependencias estructurales. Por ejemplo, la correlación CNT_CHILDREN–CNT_FAM_MEMBERS alcanza 0.790882 y DEF_30_CNT_SOCIAL_CIRCLE–DEF_60_CNT_SOCIAL_CIRCLE alcanza 0.762225.

TVAE presenta la mayor divergencia marginal agregada y desviaciones particularmente elevadas en variables como WEEKDAY_APPR_PROCESS_START, NAME_INCOME_TYPE, ORGANIZATION_TYPE, NAME_TYPE_SUITE y OCCUPATION_TYPE.

Estos resultados proporcionan evidencia empírica de que una elevada fidelidad marginal no garantiza la conservación de la estructura multivariante. En consecuencia, ambas dimensiones deben evaluarse separadamente al analizar la calidad de los datos sintéticos.

Considerando exclusivamente la fidelidad marginal categórica, el ranking obtenido es:

**Gaussian Copula V2.1 > CTGAN V1-C > TVAE V1-C.**

##### 6.6.2.4 Resumen de fidelidad estadística

| Criterio      |      GC V2.1 | CTGAN V1-C | TVAE V1-C | Mejor  |
| ------------- | -----------: | ---------: | --------: | ------ |
| KS medio ↓    | **0.113289** |   0.140601 |  0.160429 | **GC** |
| KS mediana ↓  | **0.078067** |   0.105871 |  0.137477 | **GC** |
| TVD medio ↓   | **0.004933** |   0.082966 |  0.115458 | **GC** |
| TVD mediana ↓ | **0.003182** |   0.071363 |  0.091528 | **GC** |


La conclusión es bastante clara:

Gaussian Copula V2.1 presenta la mayor fidelidad marginal agregada de los tres métodos evaluados.

Pero inmediatamente:

Esta superioridad marginal no se extiende necesariamente a la preservación de dependencias entre variables.

Esa segunda frase es esencial.

#### 6.6.3 Comparación de utilidad predictiva

In [ ]:
AUC_TRTR_BASELINE = 0.765600

AUC_TSTR_GC_V21 = 0.726674
AUC_TSTS_GC_V21 = 0.621809

auc_tstr_ctgan_v1c = 0.700643
auc_tsts_ctgan_v1c = 0.775625

auc_tstr_tvae_v1c = 0.670177
auc_tsts_tvae_v1c = 0.993802

La utilidad predictiva constituye una dimensión central del estudio, dado que el objetivo no consiste únicamente en reproducir distribuciones estadísticas, sino en determinar hasta qué punto los datos sintéticos pueden utilizarse para desarrollar modelos aplicables posteriormente sobre datos reales.

La comparación utiliza tres escenarios:

- **TRTR (Train Real, Test Real):** referencia obtenida entrenando y evaluando sobre particiones del dataset real.
- **TSTR (Train Synthetic, Test Real):** evalúa la capacidad de transferencia del conocimiento aprendido a partir de datos sintéticos hacia observaciones reales.
- **TSTS (Train Synthetic, Test Synthetic):** evalúa la utilidad predictiva interna del entorno sintético.

TSTR constituye el criterio predictivo principal para la selección del sintetizador, mientras que TSTS se utiliza como diagnóstico complementario de la estructura predictiva interna generada por cada método.

In [ ]:
comparacion_predictiva = pd.DataFrame({
    "Sintetizador": [
        "Gaussian Copula V2.1",
        "CTGAN V1-C",
        "TVAE V1-C"
    ],

    "TSTR_ROC_AUC": [
        AUC_TSTR_GC_V21,
        auc_tstr_ctgan_v1c,
        auc_tstr_tvae_v1c
    ],

    "TSTS_ROC_AUC": [
        AUC_TSTS_GC_V21,
        auc_tsts_ctgan_v1c,
        auc_tsts_tvae_v1c
    ]
})

comparacion_predictiva["Gap_TRTR_TSTR"] = (
    AUC_TRTR_BASELINE
    - comparacion_predictiva["TSTR_ROC_AUC"]
)

comparacion_predictiva["TSTR_Relativo_pct"] = (
    comparacion_predictiva["TSTR_ROC_AUC"]
    / AUC_TRTR_BASELINE
    * 100
)

comparacion_predictiva["Gap_TSTR_TSTS"] = (
    comparacion_predictiva["TSTR_ROC_AUC"]
    - comparacion_predictiva["TSTS_ROC_AUC"]
)

comparacion_predictiva.round(6)

,Sintetizador,TSTR_ROC_AUC,TSTS_ROC_AUC,Gap_TRTR_TSTR,TSTR_Relativo_pct,Gap_TSTR_TSTS
0,Gaussian Copula V2.1,0.726674,0.621809,0.038926,94.915622,0.104865
1,CTGAN V1-C,0.700643,0.775625,0.064957,91.515543,-0.074982
2,TVAE V1-C,0.670177,0.993802,0.095423,87.536181,-0.323625


1. Lectura de los resultados

| Sintetizador             |       TSTR ↑ |         TSTS | Gap TRTR−TSTR ↓ | TSTR relativo |
| ------------------------ | -----------: | -----------: | --------------: | ------------: |
| **Gaussian Copula V2.1** | **0.726674** |     0.621809 |    **0.038926** |    **94.92%** |
| CTGAN V1-C               |     0.700643 |     0.775625 |        0.064957 |        91.52% |
| TVAE V1-C                |     0.670177 | **0.993802** |        0.095423 |        87.54% |


Nuestro baseline real sigue siendo:

TRTR = 0.765600

Por tanto, desde la perspectiva que más nos interesa —entrenar con sintético y generalizar sobre real— el ranking es inequívoco:

Gaussian Copula V2.1 > CTGAN V1-C > TVAE V1-C.

Gaussian queda solamente 0.038926 ROC-AUC por debajo del baseline real y alcanza el 94.92% del ROC-AUC de referencia.

2. El resultado TSTS demuestra por qué necesitábamos TSTR

Si hubiéramos evaluado únicamente TSTS, el ranking habría sido exactamente el contrario:

TVAE > CTGAN > Gaussian

porque:

TVAE = 0.993802
CTGAN = 0.775625
Gaussian = 0.621809

Pero cuando esos modelos se enfrentan a datos reales:

Gaussian = 0.726674
CTGAN = 0.700643
TVAE = 0.670177

Esto nos permite formular una conclusión metodológica muy buena:

Una elevada utilidad predictiva interna del dataset sintético no implica necesariamente una elevada capacidad de transferencia hacia datos reales.

TVAE es el ejemplo más extremo.

Su TSTS es casi perfecto:

0.993802

pero su TSTR cae a:

0.670177.

La diferencia es:

−0.323625 al calcular TSTR − TSTS.

Eso indica un fuerte desacoplamiento entre la estructura predictiva aprendida dentro del entorno sintético y la que resulta útil sobre el real.

3. Gaussian presenta el comportamiento inverso

Este resultado es quizá incluso más interesante.

Gaussian:

TSTS = 0.621809

pero:

TSTR = 0.726674

Es decir:

TSTR > TSTS

por 0.104865.

No debemos interpretar esto como que Gaussian "funciona mejor con datos reales que consigo mismo" en un sentido general.

Lo que podemos afirmar es que, bajo nuestro protocolo:

Las relaciones aprendidas utilizando Gaussian V2.1 presentan mayor capacidad discriminativa al ser evaluadas sobre el conjunto real que sobre una partición independiente generada por el propio sintetizador.

Y eso encaja bastante bien con lo que estamos descubriendo sobre Gaussian:

distribuciones marginales excelentes;
algunas relaciones estructurales muy deterioradas;
relaciones TARGET-X atenuadas;
TSTS relativamente bajo;
pero buena transferibilidad al real.

Es decir, Gaussian parece generar una representación menos discriminativa internamente, pero suficientemente alineada con determinadas características relevantes del dataset real para producir el mejor TSTR.

4. CTGAN queda en una posición intermedia muy interesante

CTGAN:

TSTR = 0.700643

TSTS = 0.775625

Tiene un gap de:

−0.074982

No tiene el desacoplamiento extremo de TVAE y, además, anteriormente vimos que:

conserva muy bien las dos dependencias estructurales;
obtiene el mejor MAE TARGET-X (0.035255);
tiene fidelidad marginal inferior a Gaussian;
consigue el segundo mejor TSTR.

Por tanto, CTGAN está apareciendo como el sintetizador más equilibrado estructuralmente, mientras Gaussian está destacando específicamente en fidelidad marginal y transferibilidad.

Esto será importante en la selección final.

##### 6.6.3.1 Resultados de utilidad predictiva

La evaluación de utilidad predictiva evidencia diferencias importantes entre los tres métodos de generación sintética.

El escenario TRTR, utilizado como referencia, alcanza un ROC-AUC de 0.765600. En el escenario TSTR, Gaussian Copula V2.1 obtiene el mejor resultado con 0.726674, seguido de CTGAN V1-C con 0.700643 y TVAE V1-C con 0.670177.

Las diferencias respecto al baseline TRTR son, respectivamente, 0.038926, 0.064957 y 0.095423. Expresado de forma relativa, los valores TSTR representan el 94.92 %, 91.52 % y 87.54 % del ROC-AUC obtenido mediante entrenamiento y evaluación sobre datos reales.

Por tanto, considerando la capacidad de transferencia hacia observaciones reales, el ranking obtenido es:

**Gaussian Copula V2.1 > CTGAN V1-C > TVAE V1-C.**

Los resultados TSTS muestran, sin embargo, un comportamiento sustancialmente diferente. Gaussian Copula V2.1 obtiene 0.621809, CTGAN V1-C alcanza 0.775625 y TVAE V1-C obtiene 0.993802.

En consecuencia, el ranking basado exclusivamente en TSTS sería inverso al observado mediante TSTR. Este resultado evidencia que una elevada capacidad predictiva dentro del entorno sintético no implica necesariamente una elevada capacidad de transferencia hacia datos reales.

El caso más extremo corresponde a TVAE V1-C, cuyo TSTS de 0.993802 contrasta con un TSTR de 0.670177, produciendo una diferencia TSTR-TSTS de -0.323625. Este resultado indica un fuerte desacoplamiento entre la estructura predictiva interna del dataset generado y aquella que resulta transferible al dataset real.

CTGAN V1-C presenta un comportamiento intermedio, con TSTS de 0.775625 y TSTR de 0.700643.

Gaussian Copula V2.1 presenta el comportamiento contrario: su TSTR de 0.726674 supera al TSTS de 0.621809. Esto indica que, bajo el protocolo experimental utilizado, las relaciones aprendidas a partir del dataset generado por Gaussian Copula presentan mayor capacidad discriminativa al ser evaluadas sobre datos reales que dentro de una partición independiente del propio entorno sintético.

Estos resultados justifican la utilización de TSTR como criterio predictivo principal del estudio. TSTS se mantiene como una métrica complementaria para caracterizar la utilidad predictiva interna y detectar posibles divergencias entre la estructura sintética y la real.

#### 6.6.4 Comparación multidimensional

##### 6.6.4.1 Consolidación de métricas

| Dimensión                           |      GC V2.1 |   CTGAN V1-C |    TVAE V1-C |
| ----------------------------------- | -----------: | -----------: | -----------: |
| KS medio ↓                          | **0.113289** |     0.140601 |     0.160429 |
| TVD medio ↓                         | **0.004933** |     0.082966 |     0.115458 |
| Error estructural CHILDREN-FAMILY ↓ |     0.363417 |     0.020672 | **0.015610** |
| Error estructural DEF30-DEF60 ↓     |     0.772771 | **0.082925** |     0.097722 |
| MAE TARGET-X ↓                      |     0.041832 | **0.035255** |     0.076550 |
| TSTR ↑                              | **0.726674** |     0.700643 |     0.670177 |
| TSTS                                |     0.621809 |     0.775625 |     0.993802 |


In [ ]:
comparacion_multidimensional = pd.DataFrame({
    "Sintetizador": [
        "Gaussian Copula V2.1",
        "CTGAN V1-C",
        "TVAE V1-C"
    ],

    "KS_Medio": [
        0.113289,
        0.140601,
        0.160429
    ],

    "TVD_Medio": [
        0.004933,
        0.082966,
        0.115458
    ],

    "Error_CHILDREN_FAMILY": [
        0.363417,
        0.020672,
        0.015610
    ],

    "Error_DEF30_DEF60": [
        0.772771,
        0.082925,
        0.097722
    ],

    "MAE_TARGET_X": [
        0.041832,
        0.035255,
        0.076550
    ],

    "TSTR_ROC_AUC": [
        0.726674,
        0.700643,
        0.670177
    ],

    "TSTS_ROC_AUC": [
        0.621809,
        0.775625,
        0.993802
    ]
})

comparacion_multidimensional.set_index(
    "Sintetizador"
).round(6)

,KS_Medio,TVD_Medio,Error_CHILDREN_FAMILY,Error_DEF30_DEF60,MAE_TARGET_X,TSTR_ROC_AUC,TSTS_ROC_AUC
Sintetizador,,,,,,,
Gaussian Copula V2.1,0.113289,0.004933,0.363417,0.772771,0.041832,0.726674,0.621809
CTGAN V1-C,0.140601,0.082966,0.020672,0.082925,0.035255,0.700643,0.775625
TVAE V1-C,0.160429,0.115458,0.015610,0.097722,0.076550,0.670177,0.993802


##### 6.6.4.2 No hagamos todavía un score ponderado

Podemos resumir:

| Dimensión                     | Mejor resultado |
| ----------------------------- | --------------- |
| Fidelidad marginal numérica   | **Gaussian**    |
| Fidelidad marginal categórica | **Gaussian**    |
| Dependencia CHILDREN-FAMILY   | **TVAE**        |
| Dependencia DEF30-DEF60       | **CTGAN**       |
| Relaciones TARGET-X           | **CTGAN**       |
| Transferibilidad TSTR         | **Gaussian**    |
| Utilidad interna TSTS         | TVAE*           |


Y ese asterisco será importantísimo:

Un mayor TSTS no se interpreta automáticamente como mayor calidad, debido a que valores elevados acompañados de un TSTR sustancialmente inferior pueden indicar una estructura predictiva sintética poco transferible.

Por tanto, no deberíamos contar simplemente "cuántas métricas gana cada uno".

##### 6.6.4.3 Nos falta una dimensión antes de cerrar esta comparación

**Coste computacional**

El coste computacional constituye un criterio complementario en la comparación de los sintetizadores.

CTGAN presentó el mayor tiempo de entrenamiento entre los modelos neuronales evaluados, con aproximadamente 155.50 minutos para 300 épocas. TVAE requirió aproximadamente 71.35 minutos bajo la misma cantidad de épocas y entorno experimental.

En la generación condicionada del dataset completo, CTGAN requirió aproximadamente 0.71 minutos, mientras que TVAE necesitó aproximadamente 2.38 minutos.

En el caso de Gaussian Copula V2.1 no se registró sistemáticamente el tiempo de entrenamiento y generación durante la ejecución experimental. No obstante, durante el desarrollo se observó cualitativamente una ejecución considerablemente más rápida que la correspondiente a los modelos neuronales.

Debido a la ausencia de una medición temporal registrada para Gaussian Copula, esta observación se utiliza únicamente como consideración cualitativa y no como resultado cuantitativo de rendimiento computacional.

Por tanto, no se establece un ranking temporal formal entre los tres métodos.

| Dimensión                          | Gaussian V2.1 |   CTGAN V1-C |    TVAE V1-C |
| ---------------------------------- | ------------: | -----------: | -----------: |
| KS medio ↓                         |  **0.113289** |     0.140601 |     0.160429 |
| TVD medio ↓                        |  **0.004933** |     0.082966 |     0.115458 |
| Error CHILDREN-FAMILY ↓            |      0.363417 |     0.020672 | **0.015610** |
| Error DEF30-DEF60 ↓                |      0.772771 | **0.082925** |     0.097722 |
| MAE TARGET-X ↓                     |      0.041832 | **0.035255** |     0.076550 |
| TSTR ↑                             |  **0.726674** |     0.700643 |     0.670177 |
| TSTS                               |      0.621809 |     0.775625 |     0.993802 |
| Coste computacional observado      |         Bajo* |         Alto |   Intermedio |
| Entrenamiento registrado           |            No |   155.50 min |    71.35 min |
| Generación condicionada registrada |            No |     0.71 min |     2.38 min |


* Observación cualitativa; no se registró una medición temporal comparable.

Y aquí aparece algo bastante bonito de tu experimento:

Gaussian
→ mejor fidelidad marginal + mejor TSTR, pero peor en las dependencias estructurales seleccionadas.

CTGAN
→ mejor equilibrio en dependencias estructurales y TARGET-X, segundo TSTR, pero mayor coste de entrenamiento registrado.

TVAE
→ conserva algunas dependencias muy bien, pero presenta peor fidelidad marginal, peor TSTR y una divergencia TSTS/TSTR extremadamente grande.

##### 6.6.4.4 Síntesis de la comparación multidimensional

La evaluación conjunta muestra que ningún sintetizador domina todas las dimensiones consideradas.

Gaussian Copula V2.1 presenta la mayor fidelidad marginal, tanto para las variables numéricas como categóricas, obteniendo el menor KS medio (0.113289) y TVD medio (0.004933). Asimismo, obtiene la mayor capacidad de transferencia predictiva, con un TSTR ROC-AUC de 0.726674, equivalente al 94.92 % del ROC-AUC de referencia TRTR.

Su principal limitación aparece en la preservación de determinadas dependencias estructurales. Las relaciones CNT_CHILDREN–CNT_FAM_MEMBERS y DEF_30_CNT_SOCIAL_CIRCLE–DEF_60_CNT_SOCIAL_CIRCLE presentan desviaciones considerablemente superiores a las observadas en CTGAN y TVAE.

CTGAN V1-C presenta un perfil diferente. Aunque su fidelidad marginal es inferior a Gaussian Copula, conserva mejor las dependencias estructurales seleccionadas y obtiene el menor MAE en las relaciones críticas TARGET-X (0.035255). Su TSTR de 0.700643 lo sitúa en segunda posición respecto a capacidad de transferencia.

TVAE V1-C conserva adecuadamente algunas dependencias estructurales, pero presenta la mayor divergencia marginal agregada y el menor TSTR (0.670177). Asimismo, la elevada diferencia entre su TSTS (0.993802) y TSTR evidencia que su elevada capacidad predictiva interna no se traduce en una capacidad equivalente de transferencia hacia datos reales.

Respecto al coste computacional, CTGAN requirió aproximadamente 155.50 minutos de entrenamiento y TVAE 71.35 minutos. Gaussian Copula presentó cualitativamente una ejecución más rápida durante el desarrollo experimental, aunque no se registró una medición temporal sistemática, por lo que no se establece un ranking cuantitativo formal en esta dimensión.

En conjunto, los resultados confirman que la evaluación de datos sintéticos requiere considerar simultáneamente fidelidad marginal, preservación de dependencias y utilidad predictiva. Ninguna de estas dimensiones, evaluada aisladamente, resulta suficiente para determinar la adecuación global de un sintetizador.

### 6.7 Selección final

La selección del método de generación sintética se realiza considerando conjuntamente los resultados obtenidos en las diferentes dimensiones de evaluación.

No se utiliza una puntuación agregada basada en ponderaciones arbitrarias, dado que las métricas analizadas representan propiedades diferentes de los datos sintéticos y no son directamente intercambiables.

La selección se fundamenta principalmente en la finalidad del estudio: determinar si los datos sintéticos pueden utilizarse como sustituto de los datos reales durante el desarrollo de modelos de estimación de Probabilidad de Incumplimiento (PD).

En consecuencia, la capacidad de transferencia predictiva medida mediante TSTR constituye el criterio prioritario. La fidelidad marginal, la preservación de dependencias estructurales, las relaciones con TARGET y el comportamiento TSTS se utilizan como criterios complementarios para caracterizar las fortalezas y limitaciones de cada alternativa.

#### 6.7.1 Criterios de selección

Los criterios considerados para la selección final son los siguientes:

1. **Transferibilidad predictiva (TSTR).**  
   Constituye el criterio principal, ya que mide directamente la capacidad de un modelo entrenado con datos sintéticos para discriminar el incumplimiento sobre observaciones reales.

2. **Fidelidad estadística marginal.**  
   Evalúa la similitud entre las distribuciones reales y sintéticas mediante KS para variables numéricas y TVD para variables categóricas y discretas.

3. **Preservación de dependencias.**  
   Permite identificar si el sintetizador conserva relaciones relevantes entre variables, incluyendo dependencias estructurales y asociaciones con TARGET.

4. **Utilidad predictiva interna (TSTS).**  
   Se utiliza como diagnóstico complementario. Un TSTS elevado no se considera suficiente para seleccionar un sintetizador si dicha capacidad predictiva no se transfiere adecuadamente al dataset real.

5. **Coste computacional.**  
   Se considera como criterio secundario. Debido a que no se registraron tiempos comparables para todos los métodos, no se utiliza como criterio cuantitativo decisivo.

#### 6.7.2 Evaluación de candidatos

**Gaussian Copula V2.1**

Gaussian Copula V2.1 obtiene el mayor TSTR ROC-AUC (0.726674), frente a un baseline TRTR de 0.765600. Esto representa el 94.92 % del ROC-AUC de referencia y una diferencia absoluta de 0.038926.

Asimismo, presenta la mayor fidelidad marginal agregada, con el menor KS medio (0.113289) y TVD medio (0.004933).

Su principal limitación corresponde a la conservación de determinadas dependencias estructurales. En particular, las relaciones CNT_CHILDREN–CNT_FAM_MEMBERS y DEF_30_CNT_SOCIAL_CIRCLE–DEF_60_CNT_SOCIAL_CIRCLE presentan desviaciones superiores a las observadas en los modelos neuronales.

Por tanto, Gaussian Copula combina la mayor transferibilidad predictiva y fidelidad marginal con limitaciones relevantes en determinadas relaciones multivariantes.

**CTGAN V1-C**

CTGAN V1-C obtiene un TSTR ROC-AUC de 0.700643, equivalente al 91.52 % del baseline TRTR.

Aunque su fidelidad marginal es inferior a Gaussian Copula, presenta una conservación considerablemente superior de las dependencias estructurales seleccionadas. Asimismo, obtiene el menor MAE en las relaciones críticas TARGET-X (0.035255).

CTGAN constituye, por tanto, una alternativa especialmente competitiva desde el punto de vista de preservación de dependencias. Sin embargo, su capacidad de transferencia hacia datos reales resulta inferior a Gaussian Copula V2.1 bajo el protocolo experimental utilizado.

Además, su entrenamiento presentó el mayor coste computacional registrado entre los modelos neuronales, con aproximadamente 155.50 minutos para 300 épocas.

**TVAE V1-C**

TVAE V1-C obtiene el menor TSTR de los tres métodos evaluados, con un ROC-AUC de 0.670177, equivalente al 87.54 % del baseline TRTR.

Aunque conserva adecuadamente algunas dependencias estructurales, presenta el mayor KS medio (0.160429), el mayor TVD medio (0.115458) y el mayor MAE en las relaciones críticas TARGET-X (0.076550).

Adicionalmente, su TSTS alcanza 0.993802, considerablemente por encima de su TSTR. Esta divergencia indica que la elevada discriminabilidad observada dentro del entorno sintético no se transfiere en la misma medida hacia observaciones reales.

En consecuencia, TVAE V1-C presenta el perfil menos adecuado para el objetivo principal del estudio bajo la configuración evaluada.

#### 6.7.3 Selección final

Considerando conjuntamente los resultados obtenidos, se selecciona **Gaussian Copula V2.1** como método de generación sintética para las siguientes etapas del estudio.

La selección se fundamenta principalmente en su capacidad de transferencia predictiva. Gaussian Copula V2.1 alcanza un TSTR ROC-AUC de 0.726674, el mayor entre los tres métodos evaluados y equivalente al 94.92 % del ROC-AUC obtenido en el escenario de referencia TRTR.

Adicionalmente, presenta la mayor fidelidad marginal agregada tanto en variables numéricas como categóricas, obteniendo el menor KS medio (0.113289) y el menor TVD medio (0.004933).

La selección no implica que Gaussian Copula sea superior en todas las dimensiones. CTGAN V1-C conserva considerablemente mejor determinadas dependencias estructurales y presenta el menor error medio en las relaciones críticas con TARGET. Estas ventajas evidencian una limitación relevante de Gaussian Copula V2.1: una elevada fidelidad marginal y una buena capacidad de transferencia no garantizan la reproducción completa de la estructura multivariante real.

Sin embargo, dado que el objetivo prioritario del estudio consiste en evaluar la utilización de datos sintéticos como sustituto de los datos reales para el entrenamiento de modelos de estimación de PD, se prioriza la capacidad TSTR sobre la similitud estructural aislada.

TVAE V1-C no es seleccionado debido a su menor capacidad TSTR, mayor divergencia marginal agregada y fuerte desacoplamiento entre TSTS y TSTR.

Por tanto, **Gaussian Copula V2.1 constituye el dataset sintético seleccionado para continuar con la etapa de modelado y evaluación posterior del TFM**, manteniendo documentadas sus limitaciones en la preservación de determinadas dependencias multivariantes.

#### 6.8 Conclusiones del notebook

En este notebook se compararon tres familias de generación de datos sintéticos aplicadas al problema de Credit Scoring: Gaussian Copula, CTGAN y TVAE.

La evaluación se realizó mediante un enfoque multidimensional que consideró fidelidad estadística marginal, preservación de dependencias estructurales, conservación de relaciones con TARGET y utilidad predictiva mediante los escenarios TSTR y TSTS.

Para reducir la asimetría asociada a la distribución de la variable objetivo, se realizó adicionalmente un experimento controlado en el que Gaussian Copula V2.1, CTGAN V1-C y TVAE V1-C reprodujeron exactamente la prevalencia real de TARGET.

Los resultados mostraron que Gaussian Copula V2.1 presenta la mayor fidelidad marginal agregada, con un KS medio de 0.113289 y un TVD medio de 0.004933. Asimismo, obtuvo la mayor capacidad de transferencia predictiva, alcanzando un TSTR ROC-AUC de 0.726674 frente al baseline TRTR de 0.765600.

CTGAN V1-C obtuvo un TSTR de 0.700643 y destacó por una mejor conservación de determinadas dependencias estructurales y por presentar el menor MAE en las relaciones críticas TARGET-X. Este comportamiento demuestra que la preservación de relaciones bivariadas y estructurales no implica necesariamente una mayor capacidad predictiva de transferencia.

TVAE V1-C obtuvo un TSTR de 0.670177. A pesar de alcanzar un TSTS de 0.993802, esta elevada discriminabilidad interna no se trasladó hacia los datos reales, evidenciando la importancia de no utilizar TSTS de forma aislada como criterio de calidad.

Los experimentos muestran además que la fidelidad marginal y la conservación de dependencias representan propiedades diferentes. Gaussian Copula reprodujo con elevada precisión las distribuciones individuales de determinadas variables categóricas, pero presentó pérdidas importantes en algunas relaciones entre ellas. CTGAN y TVAE, por el contrario, conservaron mejor determinadas dependencias aun presentando mayores desviaciones marginales.

Considerando que el objetivo principal del estudio es determinar la utilidad de los datos sintéticos como sustituto de los datos reales durante el desarrollo de modelos de estimación de PD, se selecciona **Gaussian Copula V2.1** para las siguientes etapas del TFM.

Esta selección se circunscribe al dataset, configuración y protocolo experimental utilizados, y no implica una superioridad general de Gaussian Copula frente a CTGAN o TVAE.

In [ ]:
#ESto no se puede generar
